## Imports

In [2]:
import os
import time
import glob
import numpy as np
import pandas as pd
import datetime as datetime

from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

## Datasets & Parameter

In [4]:
local_dataset_path = '/Users/E2074/local-datasets/customer/loyalty/experiment2/'

In [5]:
start_date = '20250516'
end_date = '20250526'
city = {'Bangalore' : 'loyalty_experiment2_bangalore', 
        'Delhi' : 'loyalty_experiment2_delhi', 
        'Hyderabad' : 'loyalty_experiment2_hyd'}

experiment_name = 'loyalty_experiment2'

In [6]:
start = datetime.strptime(start_date, '%Y%m%d')
end = datetime.strptime(end_date, '%Y%m%d')

date_list = [(start + timedelta(days=i)).strftime('%Y%m%d') for i in range((end - start).days + 1)]

date_list

['20250516',
 '20250517',
 '20250518',
 '20250519',
 '20250520',
 '20250521',
 '20250522',
 '20250523',
 '20250524',
 '20250525',
 '20250526']

### Orders

In [100]:
for date in date_list[10:]:
    
    print( '---------' , date , '---------')
    
    for city_name, execution_name in  city.items():
        
        print('Doing Science for', city_name)
        
        query = f"""
            
            with rapidex as (
            
            select 
                distinct 
                yyyymmdd,
                experiment_name,
                execution_name,
                sample_category,
                customer_id
            from 
                iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
            where 
                yyyymmdd = '{date}'
                and experiment_name = '{experiment_name}'
                and execution_name = '{execution_name}'
            ),
    
            segment as (
            
            select
                customer_id,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_regularity_segment,
                taxi_income_segment,
                taxi_intent_segment
            from 
                datasets.iallocator_customer_segments
            where 
                run_date = '2025-05-18'
                and taxi_recency_segment != 'UNKNOWN'
            ),
            
            experiment_group as (
            
            select
                rapidex.*,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_regularity_segment,
                taxi_income_segment,
                taxi_intent_segment
            from 
                rapidex
            left join 
                segment
                on rapidex.customer_id = segment.customer_id
            
            ),
            
            orders as (
        
            select
                yyyymmdd,
                customer_id,
                city_name,
                service_obj_service_name service_name,
                order_id,
                modified_order_status,
                customer_cancelled_epoch,
                order_requested_epoch,
                accepted_epoch,
                accept_to_cancelled,
                case when modified_order_status = 'COBRA' then ((customer_cancelled_epoch - order_requested_epoch)/1000) end cobra_ttc,
                case when modified_order_status = 'OCARA' then ((customer_cancelled_epoch - accepted_epoch)/1000) end ocara_ttc,
                case when accepted_epoch is not null then ((accepted_epoch - order_requested_epoch)/1000) end tta,
                epoch,
                quarter_hour,
                hhmmss,
                distance_final_distance,
                accept_to_pickup_distance,
                pickup_location_hex_12,
                drop_location_hex_12,
                pickup_location_hex_8,
                drop_location_hex_8,
                channel_host,
                case 
                    when cast(substr(quarter_hour,1,2) as real) > 7 and cast(substr(quarter_hour,1,2) as real) <= 11
                        then 'morning_peak'
                    when cast(substr(quarter_hour,1,2) as real) > 11 and cast(substr(quarter_hour,1,2) as real) <= 16
                        then 'afternoon'
                    when cast(substr(quarter_hour,1,2) as real) > 16 and cast(substr(quarter_hour,1,2) as real) <= 21
                        then 'evening_peak'
                    when cast(substr(quarter_hour,1,2) as real) > 21 and cast(substr(quarter_hour,1,2) as real) <= 23
                        then 'rest_evening'
                    when cast(substr(quarter_hour,1,2) as real) >= 0 and cast(substr(quarter_hour,1,2) as real) <= 7
                        then 'rest_morning'
                end as time_bucket
            from 
                orders.order_logs_fact a
            where 
                yyyymmdd = '{date}'
                and city_name = '{city_name}'
                and service_obj_service_name in ('Auto', 'Link', 'CabAC', 'Auto Pool', 'CabEconomy', 'CabPremium', 'Cab SUV', 'Bike Lite', 'Auto Pet', 'Auto NCR', 'Bike Metro', 'Scooty')
                and lower(channel_host) in ('android', 'ios')
            )
        
            select 
                a.experiment_name,
                a.execution_name,
                a.sample_category,
                a.taxi_ltr_segment,
                a.taxi_retention_segments,
                a.taxi_regularity_segment,
                a.taxi_income_segment,
                a.taxi_intent_segment,
                b.*
            from 
                experiment_group a
            join 
                orders b
                on a.yyyymmdd = b.yyyymmdd
                and a.customer_id = b.customer_id
        """
    
        df_query = pd.read_sql(query, conn4)
        df_query.to_parquet(local_dataset_path + city_name + '/{}_{}_orders_data_dump.parquet'.format(date, city_name), index=False)

--------- 20250526 ---------
Doing Science for Hyderabad


### Telemetry

In [9]:
for date in date_list[8:]:
    
    print( '---------' , date , '---------')
    
    query = f"""
    
        select 
            yyyymmdd,
            city,
            userid,
            orderid,
            quarter_hour,
            count(distinct aduuid) aduuid_count,
            max(case when slotname = 'CaptainSearch1' and eventname = 'Ad_Viewable_Impression' then userid end) dispatch_viewed_cux,
            max(case when slotname = 'CaptainSearch1' and eventname = 'Ad_Click' then userid end) dispatch_clicked_cux,
            max(case when slotname = 'PostOrderOnTheWay1' and eventname = 'Ad_Viewable_Impression' then userid end) otw_viewed_cux,
            max(case when slotname = 'PostOrderOnTheWay1' and eventname = 'Ad_Click' then userid end) otw_clicked_cux
        from 
            canonical.iceberg_log_telemetry_ads_impressions_immutable
        where 
            yyyymmdd = '{date}'
            and city in ('Bangalore', 'Delhi', 'Hyderabad')
            and responsetype = 'ONLINE_SALES'
            and eventname in ('Ad_Viewable_Impression', 'Ad_Click')
            and format = 'maxBanner'
            and slotname in ('CaptainSearch1', 'PostOrderOnTheWay1')
            and regexp_like(metadata1,  'CaptainSearchLoyalty|PostOrderOnTheWayLoyalty1')
            
        group by 1,2,3,4,5
    """

    df_query = pd.read_sql(query, conn3)
    df_query.to_parquet(local_dataset_path + 'telemetry-engagement' + '/{}_telemetry_engagement_data_dump.parquet'.format(date), index=False)

--------- 20250524 ---------
--------- 20250525 ---------
--------- 20250526 ---------


### MarketPlace

In [62]:
query = f"""

with base as (
    
    select
        city_name,
        pickup_location_hex_8,
        case 
            when cast(substr(quarter_hour,1,2) as real) > 7 and cast(substr(quarter_hour,1,2) as real) <= 11 then 'morning_peak'
            when cast(substr(quarter_hour,1,2) as real) > 11 and cast(substr(quarter_hour,1,2) as real) <= 16 then 'afternoon'
            when cast(substr(quarter_hour,1,2) as real) > 16 and cast(substr(quarter_hour,1,2) as real) <= 21 then 'evening_peak'
            when cast(substr(quarter_hour,1,2) as real) > 21 and cast(substr(quarter_hour,1,2) as real) <= 23 then 'rest_evening'
            when cast(substr(quarter_hour,1,2) as real) >= 0 and cast(substr(quarter_hour,1,2) as real) <= 7 then 'rest_morning'
        end as time_bucket,
        count(distinct order_id) gross_orders,
        count(distinct case when accepted_epoch is not null then order_id end) accepted_orders,
        try(count(distinct case when accepted_epoch is not null then order_id end)*100.00/count(distinct order_id)) aor
    from 
        orders.order_logs_fact a
    where 
        yyyymmdd >= '20250501'
        and yyyymmdd <= '20250514'
        and city_name in ('Bangalore', 'Delhi', 'Hyderabad')
        and service_obj_service_name in ('Auto', 'Link', 'CabAC', 'Auto Pool', 'CabEconomy', 'CabPremium', 'Cab SUV', 'Bike Lite', 'Auto Pet', 'Auto NCR', 'Bike Metro', 'Scooty')
        and lower(channel_host) in ('android', 'ios')
    group by 1,2,3
    having count(distinct order_id) >= 30
    ),
    
    threshold_value as (
    
    select
        city_name,
        -- pickup_location_hex_8,
        time_bucket,
        approx_percentile(aor, 0.33) p33,
        approx_percentile(aor, 0.66) p66
    from 
        base 
    group by 1,2
    )
    
    
    select
        a.*,
        case 
            when aor > p66 then '1. Good'
            when aor between p33 and p66 then '2. Balance'
            when aor < p33 then '3. Poor'
        end mp_tag
    from 
        base a
    left join 
        threshold_value b
        on a.city_name = b.city_name
        and a.time_bucket = b.time_bucket

"""

df_query = pd.read_sql(query, conn2)
df_query.to_parquet(local_dataset_path + 'marketplace' + '/marketplace_tag.parquet', index=False)

In [25]:
df_marketplace = pd.read_parquet(local_dataset_path + 'marketplace' + '/marketplace_tag.parquet')

In [26]:
df_marketplace.head(1)

,city_name,pickup_location_hex_8,time_bucket,gross_orders,accepted_orders,aor,mp_tag
0,Bangalore,88618928d9fffff,rest_morning,141,65,46.1,3. Poor


### Click time

In [6]:
for date in date_list:
    
    print( '---------' , date , '---------')
    
    query = f"""

        select
            *,
            (click - coalesce(view,render)) time_taken_to_click
        from 
            (
            select
                yyyymmdd,
                city,
                userid,
                aduuid,
                -- eventname,
                min(case when eventname = 'Ad_Impression' then epoch/1000 end) render,
                min(case when eventname = 'Ad_Viewable_Impression' then epoch/1000 end) view,
                min(case when eventname = 'Ad_Click' then epoch/1000 end) click
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd = '{date}'
                and city in ('Bangalore', 'Delhi', 'Hyderabad')
                and responsetype = 'ONLINE_SALES'
                and eventname in ('Ad_Impression', 'Ad_Viewable_Impression', 'Ad_Click')
                and format = 'maxBanner'
                and slotname in ('CaptainSearch1', 'PostOrderOnTheWay1')
                and regexp_like(metadata1,  'CaptainSearchLoyalty|PostOrderOnTheWayLoyalty1')
                
                -- and aduuid = '66d421f5d799514996f0d094_4FB282FA-4271-4602-A9CB-4994B40D6420_1748161791738'
            group by 1,2,3,4
            )
        where 
            click is not null

    """

    df_query = pd.read_sql(query, conn2)
    df_query.to_parquet(local_dataset_path + 'click-data' + '/{}click_data.parquet'.format(date), index=False)

--------- 20250516 ---------
--------- 20250517 ---------


DatabaseError: {'message': 'No nodes available to run query', 'errorCode': 65541, 'errorName': 'NO_NODES_AVAILABLE', 'errorType': 'INTERNAL_ERROR', 'failureInfo': {'type': 'io.trino.spi.TrinoException', 'message': 'No nodes available to run query', 'suppressed': [], 'stack': ['io.trino.execution.scheduler.UniformNodeSelector.computeAssignments(UniformNodeSelector.java:178)', 'io.trino.execution.scheduler.DynamicSplitPlacementPolicy.computeAssignments(DynamicSplitPlacementPolicy.java:41)', 'io.trino.execution.scheduler.SourcePartitionedScheduler.schedule(SourcePartitionedScheduler.java:343)', 'io.trino.execution.scheduler.SourcePartitionedScheduler$1.schedule(SourcePartitionedScheduler.java:188)', 'io.trino.execution.scheduler.SqlQueryScheduler$PipelinedDistributedStagesScheduler.schedule(SqlQueryScheduler.java:1606)', 'java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)', 'java.base/java.util.concurrent.FutureTask.run(FutureTask.java:264)', 'java.base/java

In [ ]:
def get_click_timing():
    base_path = local_dataset_path + 'click-data/'
    parquet_files = glob.glob(os.path.join(base_path, '*click_data.parquet'))
    df_all = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    df_all['time_taken_to_click'] = abs(df_all['time_taken_to_click'])
    return df_all

df_click_timing = get_click_timing()

In [ ]:
print(df_click_timing.shape)
print(df_click_timing.userid.nunique())
df_click_timing.head(4)

In [ ]:
quantiles = df_click_timing['time_taken_to_click'].quantile([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
quantiles

## Load Extracted Data

In [9]:
all_dfs = []

for city_folder in city.keys():
    folder_path = os.path.join(local_dataset_path, city_folder)
    print(folder_path)
    
    for file_name in os.listdir(folder_path): 
        if file_name.endswith('.parquet'): 
            file_path = os.path.join(folder_path, file_name) 
            df = pd.read_parquet(file_path) 
            all_dfs.append(df)

df_base = pd.concat(all_dfs, ignore_index=True)

/Users/E2074/local-datasets/customer/loyalty/experiment2/Bangalore
/Users/E2074/local-datasets/customer/loyalty/experiment2/Delhi
/Users/E2074/local-datasets/customer/loyalty/experiment2/Hyderabad


In [10]:
def get_customer_engagement():
    base_path = local_dataset_path + 'telemetry-engagement/'
    parquet_files = glob.glob(os.path.join(base_path, '*_telemetry_engagement_data_dump.parquet'))
    df_all = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    
    return df_all

df_customer_engagement = get_customer_engagement()

In [12]:
df_customer_engagement.head(1)

,yyyymmdd,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux
0,20250526,Bangalore,5737c6f8ddbec2203f7333f5,2,5737c6f8ddbec2203f7333f5,None,None,None


In [13]:
df_customer_engagement = df_customer_engagement.groupby(['city', 'userid']).agg(aduuid_count=('aduuid_count', 'sum'), 
                                                       dispatch_viewed_cux = ('dispatch_viewed_cux', 'first'),
                                                       dispatch_clicked_cux = ('dispatch_clicked_cux', 'first'),
                                                       otw_viewed_cux = ('otw_viewed_cux', 'first'),
                                                       otw_clicked_cux = ('otw_clicked_cux', 'first')
                                                      ).reset_index()

In [14]:
print(df_customer_engagement.shape)
print(df_customer_engagement.userid.nunique())

(453686, 7)
452241


In [15]:
df_customer_engagement['clicked'] = np.where((~df_customer_engagement['dispatch_clicked_cux'].isna()) & (~df_customer_engagement['otw_clicked_cux'].isna()), 
                                               'yes', None)

df_customer_engagement['both_view'] = np.where((~df_customer_engagement['dispatch_viewed_cux'].isna()) & (~df_customer_engagement['otw_viewed_cux'].isna()), 
                                               'yes', None)

df_customer_engagement['single_view'] = np.where((~df_customer_engagement['dispatch_viewed_cux'].isna()) | (~df_customer_engagement['otw_viewed_cux'].isna()), 
                                                 'yes', None)

In [16]:
print(df_base.shape)
display(df_base.head(1))
print(df_customer_engagement.shape)
display(df_customer_engagement.head(1))

(23029783, 32)


,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,pickup_location_hex_12,drop_location_hex_12,pickup_location_hex_8,drop_location_hex_8,channel_host,time_bucket
0,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,6210585d2a4d42590ef0a5b6,Bangalore,Auto,6833f6e9d1542e19333dea7e,dropped,NaN,1.748236e+12,1.748236e+12,NaN,NaN,NaN,7.0,1748236009534,1030,103649,1.45,0.926,8c61892602f3dff,8c6189260621dff,8861892603fffff,8861892607fffff,android,morning_peak


(453686, 10)


,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view
0,Bangalore,5737c6f8ddbec2203f7333f5,2,5737c6f8ddbec2203f7333f5,None,None,None,None,None,yes


In [17]:
print(df_base.info())
print(df_customer_engagement.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23029783 entries, 0 to 23029782
Data columns (total 32 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   experiment_name            object 
 1   execution_name             object 
 2   sample_category            object 
 3   taxi_ltr_segment           object 
 4   taxi_retention_segments    object 
 5   taxi_regularity_segment    object 
 6   taxi_income_segment        object 
 7   taxi_intent_segment        object 
 8   yyyymmdd                   object 
 9   customer_id                object 
 10  city_name                  object 
 11  service_name               object 
 12  order_id                   object 
 13  modified_order_status      object 
 14  customer_cancelled_epoch   float64
 15  order_requested_epoch      float64
 16  accepted_epoch             float64
 17  accept_to_cancelled        float64
 18  cobra_ttc                  float64
 19  ocara_ttc                  float64
 20  

#### Frequency

In [18]:
df_base.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,pickup_location_hex_12,drop_location_hex_12,pickup_location_hex_8,drop_location_hex_8,channel_host,time_bucket
0,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,6210585d2a4d42590ef0a5b6,Bangalore,Auto,6833f6e9d1542e19333dea7e,dropped,NaN,1.748236e+12,1.748236e+12,NaN,NaN,NaN,7.0,1748236009534,1030,103649,1.45,0.926,8c61892602f3dff,8c6189260621dff,8861892603fffff,8861892607fffff,android,morning_peak


In [19]:
df_customer_engagement.head(1)

,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view
0,Bangalore,5737c6f8ddbec2203f7333f5,2,5737c6f8ddbec2203f7333f5,None,None,None,None,None,yes


In [20]:
df_customer_orderes = df_base.groupby(['customer_id']).agg(orders=('order_id', 'nunique')).reset_index()

In [21]:
df_view_frequency =  df_customer_engagement.groupby(['userid']).agg(view_frequency=('aduuid_count', 'max')).reset_index()

In [22]:
df_freq_merge = pd.merge(df_customer_orderes, df_view_frequency, how='inner', left_on='customer_id', right_on='userid')

In [23]:
df_freq_merge['frequency'] = np.where(df_freq_merge['orders'] >= df_freq_merge['view_frequency'], df_freq_merge['view_frequency'], df_freq_merge['orders'] )
df_freq_merge

,customer_id,orders,userid,view_frequency,frequency
0,5737c6f8ddbec2203f7333f5,2,5737c6f8ddbec2203f7333f5,2,2
1,5737caabddbec220de03ed11,1,5737caabddbec220de03ed11,2,1
2,5737d12addbec222694efd41,7,5737d12addbec222694efd41,7,7
3,573f28f39b0ffc283676efc1,4,573f28f39b0ffc283676efc1,2,2
4,573f28f39b0ffc283676f0b7,6,573f28f39b0ffc283676f0b7,3,3
...,...,...,...,...,...
445715,6834144a78be454173632ec0,2,6834144a78be454173632ec0,1,1
445716,68341469d5f671cf325810b6,4,68341469d5f671cf325810b6,1,1
445717,6834147aa090a409c8747b97,1,6834147aa090a409c8747b97,1,1
445718,683414dc4eef7774748c65a6,2,683414dc4eef7774748c65a6,1,1


#### Merge & Derived Data

In [27]:
df_merge = pd.merge(df_base, 
                    df_customer_engagement, 
                    how='left', 
                    left_on = ['city_name', 'customer_id'],
                    right_on = ['city', 'userid']
                   )

df_merge = pd.merge(df_merge, 
                    df_marketplace, 
                    how='left', 
                    left_on = ['city_name', 'pickup_location_hex_8', 'time_bucket'],
                    right_on = ['city_name', 'pickup_location_hex_8', 'time_bucket']
                   )

df_merge = pd.merge(df_merge, 
                    df_freq_merge[['customer_id', 'frequency']], 
                    how='left', 
                    on = ['customer_id']
                   )

In [28]:
df_merge.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,pickup_location_hex_12,drop_location_hex_12,pickup_location_hex_8,drop_location_hex_8,channel_host,time_bucket,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view,gross_orders,accepted_orders,aor,mp_tag,frequency
0,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,6210585d2a4d42590ef0a5b6,Bangalore,Auto,6833f6e9d1542e19333dea7e,dropped,NaN,1.748236e+12,1.748236e+12,NaN,NaN,NaN,7.0,1748236009534,1030,103649,1.45,0.926,8c61892602f3dff,8c6189260621dff,8861892603fffff,8861892607fffff,android,morning_peak,Bangalore,6210585d2a4d42590ef0a5b6,4.0,6210585d2a4d42590ef0a5b6,None,6210585d2a4d42590ef0a5b6,None,None,yes,yes,7993.0,6470.0,80.95,1. Good,4.0


In [29]:
df_merge['frequency'] = df_merge['frequency'].fillna(0)
df_merge['mp_tag'] = df_merge['mp_tag'].fillna('4. Undefine')
df_merge.mp_tag.unique()

array(['1. Good', '3. Poor', '2. Balance', '4. Undefine'], dtype=object)

In [30]:
print(df_merge.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23029783 entries, 0 to 23029782
Data columns (total 47 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   experiment_name            object 
 1   execution_name             object 
 2   sample_category            object 
 3   taxi_ltr_segment           object 
 4   taxi_retention_segments    object 
 5   taxi_regularity_segment    object 
 6   taxi_income_segment        object 
 7   taxi_intent_segment        object 
 8   yyyymmdd                   object 
 9   customer_id                object 
 10  city_name                  object 
 11  service_name               object 
 12  order_id                   object 
 13  modified_order_status      object 
 14  customer_cancelled_epoch   float64
 15  order_requested_epoch      float64
 16  accepted_epoch             float64
 17  accept_to_cancelled        float64
 18  cobra_ttc                  float64
 19  ocara_ttc                  float64
 20  

In [31]:
fm_conds = [
    df_merge['accept_to_pickup_distance'].isna(),
    (df_merge['accept_to_pickup_distance'] >= 0) & (df_merge['accept_to_pickup_distance'] < 0.5),
    (df_merge['accept_to_pickup_distance'] >= 0.5) & (df_merge['accept_to_pickup_distance'] < 1),
    (df_merge['accept_to_pickup_distance'] >= 1) & (df_merge['accept_to_pickup_distance'] < 2),
    (df_merge['accept_to_pickup_distance'] >= 2)
]

fm_choices = ['NA', '0-500m', '0.5-1km', '1-2km', '2+km']

df_merge['fm_bin'] = np.select(fm_conds, fm_choices, default='NA')

In [32]:
lm_conds = [
    df_merge['distance_final_distance'].isna(),
    (df_merge['distance_final_distance'] >= 0) & (df_merge['distance_final_distance'] < 3),
    (df_merge['distance_final_distance'] >= 3) & (df_merge['distance_final_distance'] < 6),
    (df_merge['distance_final_distance'] >= 6) & (df_merge['distance_final_distance'] < 9),
    (df_merge['distance_final_distance'] >= 9)
]

lm_choices = ['NA', '0-3km', '3-6km', '6-9km', '9+km']

df_merge['lm_bin'] = np.select(lm_conds, lm_choices, default='NA')

In [33]:
true_test_cols = ['dispatch_viewed_cux', 'dispatch_clicked_cux', 'otw_viewed_cux', 'otw_clicked_cux', 'both_view']

for col in true_test_cols:
    tag_col = f'{col}_tag'
    df_merge[tag_col] = 'test'  # default
    df_merge.loc[df_merge['sample_category'] == 'control', tag_col] = 'control'
    df_merge.loc[(df_merge['sample_category'] == 'test') & df_merge[col].notnull(), tag_col] = 'true test'

In [34]:
df_merge['hex_8_route'] = df_merge['pickup_location_hex_8'] + ' - ' + df_merge['drop_location_hex_8']

In [35]:
df_merge.head()

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,channel_host,time_bucket,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view,gross_orders,accepted_orders,aor,mp_tag,frequency,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_view_tag,hex_8_route
0,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,6210585d2a4d42590ef0a5b6,Bangalore,Auto,6833f6e9d1542e19333dea7e,dropped,NaN,1.748236e+12,1.748236e+12,NaN,NaN,NaN,7.0,1748236009534,1030,103649,1.45,...,android,morning_peak,Bangalore,6210585d2a4d42590ef0a5b6,4.0,6210585d2a4d42590ef0a5b6,None,6210585d2a4d42590ef0a5b6,None,None,yes,yes,7993.0,6470.0,80.95,1. Good,4.0,0.5-1km,0-3km,true test,test,true test,test,true test,8861892603fffff - 8861892607fffff
1,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,6210585d2a4d42590ef0a5b6,Bangalore,Auto,68343935331c81339e8ec700,dropped,NaN,1.748253e+12,1.748253e+12,NaN,NaN,NaN,11.0,1748252981670,1515,151941,1.35,...,android,afternoon,Bangalore,6210585d2a4d42590ef0a5b6,4.0,6210585d2a4d42590ef0a5b6,None,6210585d2a4d42590ef0a5b6,None,None,yes,yes,2052.0,1832.0,89.28,1. Good,4.0,1-2km,0-3km,true test,test,true test,test,true test,8861892607fffff - 8861892603fffff
2,loyalty_experiment2,loyalty_experiment2_bangalore,test,None,None,None,None,None,20250526,68309486e8018bf98dcbd493,Bangalore,CabAC,683454c3b6b8e95d270ee7bc,COBRA,1.748260e+12,1.748260e+12,NaN,NaN,347.0,NaN,NaN,1748260035419,1715,171715,20.50,...,ios,evening_peak,Bangalore,68309486e8018bf98dcbd493,7.0,68309486e8018bf98dcbd493,None,68309486e8018bf98dcbd493,None,None,yes,yes,67.0,20.0,29.85,3. Poor,7.0,NA,9+km,true test,test,true test,test,true test,8860145231fffff - 88618924e3fffff
3,loyalty_experiment2,loyalty_experiment2_bangalore,test,None,None,None,None,None,20250526,68309486e8018bf98dcbd493,Bangalore,CabAC,683456f7fc8a9e7d71e59f65,COBRM,1.748261e+12,1.748261e+12,NaN,NaN,NaN,NaN,NaN,1748260599261,1715,172639,20.50,...,ios,evening_peak,Bangalore,68309486e8018bf98dcbd493,7.0,68309486e8018bf98dcbd493,None,68309486e8018bf98dcbd493,None,None,yes,yes,67.0,20.0,29.85,3. Poor,7.0,NA,9+km,true test,test,true test,test,true test,8860145231fffff - 88618924e3fffff
4,loyalty_experiment2,loyalty_experiment2_bangalore,test,None,None,None,None,None,20250526,68309486e8018bf98dcbd493,Bangalore,CabAC,68349bac7bcea02d97af5668,dropped,NaN,1.748278e+12,1.748278e+12,NaN,NaN,NaN,5.0,1748278188814,2215,221948,6.46,...,ios,rest_evening,Bangalore,68309486e8018bf98dcbd493,7.0,68309486e8018bf98dcbd493,None,68309486e8018bf98dcbd493,None,None,yes,yes,1772.0,1319.0,74.44,1. Good,7.0,1-2km,6-9km,true test,test,true test,test,true test,88618924e3fffff - 8861892591fffff


In [36]:
df_merge['updated_segment'] = np.where(df_merge['sample_category'] == 'control', '1. control',
                                np.where((df_merge['clicked'] == 'yes'), '5. clicked', 
                                np.where((df_merge['both_view'] == 'yes'), '4. both_viewed',
                                np.where((df_merge['single_view'] != 'yes'), '3. viewed', '2. not viewed'))))

In [37]:
df_merge.groupby(['sample_category', 'updated_segment'],  dropna=False).agg(cus=('customer_id', 'nunique')).reset_index()

,sample_category,updated_segment,cus
0,control,1. control,5627018
1,test,2. not viewed,104533
2,test,3. viewed,175742
3,test,4. both_viewed,340754
4,test,5. clicked,433


In [38]:
df_merge.head(2)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,time_bucket,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view,gross_orders,accepted_orders,aor,mp_tag,frequency,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_view_tag,hex_8_route,updated_segment
0,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,6210585d2a4d42590ef0a5b6,Bangalore,Auto,6833f6e9d1542e19333dea7e,dropped,NaN,1.748236e+12,1.748236e+12,NaN,NaN,NaN,7.0,1748236009534,1030,103649,1.45,...,morning_peak,Bangalore,6210585d2a4d42590ef0a5b6,4.0,6210585d2a4d42590ef0a5b6,None,6210585d2a4d42590ef0a5b6,None,None,yes,yes,7993.0,6470.0,80.95,1. Good,4.0,0.5-1km,0-3km,true test,test,true test,test,true test,8861892603fffff - 8861892607fffff,4. both_viewed
1,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,6210585d2a4d42590ef0a5b6,Bangalore,Auto,68343935331c81339e8ec700,dropped,NaN,1.748253e+12,1.748253e+12,NaN,NaN,NaN,11.0,1748252981670,1515,151941,1.35,...,afternoon,Bangalore,6210585d2a4d42590ef0a5b6,4.0,6210585d2a4d42590ef0a5b6,None,6210585d2a4d42590ef0a5b6,None,None,yes,yes,2052.0,1832.0,89.28,1. Good,4.0,1-2km,0-3km,true test,test,true test,test,true test,8861892607fffff - 8861892603fffff,4. both_viewed


In [39]:
print(df_merge.order_id.nunique())
print(df_merge.shape)

22830649
(23029783, 56)


In [40]:
df_merge.drop_duplicates(inplace=True)

In [41]:
print(df_merge.order_id.nunique())
print(df_merge.shape)

22830649
(22830649, 56)


In [42]:
# df_merge.to_parquet('df_merge.parquet')

In [8]:
%%time 
df_merge_all = pd.read_parquet('df_merge.parquet')

CPU times: user 2min 18s, sys: 1min 37s, total: 3min 56s
Wall time: 2min 54s


In [9]:
df_merge = df_merge_all[df_merge_all['city_name'].isin(['Delhi'])]
del df_merge_all

In [10]:
df_merge.to_parquet('delhi_df_merge.parquet')

In [39]:
comparison_df = pd.read_parquet('comparison_df.parquet')
comparison_df = comparison_df[['city_name', 'customer_id', 'group_type']]

In [40]:
df_unique = comparison_df.drop_duplicates(subset=['city_name', 'customer_id'], keep='first')

In [41]:
df_unique.shape

(1845397, 3)

In [56]:
df_unique

,city_name,customer_id,group_type
0,Bangalore,6629e62b0f0280c840475ac8,test_viewed
2,Bangalore,5f8fc97171c31f842f73cdf8,test_viewed
4,Bangalore,65255276cd9f7a8ddd08127b,test_viewed
5,Bangalore,6351350123d821098b390697,test_viewed
10,Bangalore,629c67550e8e58683ae26eab,test_viewed
...,...,...,...
3461502,Hyderabad,5daf198525a28c419c9939cd,control_matched
3461503,Hyderabad,5d46af32f3dbe16ba31ed53d,control_matched
3461504,Hyderabad,61cda8a8b387070a80d87c01,control_matched
3461506,Hyderabad,5d3fca499996733edb0829cf,control_matched


In [54]:
df_merge.to_parquet('df_merge.parquet')

In [57]:
df_unique.to_parquet('df_unique.parquet')

In [58]:
df_unique = pd.read_parquet('df_unique.parquet')

In [55]:
df_merge = pd.read_parquet('df_merge.parquet')

In [59]:
df_merge = pd.merge(df_merge,df_unique,how='left',on=['city_name','customer_id'])

In [60]:
df_merge.head()

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view,gross_orders,accepted_orders,aor,mp_tag,frequency,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_view_tag,hex_8_route,updated_segment,group_type
0,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,62172a2c23673083381cdfb6,Bangalore,Auto,6833de7ee973564ad5b917ca,COBRA,1.748230e+12,1.748230e+12,NaN,NaN,180.0,NaN,NaN,1748229758846,0845,085238,3.826,...,Bangalore,62172a2c23673083381cdfb6,15.0,62172a2c23673083381cdfb6,None,62172a2c23673083381cdfb6,None,None,yes,yes,2958.0,1359.0,45.94,3. Poor,10.0,NA,3-6km,true test,test,true test,test,true test,8861892453fffff - 8861892403fffff,4. both_viewed,test_viewed
1,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,62172a2c23673083381cdfb6,Bangalore,Auto,6833e0a44d3c4872aed07678,dropped,NaN,1.748230e+12,1.748230e+12,NaN,NaN,NaN,NaN,1748230308104,0900,090148,3.830,...,Bangalore,62172a2c23673083381cdfb6,15.0,62172a2c23673083381cdfb6,None,62172a2c23673083381cdfb6,None,None,yes,yes,2958.0,1359.0,45.94,3. Poor,10.0,0.5-1km,3-6km,true test,test,true test,test,true test,8861892453fffff - 8861892403fffff,4. both_viewed,test_viewed
2,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,62172a2c23673083381cdfb6,Bangalore,Link,6833df37c5edd61f9f31cc88,expired,NaN,1.748230e+12,NaN,NaN,NaN,NaN,NaN,1748229943606,0845,085543,3.010,...,Bangalore,62172a2c23673083381cdfb6,15.0,62172a2c23673083381cdfb6,None,62172a2c23673083381cdfb6,None,None,yes,yes,2958.0,1359.0,45.94,3. Poor,10.0,NA,3-6km,true test,test,true test,test,true test,8861892453fffff - 8861892403fffff,4. both_viewed,test_viewed
3,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,UNKNOWN,HIGH,20250526,629827474894f9482d2f4fdb,Bangalore,Auto,68347f9d5bb01a68ae0fc03c,dropped,NaN,1.748271e+12,1.748271e+12,NaN,NaN,NaN,NaN,1748271005316,2015,202005,2.850,...,Bangalore,629827474894f9482d2f4fdb,7.0,629827474894f9482d2f4fdb,None,629827474894f9482d2f4fdb,None,None,yes,yes,6210.0,4388.0,70.66,1. Good,6.0,0-500m,0-3km,true test,test,true test,test,true test,886016966bfffff - 8860169641fffff,4. both_viewed,test_viewed
4,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,UNKNOWN,HIGH,20250526,629827474894f9482d2f4fdb,Bangalore,Auto,68340774bc04203860abab7c,COBRA,1.748240e+12,1.748240e+12,NaN,NaN,140.0,NaN,NaN,1748240244628,1145,114724,2.583,...,Bangalore,629827474894f9482d2f4fdb,7.0,629827474894f9482d2f4fdb,None,629827474894f9482d2f4fdb,None,None,yes,yes,747.0,614.0,82.20,1. Good,6.0,NA,0-3km,true test,test,true test,test,true test,8860169641fffff - 886016966bfffff,4. both_viewed,test_viewed


In [61]:
print(df_merge.shape)
print(df_merge.order_id.nunique())

(22830649, 57)
22830649


### User-Defined Function

In [11]:
def get_funnel(df, grouper):
    start = time.time()
    df_analysis =  df\
    .groupby(grouper)\
    .agg(gross_customer = ('customer_id', 'nunique'),
         gross_orders = ('order_id', 'nunique'),
         net_orders = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'dropped'].nunique()),
         cobra = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'COBRA'].nunique()),
         ocara = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'OCARA'].nunique()),
         cobrm = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'COBRM'].nunique()),
         expired = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'expired'].nunique()),
         aborted = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'aborted'].nunique()),
         
         cobra_ttc_mean = ('cobra_ttc', 'mean'),
         cobra_ttc_p25=('cobra_ttc', lambda x: x.quantile(0.25)),
         cobra_ttc_p50 = ('cobra_ttc', 'median'),
         cobra_ttc_p75=('cobra_ttc', lambda x: x.quantile(0.75)),
         cobra_ttc_p90=('cobra_ttc', lambda x: x.quantile(0.90)),
         # cobra_ttc_p95=('cobra_ttc', lambda x: x.quantile(0.95)),
    
         ocara_ttc_mean = ('ocara_ttc', 'mean'),
         ocara_ttc_p25=('ocara_ttc', lambda x: x.quantile(0.25)),
         ocara_ttc_p50 = ('ocara_ttc', 'median'),
         ocara_ttc_p75=('ocara_ttc', lambda x: x.quantile(0.75)),
         ocara_ttc_p90=('ocara_ttc', lambda x: x.quantile(0.90)),
         # ocara_ttc_p95=('ocara_ttc', lambda x: x.quantile(0.95)),

         tta_mean = ('tta', 'mean'),
         tta_p25=('tta', lambda x: x.quantile(0.25)),
         tta_p50=('tta', 'median'),
         tta_p75=('tta', lambda x: x.quantile(0.75)),
         tta_p90=('tta', lambda x: x.quantile(0.90)),
         # tta_p95=('tta', lambda x: x.quantile(0.95)),


         
        )\
    .reset_index()

    df_analysis['cobra_ttc_mean'] = (df_analysis['cobra_ttc_mean']).round(1)
    df_analysis['ocara_ttc_mean'] = (df_analysis['ocara_ttc_mean']).round(1)
    df_analysis['tta_mean'] = (df_analysis['tta_mean']).round(1)
    
    df_analysis['g2n %'] = (df_analysis['net_orders']*100.00/df_analysis['gross_orders']).round(2)
    
    df_analysis['cobra %'] = (df_analysis['cobra']*100.00/df_analysis['gross_orders']).round(2)
    df_analysis['ocara %'] = (df_analysis['ocara']*100.00/df_analysis['gross_orders']).round(2)
    df_analysis['cobrm %'] = (df_analysis['cobrm']*100.00/df_analysis['gross_orders']).round(2)
    df_analysis['expired %'] = ((df_analysis['expired'] + df_analysis['aborted'] )*100.00/df_analysis['gross_orders']).round(2)


    columns = grouper + ['gross_customer', 'gross_orders', 'net_orders', 'g2n %', 'cobra %', 'ocara %', 'cobrm %', 'expired %',
                         'cobra_ttc_mean', 'ocara_ttc_mean', 'tta_mean',
                         'cobra_ttc_p25', 'cobra_ttc_p50', 'cobra_ttc_p75', 'cobra_ttc_p90', # 'cobra_ttc_p95', 
                         'ocara_ttc_p25', 'ocara_ttc_p50', 'ocara_ttc_p75', 'ocara_ttc_p90', # 'ocara_ttc_p95', 
                         'tta_p25', 'tta_p50', 'tta_p75', 'tta_p90', # 'tta_p95', 
                         # 'cobrm', 'cobra', 'ocara', 'expired', 'aborted'
                        ]
    end = time.time()
    print(f"Execution time: {end - start:.3f} seconds")
    return df_analysis[columns]

## Analysis

In [63]:
df_merge.head()

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view,gross_orders,accepted_orders,aor,mp_tag,frequency,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_view_tag,hex_8_route,updated_segment,group_type
0,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,62172a2c23673083381cdfb6,Bangalore,Auto,6833de7ee973564ad5b917ca,COBRA,1.748230e+12,1.748230e+12,NaN,NaN,180.0,NaN,NaN,1748229758846,0845,085238,3.826,...,Bangalore,62172a2c23673083381cdfb6,15.0,62172a2c23673083381cdfb6,None,62172a2c23673083381cdfb6,None,None,yes,yes,2958.0,1359.0,45.94,3. Poor,10.0,NA,3-6km,true test,test,true test,test,true test,8861892453fffff - 8861892403fffff,4. both_viewed,test_viewed
1,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,62172a2c23673083381cdfb6,Bangalore,Auto,6833e0a44d3c4872aed07678,dropped,NaN,1.748230e+12,1.748230e+12,NaN,NaN,NaN,NaN,1748230308104,0900,090148,3.830,...,Bangalore,62172a2c23673083381cdfb6,15.0,62172a2c23673083381cdfb6,None,62172a2c23673083381cdfb6,None,None,yes,yes,2958.0,1359.0,45.94,3. Poor,10.0,0.5-1km,3-6km,true test,test,true test,test,true test,8861892453fffff - 8861892403fffff,4. both_viewed,test_viewed
2,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,HIGH_INCOME,HIGH,20250526,62172a2c23673083381cdfb6,Bangalore,Link,6833df37c5edd61f9f31cc88,expired,NaN,1.748230e+12,NaN,NaN,NaN,NaN,NaN,1748229943606,0845,085543,3.010,...,Bangalore,62172a2c23673083381cdfb6,15.0,62172a2c23673083381cdfb6,None,62172a2c23673083381cdfb6,None,None,yes,yes,2958.0,1359.0,45.94,3. Poor,10.0,NA,3-6km,true test,test,true test,test,true test,8861892453fffff - 8861892403fffff,4. both_viewed,test_viewed
3,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,UNKNOWN,HIGH,20250526,629827474894f9482d2f4fdb,Bangalore,Auto,68347f9d5bb01a68ae0fc03c,dropped,NaN,1.748271e+12,1.748271e+12,NaN,NaN,NaN,NaN,1748271005316,2015,202005,2.850,...,Bangalore,629827474894f9482d2f4fdb,7.0,629827474894f9482d2f4fdb,None,629827474894f9482d2f4fdb,None,None,yes,yes,6210.0,4388.0,70.66,1. Good,6.0,0-500m,0-3km,true test,test,true test,test,true test,886016966bfffff - 8860169641fffff,4. both_viewed,test_viewed
4,loyalty_experiment2,loyalty_experiment2_bangalore,test,PHH,ELITE,DAILY,UNKNOWN,HIGH,20250526,629827474894f9482d2f4fdb,Bangalore,Auto,68340774bc04203860abab7c,COBRA,1.748240e+12,1.748240e+12,NaN,NaN,140.0,NaN,NaN,1748240244628,1145,114724,2.583,...,Bangalore,629827474894f9482d2f4fdb,7.0,629827474894f9482d2f4fdb,None,629827474894f9482d2f4fdb,None,None,yes,yes,747.0,614.0,82.20,1. Good,6.0,NA,0-3km,true test,test,true test,test,true test,8860169641fffff - 886016966bfffff,4. both_viewed,test_viewed


### Daily Trend

In [76]:
df_daily_city_yyyymmdd = get_funnel(df_merge, ['city_name','yyyymmdd'])
df_daily_city_yyyymmdd

Execution time: 85.757 seconds


,city_name,yyyymmdd,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Bangalore,20250520,516518,1183405,514146,43.45,29.08,12.43,1.16,13.89,135.1,244.3,119.5,58.0,111.0,193.0,275.0,47.0,137.0,323.0,582.0,8.0,32.0,138.0,283.0
1,Bangalore,20250521,480407,979933,543044,55.42,20.77,14.55,1.08,8.19,121.7,231.2,112.6,52.0,100.0,173.0,252.0,43.0,127.0,308.0,560.0,8.0,28.0,122.0,267.0
2,Bangalore,20250522,485998,991143,556802,56.18,20.37,14.52,1.02,7.91,121.3,225.2,111.3,52.0,99.0,172.0,252.0,43.0,124.0,301.0,547.0,8.0,27.0,120.0,267.0
3,Bangalore,20250523,499242,1023917,573224,55.98,20.61,14.74,1.06,7.60,120.5,224.4,109.6,51.0,99.0,172.0,250.0,41.0,122.0,299.0,546.0,8.0,27.0,120.0,266.0
4,Bangalore,20250524,504817,1046471,570473,54.51,22.09,15.51,0.52,7.35,121.1,231.7,113.9,52.0,98.0,171.0,251.0,44.0,126.0,308.0,564.0,8.0,28.0,124.0,270.0
5,Bangalore,20250525,473980,1033641,505493,48.90,26.13,14.73,0.64,9.57,127.9,235.0,114.5,55.0,104.0,181.0,264.0,45.0,130.0,313.0,566.0,8.0,28.0,125.0,274.0
6,Bangalore,20250526,571692,1382051,554638,40.13,31.62,12.13,1.12,15.00,137.5,236.7,119.9,60.0,114.0,196.0,278.0,47.0,135.0,314.0,564.0,9.0,38.0,148.0,281.0
7,Delhi,20250520,627011,1187093,716519,60.36,16.88,16.71,0.74,5.31,129.4,271.1,120.0,52.0,103.0,182.0,275.0,51.0,155.0,363.0,638.0,7.0,18.0,110.0,295.0
8,Delhi,20250521,668392,1326891,720285,54.28,20.81,16.80,0.72,7.38,134.0,290.4,129.0,55.0,108.0,188.0,281.0,56.0,168.0,383.0,670.0,7.0,24.0,121.0,313.0
9,Delhi,20250522,605838,1098715,707061,64.35,14.20,16.33,0.70,4.43,127.3,264.3,117.9,51.0,101.0,179.0,272.0,48.0,150.0,355.0,625.0,6.0,16.0,103.0,290.0


- Excluded few days because of huge variations (2025-05-16 to 19)

In [77]:
df_daily_city_yyyymmdd_group_tc = get_funnel(df_merge, ['city_name','yyyymmdd','sample_category'])
df_daily_city_yyyymmdd_group_tc

Execution time: 80.008 seconds


,city_name,yyyymmdd,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Bangalore,20250520,control,465742,1068227,463437,43.38,29.22,12.42,1.17,13.81,135.1,243.8,119.6,58.0,111.0,193.0,275.0,47.0,137.0,323.00,581.0,8.0,32.0,138.00,283.0
1,Bangalore,20250520,test,50776,115178,50709,44.03,27.74,12.49,1.11,14.63,135.4,248.5,118.4,57.0,111.0,193.0,276.0,47.0,141.0,331.00,591.0,8.0,31.0,138.00,285.0
2,Bangalore,20250521,control,432816,883161,489027,55.37,20.83,14.57,1.08,8.15,121.6,231.5,112.6,52.0,100.0,173.0,252.0,43.0,127.0,308.00,560.0,8.0,27.0,122.00,267.0
3,Bangalore,20250521,test,47591,96772,54017,55.82,20.23,14.38,1.03,8.54,122.2,228.4,112.0,52.0,101.0,173.0,253.0,43.0,126.0,307.00,561.0,8.0,28.0,121.00,270.0
4,Bangalore,20250522,control,437998,893316,501768,56.17,20.43,14.53,1.02,7.86,121.3,225.6,111.0,52.0,99.0,172.0,252.0,43.0,124.0,302.00,548.0,8.0,27.0,120.00,267.0
5,Bangalore,20250522,test,48000,97827,55034,56.26,19.84,14.44,1.07,8.39,121.6,221.8,114.0,52.0,100.0,173.0,253.0,43.0,122.0,296.00,544.0,8.0,27.0,121.25,267.0
6,Bangalore,20250523,control,449857,922254,515976,55.95,20.66,14.74,1.08,7.57,120.6,224.6,109.2,51.0,99.0,172.0,250.0,41.0,122.0,300.00,546.0,8.0,26.0,120.00,265.0
7,Bangalore,20250523,test,49385,101663,57248,56.31,20.12,14.75,0.94,7.88,120.1,222.3,114.1,51.0,99.0,170.0,248.0,41.0,122.0,296.00,544.0,8.0,28.0,126.00,270.0
8,Bangalore,20250524,control,454795,942780,514152,54.54,22.10,15.52,0.52,7.30,121.0,231.4,113.6,52.0,98.0,171.0,251.0,44.0,126.0,308.00,563.0,8.0,28.0,124.00,270.0
9,Bangalore,20250524,test,50022,103691,56321,54.32,21.93,15.43,0.53,7.77,121.6,234.6,116.3,52.0,98.0,172.0,252.0,43.0,123.0,306.00,572.0,8.0,28.0,129.00,274.0


In [78]:
df_daily_city_yyyymmdd_group_tc_dispatch_click = get_funnel(df_merge, ['city_name','yyyymmdd','sample_category','dispatch_clicked_cux_tag'])
df_daily_city_yyyymmdd_group_tc_dispatch_view = get_funnel(df_merge, ['city_name','yyyymmdd','sample_category','dispatch_viewed_cux_tag'])
display(df_daily_city_yyyymmdd_group_tc_dispatch_click)
display(df_daily_city_yyyymmdd_group_tc_dispatch_view)

Execution time: 84.931 seconds
Execution time: 83.272 seconds


,city_name,yyyymmdd,sample_category,dispatch_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Bangalore,20250520,control,control,465742,1068227,463437,43.38,29.22,12.42,1.17,13.81,135.1,243.8,119.6,58.00,111.0,193.00,275.0,47.00,137.0,323.00,581.0,8.0,32.0,138.00,283.0
1,Bangalore,20250520,test,test,49882,112114,49782,44.40,27.59,12.51,1.11,14.40,134.7,249.1,118.1,57.00,110.0,192.00,275.0,47.00,142.0,332.00,593.0,8.0,31.0,137.00,284.0
2,Bangalore,20250520,test,true test,894,3064,927,30.25,33.45,11.62,1.37,23.30,157.9,226.5,131.5,67.00,134.0,226.00,317.0,39.75,113.0,287.00,484.5,8.0,51.5,167.50,302.5
3,Bangalore,20250521,control,control,432816,883161,489027,55.37,20.83,14.57,1.08,8.15,121.6,231.5,112.6,52.00,100.0,173.00,252.0,43.00,127.0,308.00,560.0,8.0,27.0,122.00,267.0
4,Bangalore,20250521,test,test,46769,94636,52971,55.97,20.20,14.36,1.02,8.45,122.0,228.8,111.6,52.00,101.0,172.00,252.0,43.00,127.0,308.00,564.0,8.0,28.0,121.00,270.0
5,Bangalore,20250521,test,true test,822,2136,1046,48.97,21.82,15.22,1.54,12.45,131.3,212.2,125.5,58.25,112.5,182.75,264.0,38.00,106.0,241.00,485.2,9.0,56.0,140.00,280.0
6,Bangalore,20250522,control,control,437998,893316,501768,56.17,20.43,14.53,1.02,7.86,121.3,225.6,111.0,52.00,99.0,172.00,252.0,43.00,124.0,302.00,548.0,8.0,27.0,120.00,267.0
7,Bangalore,20250522,test,test,47180,95570,54009,56.51,19.76,14.44,1.06,8.23,121.1,222.3,114.0,52.00,100.0,172.00,251.0,43.00,122.0,297.00,545.0,8.0,26.0,121.00,266.0
8,Bangalore,20250522,test,true test,820,2257,1025,45.41,23.17,14.71,1.42,15.29,138.3,200.6,112.8,56.50,117.0,195.00,292.8,38.75,118.0,282.50,501.0,11.0,40.5,145.00,289.5
9,Bangalore,20250523,control,control,449857,922254,515976,55.95,20.66,14.74,1.08,7.57,120.6,224.6,109.2,51.00,99.0,172.00,250.0,41.00,122.0,300.00,546.0,8.0,26.0,120.00,265.0


,city_name,yyyymmdd,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Bangalore,20250520,control,control,465742,1068227,463437,43.38,29.22,12.42,1.17,13.81,135.1,243.8,119.6,58.00,111.0,193.00,275.0,47.0,137.0,323.00,581.0,8.0,32.0,138.00,283.0
1,Bangalore,20250520,test,test,10356,16862,9041,53.62,20.44,15.27,1.10,9.57,123.4,267.0,130.7,51.00,99.0,173.00,256.0,49.0,151.0,367.00,645.0,7.0,27.0,128.00,288.6
2,Bangalore,20250520,test,true test,40420,98316,41668,42.38,28.99,12.01,1.11,15.50,136.9,244.5,115.7,58.00,113.0,195.00,278.0,47.0,139.0,325.00,581.0,8.0,32.0,141.00,284.0
3,Bangalore,20250521,control,control,432816,883161,489027,55.37,20.83,14.57,1.08,8.15,121.6,231.5,112.6,52.00,100.0,173.00,252.0,43.0,127.0,308.00,560.0,8.0,27.0,122.00,267.0
4,Bangalore,20250521,test,test,10137,16635,9036,54.32,20.44,15.50,1.19,8.54,119.9,249.4,120.8,52.00,99.0,169.00,249.0,52.0,139.0,334.00,621.8,8.0,29.0,121.00,288.2
5,Bangalore,20250521,test,true test,37454,80137,44981,56.13,20.19,14.15,1.00,8.53,122.7,223.6,110.0,52.00,101.0,173.00,254.0,42.0,123.0,301.00,546.5,8.0,28.0,122.00,267.0
6,Bangalore,20250522,control,control,437998,893316,501768,56.17,20.43,14.53,1.02,7.86,121.3,225.6,111.0,52.00,99.0,172.00,252.0,43.0,124.0,302.00,548.0,8.0,27.0,120.00,267.0
7,Bangalore,20250522,test,test,10293,16548,9288,56.13,18.57,16.23,1.12,7.95,115.7,241.1,130.8,50.00,97.0,162.00,241.8,45.0,127.0,306.00,586.5,8.0,28.0,126.75,294.0
8,Bangalore,20250522,test,true test,37707,81279,45746,56.28,20.10,14.08,1.06,8.48,122.7,217.3,110.1,52.00,101.0,175.00,255.0,42.0,120.5,293.00,532.0,8.0,26.0,121.00,261.0
9,Bangalore,20250523,control,control,449857,922254,515976,55.95,20.66,14.74,1.08,7.57,120.6,224.6,109.2,51.00,99.0,172.00,250.0,41.0,122.0,300.00,546.0,8.0,26.0,120.00,265.0


In [79]:
df_daily_city_yyyymmdd_group_tc_otw_click = get_funnel(df_merge, ['city_name','yyyymmdd','sample_category','otw_clicked_cux_tag'])
df_daily_city_yyyymmdd_group_tc_otw_view = get_funnel(df_merge, ['city_name','yyyymmdd','sample_category','otw_viewed_cux_tag'])
display(df_daily_city_yyyymmdd_group_tc_otw_click)
display(df_daily_city_yyyymmdd_group_tc_otw_view)

Execution time: 83.764 seconds
Execution time: 83.399 seconds


,city_name,yyyymmdd,sample_category,otw_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Bangalore,20250520,control,control,465742,1068227,463437,43.38,29.22,12.42,1.17,13.81,135.1,243.8,119.6,58.00,111.0,193.00,275.0,47.00,137.0,323.0,581.0,8.0,32.0,138.00,283.0
1,Bangalore,20250520,test,test,50022,113097,49612,43.87,27.87,12.45,1.12,14.70,135.2,248.2,118.5,57.00,111.0,193.00,276.0,47.00,141.0,331.0,591.0,8.0,31.0,137.00,285.0
2,Bangalore,20250520,test,true test,754,2081,1097,52.72,20.90,14.70,0.43,11.24,147.4,261.3,114.3,67.00,130.0,208.00,293.6,45.25,134.5,327.0,574.5,8.0,51.0,152.75,278.0
3,Bangalore,20250521,control,control,432816,883161,489027,55.37,20.83,14.57,1.08,8.15,121.6,231.5,112.6,52.00,100.0,173.00,252.0,43.00,127.0,308.0,560.0,8.0,27.0,122.00,267.0
4,Bangalore,20250521,test,test,46789,94794,52803,55.70,20.36,14.34,1.04,8.56,122.3,229.2,112.5,52.00,101.0,173.00,253.0,44.00,127.0,308.0,561.0,8.0,28.0,122.00,271.0
5,Bangalore,20250521,test,true test,802,1978,1214,61.38,14.36,16.38,0.46,7.43,120.3,192.8,89.2,52.75,97.0,171.00,249.0,24.75,81.5,259.0,488.7,7.0,25.0,112.25,237.1
6,Bangalore,20250522,control,control,437998,893316,501768,56.17,20.43,14.53,1.02,7.86,121.3,225.6,111.0,52.00,99.0,172.00,252.0,43.00,124.0,302.0,548.0,8.0,27.0,120.00,267.0
7,Bangalore,20250522,test,test,47186,95667,53824,56.26,19.90,14.36,1.08,8.39,121.5,221.7,114.5,52.00,100.0,173.00,253.0,43.00,121.0,296.0,544.3,8.0,27.0,122.00,268.0
8,Bangalore,20250522,test,true test,814,2160,1210,56.02,17.04,18.06,0.69,8.19,123.2,226.5,96.8,57.00,107.5,169.25,251.3,41.25,129.5,292.5,523.9,8.0,22.0,106.50,255.1
9,Bangalore,20250523,control,control,449857,922254,515976,55.95,20.66,14.74,1.08,7.57,120.6,224.6,109.2,51.00,99.0,172.00,250.0,41.00,122.0,300.0,546.0,8.0,26.0,120.00,265.0


,city_name,yyyymmdd,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Bangalore,20250520,control,control,465742,1068227,463437,43.38,29.22,12.42,1.17,13.81,135.1,243.8,119.6,58.00,111.0,193.0,275.0,47.00,137.0,323.00,581.0,8.0,32.0,138.00,283.0
1,Bangalore,20250520,test,test,13467,23931,8229,34.39,34.11,10.43,1.48,19.59,137.4,256.2,127.4,58.00,113.0,196.0,280.0,46.00,141.0,350.00,624.0,7.0,28.0,132.00,291.0
2,Bangalore,20250520,test,true test,37309,91247,42480,46.55,26.07,13.03,1.02,13.33,134.7,246.9,116.5,57.00,111.0,192.0,275.0,47.00,141.0,329.00,584.0,8.0,32.0,139.00,284.0
3,Bangalore,20250521,control,control,432816,883161,489027,55.37,20.83,14.57,1.08,8.15,121.6,231.5,112.6,52.00,100.0,173.0,252.0,43.00,127.0,308.00,560.0,8.0,27.0,122.00,267.0
4,Bangalore,20250521,test,test,11529,19206,8994,46.83,25.97,14.14,1.43,11.64,121.9,236.4,112.6,53.00,101.0,172.0,253.0,47.00,133.0,319.00,602.6,8.0,28.0,118.50,260.6
5,Bangalore,20250521,test,true test,36062,77566,45023,58.04,18.81,14.44,0.93,7.77,122.4,226.4,111.8,52.00,101.0,173.0,252.0,43.00,124.5,305.00,552.0,8.0,28.0,122.00,271.0
6,Bangalore,20250522,control,control,437998,893316,501768,56.17,20.43,14.53,1.02,7.86,121.3,225.6,111.0,52.00,99.0,172.0,252.0,43.00,124.0,302.00,548.0,8.0,27.0,120.00,267.0
7,Bangalore,20250522,test,test,11436,18846,9025,47.89,25.09,14.62,1.35,11.05,120.9,238.2,128.7,53.00,100.0,171.0,250.0,46.00,126.0,310.00,577.0,8.0,31.0,135.00,284.2
8,Bangalore,20250522,test,true test,36564,78981,46009,58.25,18.59,14.40,1.01,7.76,121.8,217.8,110.5,52.00,100.0,173.0,254.0,42.00,121.0,293.00,536.0,8.0,26.0,119.00,261.0
9,Bangalore,20250523,control,control,449857,922254,515976,55.95,20.66,14.74,1.08,7.57,120.6,224.6,109.2,51.00,99.0,172.0,250.0,41.00,122.0,300.00,546.0,8.0,26.0,120.00,265.0


In [80]:
df_daily_city_yyyymmdd_group_tc_both_view = get_funnel(df_merge, ['city_name','yyyymmdd','sample_category','both_view_tag'])
display(df_daily_city_yyyymmdd_group_tc_both_view)

Execution time: 83.379 seconds


,city_name,yyyymmdd,sample_category,both_view_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Bangalore,20250520,control,control,465742,1068227,463437,43.38,29.22,12.42,1.17,13.81,135.1,243.8,119.6,58.0,111.0,193.00,275.0,47.00,137.0,323.00,581.0,8.0,32.0,138.00,283.0
1,Bangalore,20250520,test,test,16065,28380,10965,38.64,31.02,11.39,1.34,17.62,136.2,262.4,129.1,57.0,111.0,195.00,279.0,48.00,146.5,352.25,627.7,8.0,30.0,133.00,292.0
2,Bangalore,20250520,test,true test,34711,86798,39744,45.79,26.67,12.84,1.04,13.66,135.1,244.5,115.3,57.0,111.0,193.00,275.0,47.00,139.0,325.00,582.0,8.0,32.0,139.00,283.0
3,Bangalore,20250521,control,control,432816,883161,489027,55.37,20.83,14.57,1.08,8.15,121.6,231.5,112.6,52.0,100.0,173.00,252.0,43.00,127.0,308.00,560.0,8.0,27.0,122.00,267.0
4,Bangalore,20250521,test,test,13519,22608,11139,49.27,24.31,14.27,1.38,10.77,121.6,241.4,118.4,52.0,100.0,171.00,253.0,49.00,135.0,323.00,608.0,8.0,30.0,122.00,276.5
5,Bangalore,20250521,test,true test,34072,74164,42878,57.82,18.99,14.41,0.93,7.85,122.5,224.5,110.0,52.0,101.0,173.00,252.0,42.00,123.0,303.00,549.0,8.0,28.0,121.00,268.2
6,Bangalore,20250522,control,control,437998,893316,501768,56.17,20.43,14.53,1.02,7.86,121.3,225.6,111.0,52.0,99.0,172.00,252.0,43.00,124.0,302.00,548.0,8.0,27.0,120.00,267.0
7,Bangalore,20250522,test,test,13605,22472,11357,50.54,23.04,14.89,1.30,10.23,120.5,236.9,127.9,53.0,100.0,170.00,250.0,45.00,124.5,307.00,578.5,8.0,29.0,132.00,284.0
8,Bangalore,20250522,test,true test,34395,75355,43677,57.96,18.89,14.31,1.00,7.84,121.9,217.1,109.7,52.0,100.0,174.00,254.0,42.00,121.0,293.00,531.0,7.0,26.0,119.00,260.0
9,Bangalore,20250523,control,control,449857,922254,515976,55.95,20.66,14.74,1.08,7.57,120.6,224.6,109.2,51.0,99.0,172.00,250.0,41.00,122.0,300.00,546.0,8.0,26.0,120.00,265.0


### Overall

In [81]:
df_overall = get_funnel(df_merge, ['experiment_name'])
df_overall

Execution time: 81.463 seconds


,experiment_name,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,loyalty_experiment2,6246892,22830649,13100985,57.38,19.47,15.21,0.74,7.19,131.9,258.9,120.5,55.0,106.0,186.0,276.0,49.0,146.0,348.0,616.0,7.0,22.0,117.0,293.0


In [82]:
df_overall_group_tc = get_funnel(df_merge, ['experiment_name', 'sample_category'])
df_overall_group_tc

Execution time: 87.839 seconds


,experiment_name,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,loyalty_experiment2,control,5627018,20573304,11804291,57.38,19.51,15.19,0.74,7.17,131.9,259.0,120.6,55.0,106.0,186.0,275.0,49.0,147.0,348.0,616.0,7.0,22.0,117.0,293.0
1,loyalty_experiment2,test,619874,2257345,1296694,57.44,19.07,15.32,0.72,7.44,131.6,257.1,119.6,53.0,105.0,186.0,278.0,48.0,145.0,346.0,615.0,7.0,21.0,116.0,292.0


In [14]:
df_overall_group_tc_up = get_funnel(df_merge, ['experiment_name', 'updated_segment'])
df_overall_group_tc_up

Execution time: 107.614 seconds


,experiment_name,updated_segment,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,loyalty_experiment2,1. control,5627018,20573304,11804818,57.38,19.51,15.19,0.74,7.17,131.9,259.2,69.6,55.0,106.0,186.0,275.0,49.0,147.0,348.0,616.0,6.0,12.0,64.0,181.0
1,loyalty_experiment2,2. not viewed,104533,269086,129515,48.13,25.41,13.50,1.18,11.78,136.2,266.3,73.9,55.0,110.0,194.0,286.0,50.0,151.0,357.0,641.0,6.0,12.0,68.0,194.0
2,loyalty_experiment2,3. viewed,175742,341452,181555,53.17,21.08,16.52,0.80,8.43,132.4,285.7,79.7,54.0,106.0,186.0,278.0,55.0,163.0,384.0,676.0,6.0,13.0,71.0,202.0
3,loyalty_experiment2,4. both_viewed,340754,1641472,982978,59.88,17.60,15.36,0.63,6.53,130.3,249.5,67.3,53.0,104.0,184.0,275.0,46.0,140.0,337.0,598.0,6.0,12.0,63.0,178.0
4,loyalty_experiment2,5. clicked,433,5335,2713,50.85,19.38,18.39,0.99,10.38,135.6,255.6,87.7,53.0,110.0,193.0,288.7,39.0,130.0,317.0,555.0,6.0,14.0,92.0,237.0


In [17]:
df_overall_group_tc_up.to_clipboard(index=False)

In [18]:
df_city_group_tc_up = get_funnel(df_merge, ['city_name', 'updated_segment'])
df_city_group_tc_up

Execution time: 94.910 seconds


,city_name,updated_segment,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Bangalore,1. control,1689518,6887648,3439542,49.94,24.93,13.96,0.96,10.21,128.4,232.8,71.4,55.0,105.0,182.00,264.0,44.00,129.0,310.00,561.0,6.0,14.0,79.00,192.0
1,Bangalore,2. not viewed,35651,103212,39044,37.83,32.69,11.50,1.34,16.65,135.6,244.3,78.3,57.0,111.0,193.00,277.0,47.00,135.0,319.00,586.0,6.0,16.0,90.00,211.0
2,Bangalore,3. viewed,50653,107152,51601,48.16,25.22,15.02,0.96,10.64,125.8,249.4,79.2,54.0,103.0,177.00,259.0,49.00,135.0,328.00,603.0,6.0,15.0,85.00,206.0
3,Bangalore,4. both_viewed,99346,540365,286772,53.07,22.29,14.20,0.85,9.59,127.5,226.9,70.2,54.0,104.0,181.00,263.0,43.00,126.0,303.00,553.0,6.0,14.0,78.00,191.0
4,Bangalore,5. clicked,176,2184,1054,48.26,22.02,15.75,0.55,13.42,129.8,251.3,76.4,52.0,105.0,188.00,274.0,35.75,120.0,306.25,570.2,6.0,15.0,91.00,215.7
5,Delhi,1. control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.00,273.0,52.00,159.0,370.00,650.0,6.0,11.0,58.00,171.0
6,Delhi,2. not viewed,35173,84997,44853,52.77,20.62,15.63,1.38,9.61,132.2,286.8,70.6,49.0,104.0,190.00,290.0,50.00,164.0,382.00,684.6,6.0,10.0,57.00,174.0
7,Delhi,3. viewed,58412,107677,61400,57.02,15.88,19.56,0.84,6.70,121.4,308.7,77.9,48.0,98.0,172.00,259.0,56.00,178.0,409.00,718.0,6.0,11.0,59.00,184.0
8,Delhi,4. both_viewed,120520,585706,364743,62.27,15.18,16.94,0.67,4.94,126.4,264.4,66.7,49.0,100.0,178.00,274.0,47.00,148.0,355.00,624.0,6.0,11.0,59.00,171.0
9,Delhi,5. clicked,169,2038,1112,54.56,16.49,20.12,0.88,7.95,140.6,279.7,95.2,51.0,118.5,200.25,294.5,35.25,126.0,330.00,576.5,6.0,12.0,81.75,245.1


In [19]:
df_city_group_tc_up.to_clipboard(index=False)

In [83]:
df_overall_group_tc_dispatch_click = get_funnel(df_merge, ['experiment_name','sample_category','dispatch_clicked_cux_tag'])
df_overall_group_tc_dispatch_view = get_funnel(df_merge, ['experiment_name','sample_category','dispatch_viewed_cux_tag'])
display(df_overall_group_tc_dispatch_click)
display(df_overall_group_tc_dispatch_view)

Execution time: 90.063 seconds
Execution time: 91.720 seconds


,experiment_name,sample_category,dispatch_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,loyalty_experiment2,control,control,5627018,20573304,11804291,57.38,19.51,15.19,0.74,7.17,131.9,259.0,120.6,55.0,106.0,186.0,275.0,49.0,147.0,348.0,616.0,7.0,22.0,117.0,293.0
1,loyalty_experiment2,test,test,615464,2218180,1279273,57.67,18.96,15.31,0.71,7.34,131.2,257.7,119.6,53.0,105.0,185.0,277.0,48.0,145.0,347.0,616.0,7.0,21.0,116.0,291.0
2,loyalty_experiment2,test,true test,4448,39165,17421,44.48,25.11,15.94,1.15,13.31,146.7,225.6,118.4,59.0,120.0,209.0,306.0,39.0,120.0,287.0,533.0,8.0,34.0,146.0,305.0


,experiment_name,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,loyalty_experiment2,control,control,5627018,20573304,11804291,57.38,19.51,15.19,0.74,7.17,131.9,259.0,120.6,55.0,106.0,186.0,275.0,49.0,147.0,348.0,616.0,7.0,22.0,117.0,293.0
1,loyalty_experiment2,test,test,209985,424136,235111,55.43,19.49,16.58,0.74,7.73,131.8,283.1,134.8,54.0,105.0,185.0,277.0,55.0,161.0,379.0,672.0,7.0,22.0,123.0,324.0
2,loyalty_experiment2,test,true test,411091,1833209,1061583,57.91,18.97,15.03,0.71,7.38,131.5,250.5,115.8,53.0,105.0,186.0,278.0,46.0,141.0,338.0,600.0,7.0,21.0,115.0,285.0


In [28]:
df_overall_group_tc_otw_click = get_funnel(df_merge, ['experiment_name','sample_category','otw_clicked_cux_tag'])
df_overall_group_tc_otw_view = get_funnel(df_merge, ['experiment_name','sample_category','otw_viewed_cux_tag'])
display(df_overall_group_tc_otw_click)
display(df_overall_group_tc_otw_view)

Execution time: 31.747 seconds
Execution time: 30.204 seconds


,experiment_name,sample_category,otw_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,loyalty_experiment2,control,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.00,103.0,181.00,273.0,52.0,159.0,370.00,650.0,6.0,11.0,58.0,171.0
1,loyalty_experiment2,test,test,212508,766713,463535,60.46,15.92,17.13,0.77,5.72,126.6,273.8,68.7,49.00,100.0,178.00,274.0,49.0,155.0,366.00,645.0,6.0,11.0,59.0,173.0
2,loyalty_experiment2,test,true test,1766,13705,8573,62.55,13.47,19.07,0.38,4.52,126.9,263.5,68.3,46.25,101.0,184.75,274.5,44.0,135.5,336.75,611.7,6.0,10.0,57.0,179.0


,experiment_name,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,loyalty_experiment2,control,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,loyalty_experiment2,test,test,81185,163749,86792,53.00,19.35,17.72,1.17,8.75,127.5,301.3,76.2,49.0,102.0,181.0,276.0,54.0,174.0,402.0,709.0,6.0,11.0,59.0,184.0
2,loyalty_experiment2,test,true test,133089,616669,385316,62.48,14.95,17.02,0.66,4.89,126.2,266.0,66.9,49.0,100.0,178.0,274.0,47.0,149.0,356.0,627.0,6.0,11.0,58.0,170.0


In [29]:
df_overall_group_tc_both_view = get_funnel(df_merge, ['experiment_name','sample_category','both_view'])
display(df_overall_group_tc_both_view)

Execution time: 13.742 seconds


,experiment_name,sample_category,both_view,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,loyalty_experiment2,control,yes,16,137,55,40.15,13.14,45.99,0.00,0.73,64.0,67.0,51.1,22.5,44.5,64.75,140.4,13.0,17.0,31.5,157.8,5.0,7.0,16.75,74.7
1,loyalty_experiment2,test,yes,120683,587728,365844,62.25,15.18,16.95,0.67,4.95,126.5,264.5,66.8,49.0,100.0,178.00,274.0,47.0,148.0,355.0,624.0,6.0,11.0,59.00,171.0


### City

In [30]:
df_city = get_funnel(df_merge, ['city_name'])
df_city

Execution time: 24.191 seconds


,city_name,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,2147980,7835049,4773453,60.92,15.97,16.89,0.78,5.43,128.9,277.8,68.7,52.0,103.0,181.0,274.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0


In [31]:
df_city_group_tc = get_funnel(df_merge, ['city_name', 'sample_category'])
df_city_group_tc

Execution time: 28.759 seconds


,city_name,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.4,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,214274,780418,472108,60.49,15.87,17.16,0.77,5.7,126.6,273.6,68.7,49.0,100.0,178.0,274.0,49.0,154.0,365.0,645.0,6.0,11.0,59.0,173.0


In [32]:
df_city_group_tc_dispatch_click = get_funnel(df_merge, ['city_name','sample_category','dispatch_clicked_cux_tag'])
df_city_group_tc_dispatch_view = get_funnel(df_merge, ['city_name','sample_category','dispatch_viewed_cux_tag'])

Execution time: 29.480 seconds
Execution time: 30.058 seconds


In [33]:
display(df_city_group_tc_dispatch_click)
display(df_city_group_tc_dispatch_view)

,city_name,sample_category,dispatch_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,test,212553,765556,464744,60.71,15.76,17.15,0.76,5.62,126.1,274.3,68.5,49.0,100.0,178.0,273.0,49.0,155.0,366.0,646.0,6.0,11.0,58.0,172.0
2,Delhi,test,true test,1721,14862,7364,49.55,21.69,17.71,1.22,9.83,146.3,242.7,80.5,54.0,119.0,210.0,313.8,39.0,127.0,311.5,558.9,6.0,14.0,85.0,225.0


,city_name,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,test,70818,136618,80872,59.20,14.66,19.32,0.77,6.05,121.2,305.9,75.8,47.0,97.0,172.0,260.0,56.0,175.0,405.0,714.2,6.0,11.0,57.0,179.0
2,Delhi,test,true test,143456,643800,391236,60.77,16.13,16.71,0.77,5.63,127.6,265.7,67.2,50.0,101.0,180.0,277.0,47.0,149.0,357.0,627.0,6.0,11.0,59.0,172.0


In [34]:
df_city_group_tc_otw_click = get_funnel(df_merge, ['city_name','sample_category','otw_clicked_cux_tag'])
df_city_group_tc_otw_view = get_funnel(df_merge, ['city_name','sample_category','otw_viewed_cux_tag'])
display(df_city_group_tc_otw_click)
display(df_city_group_tc_otw_view)

Execution time: 28.797 seconds
Execution time: 29.309 seconds


,city_name,sample_category,otw_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.00,103.0,181.00,273.0,52.0,159.0,370.00,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,test,212508,766713,463535,60.46,15.92,17.13,0.77,5.72,126.6,273.8,68.7,49.00,100.0,178.00,274.0,49.0,155.0,366.00,645.0,6.0,11.0,59.0,173.0
2,Delhi,test,true test,1766,13705,8573,62.55,13.47,19.07,0.38,4.52,126.9,263.5,68.3,46.25,101.0,184.75,274.5,44.0,135.5,336.75,611.7,6.0,10.0,57.0,179.0


,city_name,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,test,81185,163749,86792,53.00,19.35,17.72,1.17,8.75,127.5,301.3,76.2,49.0,102.0,181.0,276.0,54.0,174.0,402.0,709.0,6.0,11.0,59.0,184.0
2,Delhi,test,true test,133089,616669,385316,62.48,14.95,17.02,0.66,4.89,126.2,266.0,66.9,49.0,100.0,178.0,274.0,47.0,149.0,356.0,627.0,6.0,11.0,58.0,170.0


In [35]:
df_city_group_tc_both_view = get_funnel(df_merge, ['city_name','sample_category','both_view_tag'])
df_city_group_tc_both_view

Execution time: 29.175 seconds


,city_name,sample_category,both_view_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,test,93591,192690,106264,55.15,17.97,17.83,1.08,7.98,126.9,300.3,74.9,49.0,101.0,180.0,275.0,54.0,172.0,399.0,707.0,6.0,11.0,58.0,180.0
2,Delhi,test,true test,120683,587728,365844,62.25,15.18,16.95,0.67,4.95,126.5,264.5,66.8,49.0,100.0,178.0,274.0,47.0,148.0,355.0,624.0,6.0,11.0,59.0,171.0


In [36]:
df_city_mp_group_tc_both_view = get_funnel(df_merge, ['city_name', 'mp_tag', 'sample_category','both_view_tag'])
df_city_mp_group_tc_both_view

Execution time: 31.442 seconds


,city_name,mp_tag,sample_category,both_view_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.0,89.0,157.00,245.0,51.00,159.0,371.00,649.0,6.0,9.0,46.0,139.0
1,Delhi,1. Good,test,test,54488,92361,56898,61.60,13.99,18.65,0.49,5.26,116.5,296.7,68.7,44.0,91.0,163.00,256.0,53.00,169.0,399.00,697.0,6.0,9.0,48.0,153.0
2,Delhi,1. Good,test,true test,83786,256729,176799,68.87,10.33,17.56,0.38,2.87,108.6,261.4,57.7,42.0,84.0,149.00,239.0,45.00,147.0,353.00,617.0,6.0,9.0,43.0,132.0
3,Delhi,2. Balance,control,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.0,101.0,176.00,266.0,52.00,159.0,372.00,656.0,6.0,12.0,64.0,179.0
4,Delhi,2. Balance,test,test,40387,67445,36493,54.11,18.89,18.00,0.78,8.22,126.9,305.2,78.2,49.0,103.0,181.00,274.0,55.00,174.0,400.00,717.0,6.0,12.0,64.0,190.0
5,Delhi,2. Balance,test,true test,69583,204703,126163,61.63,15.89,17.13,0.57,4.77,124.1,266.6,69.9,49.0,99.0,175.00,267.0,47.00,150.0,359.75,635.0,6.0,12.0,64.0,180.0
6,Delhi,3. Poor,control,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.0,121.0,208.00,302.0,54.00,158.0,363.00,636.0,6.0,16.0,95.0,230.0
7,Delhi,3. Poor,test,test,16992,30953,12470,40.29,27.73,15.33,2.47,14.19,141.3,295.8,90.6,56.0,115.0,203.25,297.0,57.00,178.0,389.00,694.0,6.0,14.0,94.0,238.0
8,Delhi,3. Poor,test,true test,39022,123573,62119,50.27,23.99,15.37,1.14,9.23,144.7,265.3,85.0,59.0,118.0,206.00,306.0,50.00,149.0,350.00,614.2,6.0,16.0,98.0,232.0
9,Delhi,4. Undefine,control,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.0,119.0,224.00,335.0,77.00,230.0,533.00,923.0,6.0,11.0,78.0,295.0


In [139]:
freq_bins = [1, 3, 5, 10, float('inf')]
freq_labels = ['a. 1-3', 'b. 3-5', 'c. 5-10', 'd. 10+']

# Create a new column with the binned data
df_merge['frequency_bins'] = pd.cut(df_merge['frequency'], bins=freq_bins, labels=freq_labels, right=False)

df_merge['frequency_bins'] = np.where((df_merge['sample_category'] == 'control') | (df_merge['updated_segment'].isin(['1. control', '2. not viewed'])), 'NA',  df_merge['frequency_bins'])

In [141]:
df_city_group_tc_freq = get_funnel(df_merge, ['city_name', 'sample_category','frequency_bins'])
df_city_group_tc_freq 

Execution time: 24.984 seconds


,city_name,sample_category,frequency_bins,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,NA,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.00,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,NA,35173,84997,44853,52.77,20.62,15.63,1.38,9.61,132.2,286.8,70.6,49.0,104.0,190.0,290.0,50.0,164.0,382.00,684.6,6.0,10.0,57.0,174.0
2,Delhi,test,a. 1-3,59799,112612,78053,69.31,8.99,17.36,0.41,3.92,125.5,320.1,76.9,48.0,99.0,177.0,272.0,61.0,182.0,422.00,741.0,6.0,10.0,57.0,184.0
3,Delhi,test,b. 3-5,27559,125070,76426,61.11,14.90,17.72,0.76,5.51,127.1,274.7,67.5,49.0,100.0,179.0,278.0,48.0,154.0,365.00,643.0,6.0,10.0,58.0,173.0
4,Delhi,test,c. 5-10,24814,207209,127193,61.38,16.46,16.33,0.63,5.20,127.4,250.2,63.1,50.0,101.0,178.0,276.0,44.0,141.0,337.00,591.0,6.0,11.0,60.0,165.0
5,Delhi,test,d. 10+,8868,143485,84542,58.92,18.44,16.86,0.84,4.93,125.2,230.0,62.6,49.0,99.0,177.0,271.0,40.0,130.0,315.25,557.5,6.0,11.0,60.0,167.0


In [144]:
df_city_group_tc_freq = get_funnel(df_merge, ['mp_tag', 'sample_category','frequency_bins'])
df_city_group_tc_freq 

Execution time: 26.269 seconds


,mp_tag,sample_category,frequency_bins,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,1. Good,control,NA,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.00,89.0,157.00,245.0,51.00,159.0,371.00,649.0,6.0,9.0,46.00,139.0
1,1. Good,test,NA,20702,38358,23273,60.67,15.74,16.97,0.69,5.92,121.3,283.8,63.5,45.00,93.0,171.00,270.0,49.00,160.0,380.00,673.0,6.0,9.0,44.00,142.0
2,1. Good,test,a. 1-3,35757,55106,40365,73.25,6.51,17.46,0.25,2.53,111.3,324.4,69.5,43.00,87.0,154.75,238.0,62.00,187.0,426.00,729.1,5.0,9.0,46.00,155.0
3,1. Good,test,b. 3-5,20580,57134,38356,67.13,10.87,18.43,0.38,3.18,112.4,267.5,58.6,43.00,86.0,156.00,250.0,47.00,151.0,362.00,634.0,6.0,9.0,42.00,135.0
4,1. Good,test,c. 5-10,19855,86456,59117,68.38,11.25,17.01,0.37,2.99,108.6,242.1,53.0,42.00,85.0,149.00,239.0,41.00,136.0,328.00,577.0,6.0,9.0,41.00,123.0
5,1. Good,test,d. 10+,7825,58358,39171,67.12,12.04,17.63,0.49,2.71,103.7,223.6,51.7,41.00,80.0,142.00,225.0,38.00,124.5,307.00,548.0,6.0,9.0,43.00,126.0
6,2. Balance,control,NA,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.00,101.0,176.00,266.0,52.00,159.0,372.00,656.0,6.0,12.0,64.00,179.0
7,2. Balance,test,NA,16316,29404,15213,51.74,21.73,15.40,1.05,10.08,131.2,294.0,71.9,49.00,104.0,189.00,288.0,52.00,169.0,383.00,716.0,6.0,11.0,62.00,178.0
8,2. Balance,test,a. 1-3,26059,38471,26290,68.34,9.93,17.30,0.36,4.07,124.7,316.5,80.6,47.00,100.0,178.00,269.0,61.00,181.0,420.00,753.4,6.0,11.0,63.00,200.0
9,2. Balance,test,b. 3-5,17736,43446,26081,60.03,15.93,17.78,0.61,5.64,125.8,281.6,70.9,50.00,100.0,177.00,271.0,48.00,157.0,372.75,658.5,6.0,11.0,62.00,185.0


In [145]:
df_city_group_tc_freq.to_clipboard(index=False)

In [140]:
df_city_group_freq = get_funnel(df_merge, ['city_name', 'updated_segment','frequency_bins'])
df_city_group_freq 

Execution time: 24.985 seconds


,city_name,updated_segment,frequency_bins,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. control,NA,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.00,103.0,181.00,273.0,52.00,159.0,370.00,650.0,6.00,11.0,58.00,171.0
1,Delhi,2. not viewed,NA,35173,84997,44853,52.77,20.62,15.63,1.38,9.61,132.2,286.8,70.6,49.00,104.0,190.00,290.0,50.00,164.0,382.00,684.6,6.00,10.0,57.00,174.0
2,Delhi,3. viewed,a. 1-3,247,445,248,55.73,14.16,20.67,0.22,9.21,133.3,264.8,73.4,58.50,99.0,185.50,276.4,57.00,169.5,341.75,683.7,6.00,10.0,65.25,158.6
3,Delhi,3. viewed,b. 3-5,59,100,59,59.00,9.00,27.00,2.00,3.00,101.1,235.1,70.2,25.00,63.0,84.00,216.8,39.50,133.0,408.50,609.8,5.00,8.0,60.50,205.2
4,Delhi,3. viewed,c. 5-10,36,66,44,66.67,12.12,18.18,0.00,3.03,99.6,242.5,40.8,49.50,102.0,127.50,170.4,34.75,79.0,273.75,750.6,5.00,8.0,19.00,79.0
5,Delhi,3. viewed,d. 10+,9,21,8,38.10,19.05,14.29,19.05,9.52,125.8,166.3,53.6,81.25,124.5,169.00,197.8,88.00,145.0,234.00,287.4,5.50,7.0,36.00,205.0
6,Delhi,4. both_viewed,a. 1-3,59541,112148,77790,69.36,8.97,17.34,0.42,3.90,125.5,320.3,76.9,48.00,99.0,177.00,271.9,61.00,182.0,422.00,741.0,6.00,10.0,57.00,184.5
7,Delhi,4. both_viewed,b. 3-5,27474,124857,76304,61.11,14.90,17.71,0.76,5.51,127.1,274.7,67.5,49.00,100.0,179.00,278.0,48.00,154.0,365.00,642.0,6.00,10.0,58.00,173.0
8,Delhi,4. both_viewed,c. 5-10,24722,206648,126852,61.39,16.47,16.33,0.63,5.19,127.4,249.8,63.0,50.00,101.0,178.00,276.0,44.00,141.0,337.00,591.0,6.00,11.0,60.00,165.0
9,Delhi,4. both_viewed,d. 10+,8783,142053,83797,58.99,18.45,16.82,0.84,4.90,125.1,230.0,62.3,49.00,99.0,177.00,270.0,41.00,130.0,316.00,558.0,6.00,11.0,60.00,166.0


### Time Bucket

In [37]:
df_time_bucket_group_tc = get_funnel(df_merge, ['city_name', 'time_bucket', 'sample_category'])
df_time_bucket_group_tc

Execution time: 31.944 seconds


,city_name,time_bucket,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,afternoon,control,860161,1802581,1136728,63.06,14.24,17.03,0.77,4.89,120.2,295.0,67.7,49.0,94.0,168.0,258.0,57.0,172.0,394.0,683.0,6.0,11.0,55.0,161.0
1,Delhi,afternoon,test,94947,199019,124245,62.43,14.13,17.32,0.77,5.35,118.8,291.6,67.5,45.0,92.0,167.0,263.0,54.0,167.0,391.0,682.0,6.0,11.0,55.0,163.0
2,Delhi,evening_peak,control,933410,2241265,1352003,60.32,15.81,17.99,0.58,5.31,122.9,275.5,69.1,50.0,98.0,172.0,261.0,51.0,155.0,361.0,642.0,6.0,12.0,60.0,169.0
3,Delhi,evening_peak,test,103412,248347,148381,59.75,16.23,18.28,0.59,5.16,118.8,271.3,68.0,47.0,95.0,168.0,255.0,48.0,152.0,358.0,636.0,6.0,11.0,60.0,168.0
4,Delhi,morning_peak,control,783656,1961853,1216493,62.01,17.82,14.49,0.50,5.19,139.9,255.5,68.9,59.0,114.0,197.0,291.0,48.0,146.0,347.0,608.0,6.0,11.0,66.0,183.0
5,Delhi,morning_peak,test,86621,217051,134256,61.85,17.19,14.81,0.45,5.69,138.7,249.0,69.8,55.0,112.0,196.0,300.0,44.0,140.0,339.0,596.8,6.0,11.0,67.0,189.0
6,Delhi,rest_evening,control,219805,362435,207502,57.25,15.52,20.98,1.01,5.24,120.9,293.4,76.0,50.0,96.0,169.0,257.0,57.0,173.0,390.0,681.2,6.0,11.0,59.0,184.0
7,Delhi,rest_evening,test,24177,39893,22570,56.58,15.75,21.29,0.99,5.39,116.6,292.3,76.7,47.0,93.0,162.0,247.0,53.0,171.0,387.0,675.0,6.0,11.0,59.0,182.0
8,Delhi,rest_morning,control,354443,686497,388619,56.61,16.09,17.34,2.22,7.74,140.2,289.2,66.1,58.0,112.0,197.0,294.0,51.0,165.0,384.0,671.0,5.0,8.0,39.0,156.0
9,Delhi,rest_morning,test,39268,76108,42656,56.05,15.58,17.69,2.11,8.57,138.5,282.4,66.6,54.0,110.0,196.0,305.0,47.0,160.0,379.0,666.5,5.0,8.0,40.0,158.0


### Marketplace

In [38]:
df_overall_mp_group_tc = get_funnel(df_merge, ['experiment_name','mp_tag','sample_category'])
df_overall_mp_group_tc

Execution time: 31.181 seconds


,experiment_name,mp_tag,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,loyalty_experiment2,1. Good,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.0,89.0,157.0,245.0,51.00,159.0,371.0,649.0,6.0,9.0,46.00,139.0
1,loyalty_experiment2,1. Good,test,138274,349090,233697,66.94,11.30,17.85,0.41,3.50,111.2,271.1,60.4,43.0,87.0,154.0,245.0,47.00,153.0,365.0,639.0,6.0,9.0,45.00,138.0
2,loyalty_experiment2,2. Balance,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.0,101.0,176.0,266.0,52.00,159.0,372.0,656.0,6.0,12.0,64.00,179.0
3,loyalty_experiment2,2. Balance,test,109970,272148,162656,59.77,16.63,17.35,0.62,5.63,124.9,276.5,71.8,49.0,100.0,177.0,269.0,49.00,156.0,369.5,655.0,6.0,12.0,64.00,182.0
4,loyalty_experiment2,3. Poor,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.0,121.0,208.0,302.0,54.00,158.0,363.0,636.0,6.0,16.0,95.00,230.0
5,loyalty_experiment2,3. Poor,test,56014,154526,74589,48.27,24.74,15.37,1.41,10.22,143.9,271.4,86.0,59.0,117.0,206.0,304.0,51.00,154.0,357.0,629.7,6.0,16.0,98.00,233.0
6,loyalty_experiment2,4. Undefine,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.0,119.0,224.0,335.0,77.00,230.0,533.0,923.0,6.0,11.0,78.00,295.0
7,loyalty_experiment2,4. Undefine,test,2327,4654,1166,25.05,20.00,14.91,15.23,24.80,149.3,378.8,110.4,49.0,110.0,223.0,336.0,62.25,187.5,484.5,926.0,6.0,11.0,82.25,301.9


In [39]:
df_city_mp_group_tc = get_funnel(df_merge, ['city_name','mp_tag','sample_category'])
df_city_mp_group_tc

Execution time: 31.071 seconds


,city_name,mp_tag,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.0,89.0,157.0,245.0,51.00,159.0,371.0,649.0,6.0,9.0,46.00,139.0
1,Delhi,1. Good,test,138274,349090,233697,66.94,11.30,17.85,0.41,3.50,111.2,271.1,60.4,43.0,87.0,154.0,245.0,47.00,153.0,365.0,639.0,6.0,9.0,45.00,138.0
2,Delhi,2. Balance,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.0,101.0,176.0,266.0,52.00,159.0,372.0,656.0,6.0,12.0,64.00,179.0
3,Delhi,2. Balance,test,109970,272148,162656,59.77,16.63,17.35,0.62,5.63,124.9,276.5,71.8,49.0,100.0,177.0,269.0,49.00,156.0,369.5,655.0,6.0,12.0,64.00,182.0
4,Delhi,3. Poor,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.0,121.0,208.0,302.0,54.00,158.0,363.0,636.0,6.0,16.0,95.00,230.0
5,Delhi,3. Poor,test,56014,154526,74589,48.27,24.74,15.37,1.41,10.22,143.9,271.4,86.0,59.0,117.0,206.0,304.0,51.00,154.0,357.0,629.7,6.0,16.0,98.00,233.0
6,Delhi,4. Undefine,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.0,119.0,224.0,335.0,77.00,230.0,533.0,923.0,6.0,11.0,78.00,295.0
7,Delhi,4. Undefine,test,2327,4654,1166,25.05,20.00,14.91,15.23,24.80,149.3,378.8,110.4,49.0,110.0,223.0,336.0,62.25,187.5,484.5,926.0,6.0,11.0,82.25,301.9


In [40]:
df_city_mp_group_tc_dispatch_click = get_funnel(df_merge, ['city_name','mp_tag','sample_category', 'dispatch_clicked_cux_tag'])
df_city_mp_group_tc_dispatch_view = get_funnel(df_merge, ['city_name','mp_tag','sample_category', 'dispatch_viewed_cux_tag'])
display(df_city_mp_group_tc_dispatch_click)
display(df_city_mp_group_tc_dispatch_view)

Execution time: 30.897 seconds
Execution time: 30.719 seconds


,city_name,mp_tag,sample_category,dispatch_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.00,89.0,157.00,245.0,51.0,159.0,371.00,649.0,6.00,9.0,46.00,139.0
1,Delhi,1. Good,test,test,137084,343888,230684,67.08,11.24,17.82,0.40,3.46,110.9,271.8,60.4,43.00,86.0,154.00,244.0,47.0,154.0,366.00,640.0,6.00,9.0,45.00,137.0
2,Delhi,1. Good,test,true test,1190,5202,3013,57.92,15.28,19.53,0.87,6.40,124.1,232.5,64.0,43.00,97.0,176.00,277.0,36.0,116.5,287.25,537.5,6.00,10.0,54.00,165.1
3,Delhi,2. Balance,control,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.00,101.0,176.00,266.0,52.0,159.0,372.00,656.0,6.00,12.0,64.00,179.0
4,Delhi,2. Balance,test,test,108793,266749,159929,59.95,16.53,17.33,0.61,5.57,124.5,276.8,71.7,49.00,100.0,176.00,268.0,49.0,157.0,370.00,657.0,6.00,12.0,64.00,181.0
5,Delhi,2. Balance,test,true test,1177,5399,2727,50.51,21.82,17.98,1.09,8.59,141.4,263.2,81.4,52.00,112.5,204.50,302.0,39.0,132.0,327.00,609.0,7.00,16.0,92.75,231.0
6,Delhi,3. Poor,control,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.00,121.0,208.00,302.0,54.0,158.0,363.00,636.0,6.00,16.0,95.00,230.0
7,Delhi,3. Poor,test,test,55176,150376,72982,48.53,24.60,15.37,1.41,10.09,143.2,272.6,85.5,58.00,117.0,204.75,302.0,51.0,155.0,358.00,632.0,6.00,16.0,97.00,232.0
8,Delhi,3. Poor,test,true test,838,4150,1607,38.72,29.59,15.33,1.25,15.11,165.4,226.9,107.9,67.75,145.0,238.00,336.0,48.0,138.0,324.25,551.5,7.00,32.0,136.00,285.0
9,Delhi,4. Undefine,control,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.00,119.0,224.00,335.0,77.0,230.0,533.00,923.0,6.00,11.0,78.00,295.0


,city_name,mp_tag,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.00,89.0,157.0,245.0,51.0,159.0,371.00,649.0,6.0,9.0,46.0,139.0
1,Delhi,1. Good,test,test,42130,69470,44586,64.18,11.65,19.63,0.33,4.22,110.9,301.0,70.5,43.00,88.0,155.0,243.0,54.0,171.0,403.75,706.0,6.0,10.0,50.0,155.0
2,Delhi,1. Good,test,true test,96144,279620,189111,67.63,11.21,17.41,0.43,3.32,111.2,262.7,57.9,43.00,86.0,154.0,245.0,45.0,148.0,355.00,621.0,6.0,9.0,43.0,133.0
3,Delhi,2. Balance,control,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.00,101.0,176.0,266.0,52.0,159.0,372.00,656.0,6.0,12.0,64.0,179.0
4,Delhi,2. Balance,test,test,29677,47882,27778,58.01,15.56,19.63,0.56,6.24,123.1,311.0,80.0,49.00,101.0,176.0,260.0,57.0,177.0,408.00,720.0,6.0,12.0,64.0,193.0
5,Delhi,2. Balance,test,true test,80293,224266,134878,60.14,16.86,16.86,0.64,5.50,125.3,267.9,70.1,49.00,100.0,177.0,271.0,47.0,151.0,361.00,639.0,6.0,12.0,64.0,181.0
6,Delhi,3. Poor,control,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.00,121.0,208.0,302.0,54.0,158.0,363.00,636.0,6.0,16.0,95.0,230.0
7,Delhi,3. Poor,test,test,10450,18115,8189,45.21,23.73,17.44,2.24,11.39,135.4,306.4,87.1,54.00,110.0,194.0,285.0,60.0,185.5,398.00,733.1,6.0,14.0,84.0,228.0
8,Delhi,3. Poor,test,true test,45564,136411,66400,48.68,24.87,15.09,1.30,10.07,145.0,266.0,85.8,59.00,118.0,207.0,307.0,50.0,149.0,351.00,617.0,6.0,16.0,99.0,234.0
9,Delhi,4. Undefine,control,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.00,119.0,224.0,335.0,77.0,230.0,533.00,923.0,6.0,11.0,78.0,295.0


In [41]:
df_city_mp_group_tc_otw_click = get_funnel(df_merge, ['city_name','mp_tag','sample_category', 'otw_clicked_cux_tag'])
df_city_mp_group_tc_otw_view = get_funnel(df_merge, ['city_name','mp_tag','sample_category', 'otw_viewed_cux_tag'])
display(df_city_mp_group_tc_otw_click)
display(df_city_mp_group_tc_otw_view)

Execution time: 31.986 seconds
Execution time: 30.725 seconds


,city_name,mp_tag,sample_category,otw_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.0,89.0,157.00,245.0,51.0,159.0,371.00,649.0,6.00,9.0,46.00,139.0
1,Delhi,1. Good,test,test,136903,343036,229618,66.94,11.33,17.81,0.41,3.52,111.1,271.5,60.4,43.0,87.0,154.00,245.0,47.0,153.0,365.00,640.0,6.00,9.0,45.00,138.0
2,Delhi,1. Good,test,true test,1371,6054,4079,67.38,9.25,20.30,0.40,2.68,112.0,252.3,61.3,43.0,87.0,160.25,241.0,43.0,126.0,328.00,595.6,5.00,9.0,42.00,146.0
3,Delhi,2. Balance,control,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.0,101.0,176.00,266.0,52.0,159.0,372.00,656.0,6.00,12.0,64.00,179.0
4,Delhi,2. Balance,test,test,108755,267465,159730,59.72,16.68,17.32,0.63,5.65,125.0,276.2,72.0,49.0,100.0,177.00,269.0,49.0,156.0,369.50,655.6,6.00,12.0,64.00,182.0
5,Delhi,2. Balance,test,true test,1215,4683,2926,62.48,14.22,18.62,0.26,4.42,118.8,291.3,64.6,42.0,95.0,176.00,256.5,50.0,157.0,370.25,632.5,6.00,11.0,58.00,180.0
6,Delhi,3. Poor,control,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.0,121.0,208.00,302.0,54.0,158.0,363.00,636.0,6.00,16.0,95.00,230.0
7,Delhi,3. Poor,test,test,55288,151589,73028,48.17,24.81,15.33,1.42,10.26,143.8,272.2,85.9,59.0,117.0,206.00,304.0,52.0,155.0,357.00,631.3,6.00,16.0,98.00,233.0
8,Delhi,3. Poor,test,true test,726,2937,1561,53.15,20.94,17.23,0.48,8.21,149.4,236.3,90.3,57.0,122.0,213.50,319.6,38.0,125.5,310.75,556.5,6.00,15.0,92.75,240.0
9,Delhi,4. Undefine,control,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.0,119.0,224.00,335.0,77.0,230.0,533.00,923.0,6.00,11.0,78.00,295.0


,city_name,mp_tag,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.0,89.0,157.00,245.0,51.0,159.0,371.0,649.0,6.0,9.0,46.0,139.0
1,Delhi,1. Good,test,test,46140,76888,45932,59.74,15.24,18.62,0.53,5.87,117.8,298.4,69.4,45.0,92.0,165.00,257.0,53.0,172.0,403.0,700.0,6.0,10.0,49.0,155.0
2,Delhi,1. Good,test,true test,92134,272202,187765,68.98,10.19,17.63,0.37,2.83,108.3,263.0,58.1,42.0,84.0,149.00,239.0,45.0,147.0,354.0,620.0,6.0,9.0,43.0,133.0
3,Delhi,2. Balance,control,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.0,101.0,176.00,266.0,52.0,159.0,372.0,656.0,6.0,12.0,64.0,179.0
4,Delhi,2. Balance,test,test,34780,57598,29990,52.07,20.21,17.97,0.83,8.93,127.0,304.9,79.7,49.0,103.0,181.00,274.0,54.0,173.0,402.0,716.0,6.0,12.0,66.0,194.0
5,Delhi,2. Balance,test,true test,75190,214550,132666,61.83,15.68,17.18,0.57,4.74,124.2,268.6,70.0,49.0,99.0,176.00,268.0,48.0,152.0,361.0,639.2,6.0,12.0,64.0,180.0
6,Delhi,3. Poor,control,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.0,121.0,208.00,302.0,54.0,158.0,363.0,636.0,6.0,16.0,95.0,230.0
7,Delhi,3. Poor,test,test,15035,27476,10520,38.29,29.04,15.11,2.62,14.95,141.4,297.5,92.6,56.0,116.0,203.00,297.0,56.0,178.0,387.0,704.0,6.0,15.0,97.0,240.0
8,Delhi,3. Poor,test,true test,40979,127050,64069,50.43,23.81,15.42,1.14,9.20,144.6,265.9,84.8,59.0,118.0,206.00,306.0,51.0,149.0,351.0,615.8,6.0,16.0,98.0,232.0
9,Delhi,4. Undefine,control,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.0,119.0,224.00,335.0,77.0,230.0,533.0,923.0,6.0,11.0,78.0,295.0


In [42]:
df_city_mp_group_tc_both_view = get_funnel(df_merge, ['city_name','mp_tag','sample_category', 'both_view'])
display(df_city_mp_group_tc_both_view)

Execution time: 13.883 seconds


,city_name,mp_tag,sample_category,both_view,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,yes,10,65,23,35.38,18.46,46.15,0.00,0.00,62.3,68.1,34.6,28.5,57.5,64.25,81.2,13.0,17.0,76.75,171.6,5.00,6.0,9.00,34.4
1,Delhi,1. Good,test,yes,83786,256729,176799,68.87,10.33,17.56,0.38,2.87,108.6,261.4,57.7,42.0,84.0,149.00,239.0,45.0,147.0,353.00,617.0,6.00,9.0,43.00,132.0
2,Delhi,2. Balance,control,yes,9,58,25,43.10,8.62,48.28,0.00,0.00,58.0,26.8,43.1,19.0,22.0,28.00,132.4,13.0,16.5,21.75,30.9,6.00,7.0,18.00,64.4
3,Delhi,2. Balance,test,yes,69583,204703,126163,61.63,15.89,17.13,0.57,4.77,124.1,266.6,69.9,49.0,99.0,175.00,267.0,47.0,150.0,359.75,635.0,6.00,12.0,64.00,180.0
4,Delhi,3. Poor,control,yes,9,14,7,50.00,7.14,35.71,0.00,7.14,114.0,285.8,159.4,114.0,114.0,114.00,114.0,15.0,43.0,602.00,699.8,4.75,8.5,226.25,513.5
5,Delhi,3. Poor,test,yes,39022,123573,62119,50.27,23.99,15.37,1.14,9.23,144.7,265.3,85.0,59.0,118.0,206.00,306.0,50.0,149.0,350.00,614.2,6.00,16.0,98.00,232.0
6,Delhi,4. Undefine,test,yes,1249,2723,763,28.02,20.35,16.75,13.92,20.97,146.4,370.3,109.1,49.0,110.0,217.25,321.7,59.0,178.0,447.00,931.5,6.00,11.0,77.50,290.0


### MP vs Time Bucket

In [43]:
df_overall_mp_time_bucket = get_funnel(df_merge, ['experiment_name','mp_tag', 'time_bucket', 'sample_category'])
df_overall_mp_time_bucket

Execution time: 33.621 seconds


,experiment_name,mp_tag,time_bucket,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,loyalty_experiment2,1. Good,afternoon,control,563814,1044827,725406,69.43,10.39,17.11,0.32,2.75,107.0,281.7,61.2,44.00,84.0,147.00,230.0,54.00,163.0,377.00,652.0,6.0,10.0,47.00,137.0
1,loyalty_experiment2,1. Good,afternoon,test,62074,115030,79243,68.89,10.48,17.48,0.30,2.85,104.7,273.9,60.7,40.00,80.0,143.00,233.0,49.00,155.0,371.00,640.0,6.0,10.0,47.00,138.0
2,loyalty_experiment2,1. Good,evening_peak,control,553819,1101720,726031,65.90,11.84,18.24,0.30,3.71,110.7,275.7,62.3,45.00,86.0,153.00,238.0,49.00,153.0,360.00,641.0,6.0,10.0,51.00,144.0
3,loyalty_experiment2,1. Good,evening_peak,test,61304,122498,79654,65.02,12.45,18.68,0.27,3.57,107.0,266.8,61.0,42.00,85.0,149.00,232.0,45.00,149.0,359.00,636.0,6.0,10.0,49.00,141.0
4,loyalty_experiment2,1. Good,morning_peak,control,263811,503690,375174,74.49,8.57,14.47,0.36,2.11,113.3,259.4,50.9,47.00,90.0,155.00,239.0,45.00,146.0,352.00,620.0,5.0,8.0,27.00,116.0
5,loyalty_experiment2,1. Good,morning_peak,test,29206,55848,41471,74.26,8.28,14.80,0.39,2.27,111.5,248.7,50.5,41.00,86.0,155.00,249.0,41.00,138.0,336.00,595.7,5.0,8.0,27.00,116.0
6,loyalty_experiment2,1. Good,rest_evening,control,105669,158795,97629,61.48,12.51,21.72,0.54,3.75,115.3,295.0,73.3,47.00,91.0,160.00,249.0,56.00,175.0,394.00,689.0,5.0,10.0,55.00,170.0
7,loyalty_experiment2,1. Good,rest_evening,test,11565,17222,10662,61.91,12.36,21.55,0.46,3.72,112.2,303.5,75.1,45.00,89.0,155.00,240.2,56.00,176.0,404.50,680.0,5.0,10.0,55.00,173.0
8,loyalty_experiment2,1. Good,rest_morning,control,198779,347227,206910,59.59,14.46,18.34,1.22,6.39,138.6,295.8,67.8,58.00,111.0,194.00,291.0,55.00,172.0,388.00,673.0,5.0,8.0,36.00,155.0
9,loyalty_experiment2,1. Good,rest_morning,test,21995,38492,22667,58.89,13.99,19.08,1.13,6.92,136.8,285.7,66.7,53.00,109.0,192.00,305.0,51.00,166.0,383.00,665.6,5.0,8.0,33.00,154.0


In [44]:
df_city_mp_time_bucket = get_funnel(df_merge, ['city_name','mp_tag', 'time_bucket', 'sample_category'])
df_city_mp_time_bucket

Execution time: 32.775 seconds


,city_name,mp_tag,time_bucket,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,afternoon,control,563814,1044827,725406,69.43,10.39,17.11,0.32,2.75,107.0,281.7,61.2,44.00,84.0,147.00,230.0,54.00,163.0,377.00,652.0,6.0,10.0,47.00,137.0
1,Delhi,1. Good,afternoon,test,62074,115030,79243,68.89,10.48,17.48,0.30,2.85,104.7,273.9,60.7,40.00,80.0,143.00,233.0,49.00,155.0,371.00,640.0,6.0,10.0,47.00,138.0
2,Delhi,1. Good,evening_peak,control,553819,1101720,726031,65.90,11.84,18.24,0.30,3.71,110.7,275.7,62.3,45.00,86.0,153.00,238.0,49.00,153.0,360.00,641.0,6.0,10.0,51.00,144.0
3,Delhi,1. Good,evening_peak,test,61304,122498,79654,65.02,12.45,18.68,0.27,3.57,107.0,266.8,61.0,42.00,85.0,149.00,232.0,45.00,149.0,359.00,636.0,6.0,10.0,49.00,141.0
4,Delhi,1. Good,morning_peak,control,263811,503690,375174,74.49,8.57,14.47,0.36,2.11,113.3,259.4,50.9,47.00,90.0,155.00,239.0,45.00,146.0,352.00,620.0,5.0,8.0,27.00,116.0
5,Delhi,1. Good,morning_peak,test,29206,55848,41471,74.26,8.28,14.80,0.39,2.27,111.5,248.7,50.5,41.00,86.0,155.00,249.0,41.00,138.0,336.00,595.7,5.0,8.0,27.00,116.0
6,Delhi,1. Good,rest_evening,control,105669,158795,97629,61.48,12.51,21.72,0.54,3.75,115.3,295.0,73.3,47.00,91.0,160.00,249.0,56.00,175.0,394.00,689.0,5.0,10.0,55.00,170.0
7,Delhi,1. Good,rest_evening,test,11565,17222,10662,61.91,12.36,21.55,0.46,3.72,112.2,303.5,75.1,45.00,89.0,155.00,240.2,56.00,176.0,404.50,680.0,5.0,10.0,55.00,173.0
8,Delhi,1. Good,rest_morning,control,198779,347227,206910,59.59,14.46,18.34,1.22,6.39,138.6,295.8,67.8,58.00,111.0,194.00,291.0,55.00,172.0,388.00,673.0,5.0,8.0,36.00,155.0
9,Delhi,1. Good,rest_morning,test,21995,38492,22667,58.89,13.99,19.08,1.13,6.92,136.8,285.7,66.7,53.00,109.0,192.00,305.0,51.00,166.0,383.00,665.6,5.0,8.0,33.00,154.0


In [45]:
df_city_mp_time_bucket_group_tc_dispatch_view = get_funnel(df_merge, ['city_name','mp_tag','time_bucket', 'sample_category', 'dispatch_viewed_cux_tag'])
df_city_mp_time_bucket_group_tc_dispatch_view

Execution time: 31.246 seconds


,city_name,mp_tag,time_bucket,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,afternoon,control,control,563814,1044827,725406,69.43,10.39,17.11,0.32,2.75,107.0,281.7,61.2,44.00,84.0,147.00,230.0,54.00,163.0,377.00,652.0,6.0,10.0,47.00,137.0
1,Delhi,1. Good,afternoon,test,test,11613,15481,10792,69.71,9.58,17.67,0.33,2.71,105.7,304.8,64.8,39.00,80.0,149.00,234.8,54.00,168.0,399.50,716.0,6.0,10.0,43.00,138.0
2,Delhi,1. Good,afternoon,test,true test,50461,99549,68451,68.76,10.62,17.45,0.30,2.87,104.5,269.1,60.0,40.00,80.0,142.75,232.0,48.00,152.0,367.00,628.0,6.0,10.0,48.00,138.0
3,Delhi,1. Good,evening_peak,control,control,553819,1101720,726031,65.90,11.84,18.24,0.30,3.71,110.7,275.7,62.3,45.00,86.0,153.00,238.0,49.00,153.0,360.00,641.0,6.0,10.0,51.00,144.0
4,Delhi,1. Good,evening_peak,test,test,26620,42679,26624,62.38,12.46,20.20,0.27,4.68,110.8,296.8,70.9,44.00,88.0,155.00,241.0,52.00,168.0,401.00,699.0,6.0,10.0,51.00,158.0
5,Delhi,1. Good,evening_peak,test,true test,34684,79819,53030,66.44,12.44,17.87,0.27,2.98,104.9,248.6,55.8,42.00,83.0,145.25,228.0,42.00,138.0,336.00,598.0,6.0,10.0,48.00,132.0
6,Delhi,1. Good,morning_peak,control,control,263811,503690,375174,74.49,8.57,14.47,0.36,2.11,113.3,259.4,50.9,47.00,90.0,155.00,239.0,45.00,146.0,352.00,620.0,5.0,8.0,27.00,116.0
7,Delhi,1. Good,morning_peak,test,test,1683,2113,1589,75.20,7.00,15.29,0.19,2.27,97.7,272.0,51.9,35.00,78.0,142.25,198.9,36.00,128.0,322.00,565.4,5.0,8.0,24.00,112.0
8,Delhi,1. Good,morning_peak,test,true test,27523,53735,39882,74.22,8.33,14.78,0.40,2.27,111.9,247.8,50.5,41.25,86.0,155.00,250.0,42.00,138.0,337.00,597.0,5.0,8.0,27.00,116.0
9,Delhi,1. Good,rest_evening,control,control,105669,158795,97629,61.48,12.51,21.72,0.54,3.75,115.3,295.0,73.3,47.00,91.0,160.00,249.0,56.00,175.0,394.00,689.0,5.0,10.0,55.00,170.0


In [46]:
df_city_mp_time_bucket_group_tc_otw_view = get_funnel(df_merge, ['city_name','mp_tag','time_bucket', 'sample_category', 'otw_viewed_cux_tag'])
df_city_mp_time_bucket_group_tc_otw_view

Execution time: 32.013 seconds


,city_name,mp_tag,time_bucket,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,afternoon,control,control,563814,1044827,725406,69.43,10.39,17.11,0.32,2.75,107.0,281.7,61.2,44.00,84.0,147.00,230.0,54.00,163.0,377.00,652.0,6.0,10.0,47.00,137.0
1,Delhi,1. Good,afternoon,test,test,10791,14417,8759,60.75,17.29,15.73,0.68,5.55,110.1,295.7,62.4,41.00,82.0,157.00,245.0,52.75,171.0,409.00,703.0,6.0,9.0,39.00,133.0
2,Delhi,1. Good,afternoon,test,true test,51283,100613,70484,70.05,9.51,17.73,0.25,2.46,103.3,271.2,60.5,39.00,80.0,141.00,227.0,48.00,153.0,366.00,632.0,6.0,10.0,48.00,139.0
3,Delhi,1. Good,evening_peak,control,control,553819,1101720,726031,65.90,11.84,18.24,0.30,3.71,110.7,275.7,62.3,45.00,86.0,153.00,238.0,49.00,153.0,360.00,641.0,6.0,10.0,51.00,144.0
4,Delhi,1. Good,evening_peak,test,test,26785,42990,26480,61.60,13.12,20.22,0.26,4.80,111.4,295.6,70.0,44.00,89.0,157.00,241.0,52.00,168.0,393.00,692.0,6.0,10.0,51.00,156.0
5,Delhi,1. Good,evening_peak,test,true test,34519,79508,53174,66.88,12.08,17.85,0.28,2.91,104.4,249.1,56.3,41.00,82.0,144.00,226.0,42.00,138.0,338.00,604.0,6.0,10.0,48.00,133.0
6,Delhi,1. Good,morning_peak,control,control,263811,503690,375174,74.49,8.57,14.47,0.36,2.11,113.3,259.4,50.9,47.00,90.0,155.00,239.0,45.00,146.0,352.00,620.0,5.0,8.0,27.00,116.0
7,Delhi,1. Good,morning_peak,test,test,4337,5658,3608,63.77,15.69,14.76,0.81,4.95,118.4,269.2,55.7,43.00,93.0,165.25,253.3,42.00,153.0,365.50,658.4,5.0,8.0,28.00,122.4
8,Delhi,1. Good,morning_peak,test,true test,24869,50190,37863,75.44,7.45,14.80,0.34,1.97,109.8,246.4,50.0,41.00,84.0,152.00,248.0,41.00,135.0,333.00,587.2,5.0,8.0,27.00,115.0
9,Delhi,1. Good,rest_evening,control,control,105669,158795,97629,61.48,12.51,21.72,0.54,3.75,115.3,295.0,73.3,47.00,91.0,160.00,249.0,56.00,175.0,394.00,689.0,5.0,10.0,55.00,170.0


In [47]:
df_city_mp_time_bucket_group_tc_both_view = get_funnel(df_merge, ['city_name','mp_tag','time_bucket', 'sample_category', 'both_view'])
df_city_mp_time_bucket_group_tc_both_view

Execution time: 14.599 seconds


,city_name,mp_tag,time_bucket,sample_category,both_view,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,afternoon,control,yes,4,16,3,18.75,18.75,62.50,0.00,0.00,117.0,93.6,10.6,58.50,62.0,148.00,199.6,13.50,18.0,119.00,287.8,5.00,6.0,7.00,27.4
1,Delhi,1. Good,afternoon,test,yes,46121,93387,65187,69.80,9.75,17.67,0.26,2.52,103.2,268.7,59.9,39.00,79.0,140.00,226.4,48.00,152.0,365.00,627.0,6.00,10.0,48.00,138.0
2,Delhi,1. Good,evening_peak,control,yes,6,32,11,34.38,6.25,59.38,0.00,0.00,64.5,49.8,43.0,64.25,64.5,64.75,64.9,13.00,17.0,33.50,144.8,5.00,6.5,9.00,50.8
3,Delhi,1. Good,evening_peak,test,yes,31223,73576,49179,66.84,12.23,17.74,0.28,2.91,104.6,247.5,55.8,41.00,82.0,145.00,226.0,42.00,138.0,335.00,597.0,6.00,10.0,48.00,132.0
4,Delhi,1. Good,morning_peak,control,yes,4,13,7,53.85,38.46,7.69,0.00,0.00,25.0,160.0,49.2,20.00,24.0,30.00,32.4,160.00,160.0,160.00,160.0,5.75,6.5,13.00,116.8
5,Delhi,1. Good,morning_peak,test,yes,24139,49269,37129,75.36,7.51,14.80,0.35,1.99,110.0,245.1,50.0,41.00,84.0,152.00,248.0,41.00,135.0,333.00,587.0,5.00,8.0,27.00,116.0
6,Delhi,1. Good,rest_evening,control,yes,1,2,0,0.00,100.00,0.00,0.00,0.00,71.5,NaN,NaN,65.75,71.5,77.25,80.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Delhi,1. Good,rest_evening,test,yes,5674,9181,5801,63.18,12.07,21.28,0.47,3.00,109.0,285.9,65.2,45.00,86.0,152.00,227.3,49.00,162.5,361.75,629.4,5.00,10.0,51.00,153.0
8,Delhi,1. Good,rest_morning,control,yes,2,2,2,100.00,0.00,0.00,0.00,0.00,NaN,NaN,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.50,7.0,8.50,9.4
9,Delhi,1. Good,rest_morning,test,yes,16935,31316,19503,62.28,11.53,20.06,0.98,5.16,130.2,282.1,65.8,52.00,103.5,180.00,287.0,49.00,161.0,378.00,657.0,5.00,8.0,33.00,153.0


### Route level analysis

In [48]:
df_merge.head(2)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,time_bucket,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view,gross_orders,accepted_orders,aor,mp_tag,frequency,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_view_tag,hex_8_route,updated_segment
7707735,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682e90f919e59a597fb36199,dropped,NaN,1.747882e+12,1.747882e+12,NaN,NaN,NaN,7.0,1747882233633,0815,082033,6.98,...,morning_peak,None,None,NaN,None,None,None,None,None,None,None,2783.0,2394.0,86.02,1. Good,0.0,0-500m,6-9km,control,control,control,control,control,883da1118dfffff - 883da1034dfffff,1. control
7707736,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682eeaf88f45c100810348c9,dropped,NaN,1.747905e+12,1.747905e+12,NaN,NaN,NaN,6.0,1747905272190,1430,144432,1.85,...,afternoon,None,None,NaN,None,None,None,None,None,None,None,8123.0,6126.0,75.42,2. Balance,0.0,0-500m,0-3km,control,control,control,control,control,883da13a35fffff - 883da13843fffff,1. control


In [62]:
df_route = get_funnel(df_merge, ['city_name','hex_8_route','sample_category'])

KeyboardInterrupt: 

In [ ]:
df_route

In [ ]:
df_route1 = get_funnel(df_merge, ['city_name','hex_8_route','updated_segment'])

In [ ]:
df_route1

### Service

In [49]:
display(get_funnel(df_merge, ['city_name','service_name','sample_category']))

Execution time: 28.497 seconds


,city_name,service_name,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,Auto,control,706603,1852875,1000612,54.00,18.24,16.81,1.27,9.67,121.0,234.0,66.8,54.00,104.0,173.00,246.0,50.0,140.0,312.00,542.6,6.0,10.0,55.0,168.0
1,Delhi,Auto,test,78546,206354,108047,52.36,18.79,17.24,1.22,10.39,113.4,229.4,66.5,47.00,97.0,165.00,238.0,45.0,133.0,302.00,541.0,6.0,10.0,55.0,169.0
2,Delhi,Auto NCR,control,37730,80146,38360,47.86,23.80,14.66,3.34,10.33,145.7,288.1,56.6,62.75,120.0,209.00,306.0,58.0,163.0,371.00,669.0,5.0,7.0,37.0,152.0
3,Delhi,Auto NCR,test,4174,8676,4108,47.35,23.49,14.85,3.48,10.83,147.2,266.5,59.9,61.00,122.0,214.75,316.3,52.0,157.0,368.25,646.6,5.0,7.0,47.0,172.0
4,Delhi,Bike Lite,control,618030,1482878,1070257,72.17,12.72,13.19,0.16,1.75,114.9,284.9,61.3,46.00,88.0,156.00,251.0,52.0,169.0,389.00,660.0,6.0,12.0,58.0,148.0
5,Delhi,Bike Lite,test,67894,162489,117909,72.56,12.27,13.27,0.13,1.77,116.3,287.9,62.2,45.00,87.0,158.00,257.0,53.0,170.0,396.00,669.0,6.0,12.0,59.0,151.0
6,Delhi,Bike Metro,control,252701,552800,389895,70.53,15.00,12.44,0.14,1.89,120.4,256.2,56.9,51.00,93.0,162.00,257.0,46.0,146.0,344.00,596.0,6.0,12.0,57.0,143.0
7,Delhi,Bike Metro,test,28219,61497,43762,71.16,14.58,12.36,0.07,1.84,122.2,250.0,56.1,50.00,94.0,164.00,265.0,48.0,146.5,343.00,583.0,6.0,12.0,57.0,142.0
8,Delhi,Cab SUV,control,44837,81624,30085,36.86,16.15,32.22,3.53,11.24,153.0,347.7,93.2,57.00,119.0,225.00,336.0,49.0,174.0,444.00,868.0,4.0,6.0,26.0,239.0
9,Delhi,Cab SUV,test,5100,9011,3264,36.22,16.37,31.64,3.44,12.33,144.4,364.4,93.7,52.00,111.0,214.50,333.0,50.0,183.0,464.50,905.0,4.0,6.0,23.0,240.0


In [50]:
display(get_funnel(df_merge, ['city_name','service_name','sample_category', 'dispatch_clicked_cux_tag']))
display(get_funnel(df_merge, ['city_name','service_name','sample_category', 'dispatch_viewed_cux_tag']))

Execution time: 29.833 seconds


,city_name,service_name,sample_category,dispatch_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,Auto,control,control,706603,1852875,1000612,54.00,18.24,16.81,1.27,9.67,121.0,234.0,66.8,54.00,104.0,173.00,246.0,50.00,140.0,312.00,542.6,6.0,10.0,55.0,168.0
1,Delhi,Auto,test,test,77556,201921,106042,52.52,18.74,17.23,1.22,10.30,113.0,230.1,66.5,47.00,96.0,164.00,237.0,45.00,134.0,303.00,542.0,6.0,10.0,55.0,168.6
2,Delhi,Auto,test,true test,990,4433,2005,45.23,21.02,17.66,1.62,14.46,129.4,199.2,67.7,51.00,106.5,193.00,269.9,39.00,107.0,262.50,441.2,6.0,11.0,66.0,196.0
3,Delhi,Auto NCR,control,control,37730,80146,38360,47.86,23.80,14.66,3.34,10.33,145.7,288.1,56.6,62.75,120.0,209.00,306.0,58.00,163.0,371.00,669.0,5.0,7.0,37.0,152.0
4,Delhi,Auto NCR,test,test,4108,8386,4015,47.88,23.42,14.95,3.45,10.30,146.6,268.6,59.1,60.00,121.0,214.00,315.7,52.25,157.0,375.50,652.9,5.0,7.0,47.0,170.0
5,Delhi,Auto NCR,test,true test,66,290,93,32.07,25.52,11.72,4.48,26.21,164.0,191.5,92.2,74.50,139.0,232.50,353.2,49.50,121.5,259.00,398.2,5.0,9.0,71.0,263.0
6,Delhi,Bike Lite,control,control,618030,1482878,1070257,72.17,12.72,13.19,0.16,1.75,114.9,284.9,61.3,46.00,88.0,156.00,251.0,52.00,169.0,389.00,660.0,6.0,12.0,58.0,148.0
7,Delhi,Bike Lite,test,test,67298,160506,116635,72.67,12.20,13.25,0.13,1.76,116.0,287.3,62.1,45.00,87.0,157.00,257.0,53.00,170.0,396.00,670.0,6.0,12.0,58.0,150.0
8,Delhi,Bike Lite,test,true test,596,1983,1274,64.25,17.85,14.67,0.15,3.08,130.1,329.9,75.0,50.00,99.5,179.75,273.0,34.00,144.0,334.50,648.0,7.0,21.0,85.0,209.0
9,Delhi,Bike Metro,control,control,252701,552800,389895,70.53,15.00,12.44,0.14,1.89,120.4,256.2,56.9,51.00,93.0,162.00,257.0,46.00,146.0,344.00,596.0,6.0,12.0,57.0,143.0


Execution time: 30.491 seconds


,city_name,service_name,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,Auto,control,control,706603,1852875,1000612,54.00,18.24,16.81,1.27,9.67,121.0,234.0,66.8,54.00,104.0,173.00,246.0,50.00,140.0,312.00,542.6,6.0,10.0,55.00,168.0
1,Delhi,Auto,test,test,21925,37359,18712,50.09,19.21,18.29,1.19,11.23,116.1,258.1,77.8,50.00,99.0,168.00,243.0,51.00,152.0,342.00,604.0,6.0,11.0,59.00,185.0
2,Delhi,Auto,test,true test,56621,168995,89335,52.86,18.69,17.01,1.23,10.20,112.8,222.6,64.1,46.00,96.0,164.00,237.0,44.00,128.0,294.00,526.0,5.0,10.0,54.00,166.0
3,Delhi,Auto NCR,control,control,37730,80146,38360,47.86,23.80,14.66,3.34,10.33,145.7,288.1,56.6,62.75,120.0,209.00,306.0,58.00,163.0,371.00,669.0,5.0,7.0,37.00,152.0
4,Delhi,Auto NCR,test,test,1225,2018,835,41.38,27.40,13.43,3.62,14.17,150.3,305.0,66.2,68.00,127.0,218.00,301.8,77.50,169.0,414.50,731.0,5.0,9.0,61.75,185.4
5,Delhi,Auto NCR,test,true test,2949,6658,3273,49.16,22.30,15.27,3.44,9.82,146.1,256.3,58.3,58.00,119.0,214.00,320.0,50.00,154.0,347.00,635.4,4.0,7.0,38.25,168.1
6,Delhi,Bike Lite,control,control,618030,1482878,1070257,72.17,12.72,13.19,0.16,1.75,114.9,284.9,61.3,46.00,88.0,156.00,251.0,52.00,169.0,389.00,660.0,6.0,12.0,58.00,148.0
7,Delhi,Bike Lite,test,test,17804,27127,19417,71.58,10.59,15.97,0.09,1.77,102.8,314.2,68.5,40.00,78.0,138.00,220.0,59.00,197.0,436.00,719.0,7.0,13.0,58.00,153.0
8,Delhi,Bike Lite,test,true test,50090,135362,98492,72.76,12.60,12.73,0.13,1.77,118.5,281.2,60.9,46.00,89.0,161.00,263.0,51.00,164.0,385.00,655.0,6.0,12.0,59.00,150.0
9,Delhi,Bike Metro,control,control,252701,552800,389895,70.53,15.00,12.44,0.14,1.89,120.4,256.2,56.9,51.00,93.0,162.00,257.0,46.00,146.0,344.00,596.0,6.0,12.0,57.00,143.0


In [51]:
display(get_funnel(df_merge, ['city_name','service_name','sample_category', 'otw_clicked_cux_tag']))
display(get_funnel(df_merge, ['city_name','service_name','sample_category', 'otw_viewed_cux_tag']))

Execution time: 30.543 seconds


,city_name,service_name,sample_category,otw_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,Auto,control,control,706603,1852875,1000612,54.00,18.24,16.81,1.27,9.67,121.0,234.0,66.8,54.00,104.0,173.00,246.0,50.00,140.0,312.00,542.6,6.0,10.0,55.0,168.0
1,Delhi,Auto,test,test,77632,202315,105663,52.23,18.88,17.20,1.24,10.45,113.4,229.4,66.7,47.00,97.0,165.00,238.0,45.00,134.0,303.00,542.0,6.0,10.0,55.0,169.0
2,Delhi,Auto,test,true test,914,4039,2384,59.02,14.26,19.16,0.42,7.13,116.1,230.8,60.1,46.00,92.5,181.00,244.0,43.00,113.0,273.00,465.0,5.0,9.0,49.0,156.0
3,Delhi,Auto NCR,control,control,37730,80146,38360,47.86,23.80,14.66,3.34,10.33,145.7,288.1,56.6,62.75,120.0,209.00,306.0,58.00,163.0,371.00,669.0,5.0,7.0,37.0,152.0
4,Delhi,Auto NCR,test,test,4129,8549,4029,47.13,23.68,14.77,3.52,10.90,147.3,266.8,60.2,61.00,122.0,215.00,317.0,52.00,156.0,374.00,645.8,5.0,7.0,48.0,173.0
5,Delhi,Auto NCR,test,true test,45,127,79,62.20,11.02,19.69,0.79,6.30,139.3,252.0,47.1,61.25,135.5,205.25,238.7,72.00,178.0,337.00,619.2,5.0,7.0,16.0,121.6
6,Delhi,Bike Lite,control,control,618030,1482878,1070257,72.17,12.72,13.19,0.16,1.75,114.9,284.9,61.3,46.00,88.0,156.00,251.0,52.00,169.0,389.00,660.0,6.0,12.0,58.0,148.0
7,Delhi,Bike Lite,test,test,67356,160574,116498,72.55,12.27,13.27,0.13,1.78,116.3,288.0,62.3,45.00,87.0,158.00,257.0,53.00,170.0,396.00,667.3,6.0,12.0,59.0,150.0
8,Delhi,Bike Lite,test,true test,538,1915,1411,73.68,11.85,13.11,0.00,1.36,109.3,276.7,60.2,43.00,86.0,146.00,236.8,37.00,136.0,376.00,776.0,6.0,14.0,59.0,153.6
9,Delhi,Bike Metro,control,control,252701,552800,389895,70.53,15.00,12.44,0.14,1.89,120.4,256.2,56.9,51.00,93.0,162.00,257.0,46.00,146.0,344.00,596.0,6.0,12.0,57.0,143.0


Execution time: 29.268 seconds


,city_name,service_name,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,Auto,control,control,706603,1852875,1000612,54.00,18.24,16.81,1.27,9.67,121.0,234.0,66.8,54.00,104.0,173.0,246.0,50.0,140.0,312.00,542.6,6.0,10.0,55.0,168.0
1,Delhi,Auto,test,test,26255,46607,20185,43.31,23.74,16.25,1.73,14.97,117.1,249.0,76.3,49.00,100.0,170.0,243.0,49.0,148.0,336.00,591.7,6.0,11.0,59.0,184.8
2,Delhi,Auto,test,true test,52291,159747,87862,55.00,17.34,17.53,1.08,9.05,112.0,224.1,64.2,46.00,95.0,163.0,235.9,44.0,129.0,295.00,526.0,5.0,10.0,54.0,165.0
3,Delhi,Auto NCR,control,control,37730,80146,38360,47.86,23.80,14.66,3.34,10.33,145.7,288.1,56.6,62.75,120.0,209.0,306.0,58.0,163.0,371.00,669.0,5.0,7.0,37.0,152.0
4,Delhi,Auto NCR,test,test,1426,2392,867,36.25,29.35,12.67,4.72,17.02,152.6,321.4,67.9,65.00,131.0,221.0,317.9,80.0,185.0,437.00,742.0,5.0,9.0,65.0,194.0
5,Delhi,Auto NCR,test,true test,2748,6284,3241,51.58,21.26,15.67,3.01,8.48,144.4,249.7,57.7,59.00,116.0,210.0,316.0,48.0,148.0,338.00,591.6,4.0,7.0,37.0,164.0
6,Delhi,Bike Lite,control,control,618030,1482878,1070257,72.17,12.72,13.19,0.16,1.75,114.9,284.9,61.3,46.00,88.0,156.0,251.0,52.0,169.0,389.00,660.0,6.0,12.0,58.0,148.0
7,Delhi,Bike Lite,test,test,20086,30556,20348,66.59,14.75,15.43,0.18,3.05,116.2,313.2,69.5,43.00,86.0,158.0,258.0,59.0,200.0,437.00,719.0,7.0,13.0,59.0,157.0
8,Delhi,Bike Lite,test,true test,47808,131933,97561,73.95,11.69,12.77,0.11,1.48,116.3,280.8,60.6,46.00,87.0,158.0,257.0,51.0,163.0,384.00,654.0,6.0,12.0,58.0,149.0
9,Delhi,Bike Metro,control,control,252701,552800,389895,70.53,15.00,12.44,0.14,1.89,120.4,256.2,56.9,51.00,93.0,162.0,257.0,46.0,146.0,344.00,596.0,6.0,12.0,57.0,143.0


### Channel Host

In [52]:
display(get_funnel(df_merge, ['city_name','channel_host', 'sample_category','dispatch_clicked_cux_tag']))
display(get_funnel(df_merge, ['city_name','channel_host', 'sample_category','dispatch_viewed_cux_tag']))

Execution time: 26.059 seconds


,city_name,channel_host,sample_category,dispatch_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,android,control,control,1469019,5142755,3133071,60.92,16.35,16.48,0.86,5.39,128.1,294.8,70.2,52.0,102.0,179.0,272.0,56.00,171.0,394.00,682.0,6.00,11.0,59.0,172.0
1,Delhi,android,test,test,162679,569225,342762,60.22,16.28,16.79,0.86,5.85,125.8,288.7,70.4,49.0,100.0,177.0,273.0,52.00,165.0,387.00,676.0,6.00,11.0,59.0,175.0
2,Delhi,android,test,true test,9,23,14,60.87,30.43,8.70,0.00,0.00,101.9,379.5,141.8,34.0,67.0,181.5,201.0,374.75,379.5,384.25,387.1,11.75,19.0,65.5,340.5
3,Delhi,ios,control,control,466092,1911876,1168274,61.11,14.98,17.87,0.60,5.44,132.3,237.3,65.0,54.0,107.0,187.0,278.0,44.00,133.0,313.00,562.0,6.00,10.0,57.0,169.0
4,Delhi,ios,test,test,50043,196331,121982,62.13,14.26,18.20,0.45,4.95,126.8,235.5,63.3,49.0,101.0,180.0,274.0,43.00,131.0,313.00,563.0,6.00,10.0,55.0,164.0
5,Delhi,ios,test,true test,1720,14839,7350,49.53,21.67,17.72,1.23,9.85,146.4,242.6,80.4,54.0,119.0,210.0,314.0,39.00,127.0,310.75,559.0,6.00,14.0,85.0,225.0


Execution time: 27.673 seconds


,city_name,channel_host,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,android,control,control,1469019,5142755,3133071,60.92,16.35,16.48,0.86,5.39,128.1,294.8,70.2,52.0,102.0,179.00,272.0,56.0,171.0,394.0,682.0,6.0,11.0,59.0,172.0
1,Delhi,android,test,test,49207,91832,53021,57.74,15.27,19.59,0.88,6.52,119.8,328.9,80.3,47.0,97.0,169.00,257.0,61.0,191.0,434.0,757.0,6.0,11.0,59.0,187.0
2,Delhi,android,test,true test,113481,477416,289755,60.69,16.47,16.25,0.86,5.72,126.9,279.4,68.4,49.0,100.0,178.00,276.0,50.0,159.0,376.0,656.0,6.0,11.0,59.0,173.0
3,Delhi,ios,control,control,466092,1911876,1168274,61.11,14.98,17.87,0.60,5.44,132.3,237.3,65.0,54.0,107.0,187.00,278.0,44.0,133.0,313.0,562.0,6.0,10.0,57.0,169.0
4,Delhi,ios,test,test,21638,44786,27851,62.19,13.42,18.75,0.56,5.09,124.3,256.6,66.9,48.0,98.0,176.75,267.0,47.0,145.0,341.0,605.0,6.0,10.0,54.0,165.0
5,Delhi,ios,test,true test,30125,166384,101481,60.99,15.15,18.01,0.49,5.35,129.9,230.2,63.6,50.0,103.0,185.00,280.0,41.0,127.0,305.0,548.0,6.0,10.0,57.0,169.0


In [53]:
display(get_funnel(df_merge, ['city_name','channel_host', 'sample_category','otw_clicked_cux_tag']))
display(get_funnel(df_merge, ['city_name','channel_host', 'sample_category','otw_viewed_cux_tag']))

Execution time: 26.269 seconds


,city_name,channel_host,sample_category,otw_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,android,control,control,1469019,5142755,3133071,60.92,16.35,16.48,0.86,5.39,128.1,294.8,70.2,52.00,102.0,179.00,272.0,56.0,171.0,394.00,682.0,6.0,11.0,59.0,172.0
1,Delhi,android,test,test,162685,569235,342765,60.22,16.28,16.79,0.86,5.85,125.8,288.7,70.4,49.00,100.0,177.00,273.0,52.0,165.0,387.00,676.0,6.0,11.0,59.0,175.0
2,Delhi,android,test,true test,3,13,11,84.62,15.38,0.00,0.00,0.00,346.5,NaN,61.3,301.75,346.5,391.25,418.1,NaN,NaN,NaN,NaN,5.5,17.0,74.5,153.0
3,Delhi,ios,control,control,466092,1911876,1168274,61.11,14.98,17.87,0.60,5.44,132.3,237.3,65.0,54.00,107.0,187.00,278.0,44.0,133.0,313.00,562.0,6.0,10.0,57.0,169.0
4,Delhi,ios,test,test,49997,197478,120770,61.16,14.88,18.10,0.52,5.35,129.0,234.0,64.1,50.00,102.0,183.00,278.0,42.0,130.0,311.00,558.9,6.0,10.0,56.0,167.0
5,Delhi,ios,test,true test,1766,13692,8562,62.53,13.47,19.09,0.38,4.53,126.7,263.5,68.3,46.00,101.0,184.00,273.7,44.0,135.5,336.75,611.7,6.0,10.0,57.0,179.0


Execution time: 25.627 seconds


,city_name,channel_host,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,android,control,control,1469019,5142755,3133071,60.92,16.35,16.48,0.86,5.39,128.1,294.8,70.2,52.0,102.0,179.0,272.0,56.0,171.0,394.0,682.0,6.0,11.0,59.0,172.0
1,Delhi,android,test,test,58063,112506,57607,51.20,20.25,17.65,1.38,9.51,127.4,324.0,81.1,49.0,102.0,181.0,276.0,59.0,191.0,432.0,750.9,6.0,11.0,61.0,194.0
2,Delhi,android,test,true test,104625,456742,285169,62.44,15.30,16.58,0.74,4.95,125.3,279.5,68.1,49.0,99.0,176.0,272.0,50.0,159.0,375.0,655.0,6.0,11.0,59.0,171.0
3,Delhi,ios,control,control,466092,1911876,1168274,61.11,14.98,17.87,0.60,5.44,132.3,237.3,65.0,54.0,107.0,187.0,278.0,44.0,133.0,313.0,562.0,6.0,10.0,57.0,169.0
4,Delhi,ios,test,test,23161,51243,29185,56.95,17.37,17.87,0.71,7.09,127.9,252.0,66.1,49.0,100.0,182.0,275.0,46.0,140.0,332.0,590.2,6.0,10.0,55.0,167.0
5,Delhi,ios,test,true test,28602,159927,100147,62.62,13.96,18.26,0.44,4.72,129.2,231.0,63.8,50.0,103.0,183.0,279.0,42.0,128.0,307.0,552.0,6.0,10.0,57.0,169.0


### Segmentation

In [54]:
display(get_funnel(df_merge, ['city_name','taxi_regularity_segment','sample_category']))

Execution time: 27.131 seconds


,city_name,taxi_regularity_segment,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,control,354006,1026703,627985,61.17,16.08,16.62,0.85,5.28,128.9,298.4,70.3,53.0,103.0,180.0,271.0,60.0,178.0,401.0,688.0,6.0,11.0,58.0,172.0
1,Delhi,BI_WEEKLY,test,39421,115497,70006,60.61,15.91,16.88,0.89,5.71,127.1,296.7,70.8,51.0,100.0,177.0,274.0,56.0,175.0,398.0,680.0,6.0,11.0,59.0,177.0
2,Delhi,DAILY,control,334267,2625158,1644553,62.65,16.30,15.93,0.55,4.57,127.7,222.6,60.6,52.0,102.0,180.0,272.0,41.0,126.0,300.0,532.0,6.0,11.0,59.0,161.0
3,Delhi,DAILY,test,36849,288651,179654,62.24,16.24,16.30,0.50,4.71,125.5,219.5,60.6,48.0,100.0,178.0,273.0,38.0,121.0,298.0,533.0,6.0,11.0,59.0,162.0
4,Delhi,MONTHLY,control,183552,453122,275975,60.91,15.78,16.70,0.87,5.75,133.1,325.8,75.6,56.0,107.0,185.0,279.0,68.0,198.0,431.0,736.0,6.0,11.0,60.0,183.0
5,Delhi,MONTHLY,test,20398,51100,30709,60.10,15.75,17.08,0.95,6.13,131.6,314.0,76.1,53.0,105.0,184.0,279.0,66.0,190.5,426.0,717.5,6.0,11.0,59.0,186.0
6,Delhi,RARE_NEED,control,180887,449883,272826,60.64,16.10,16.36,0.85,6.05,134.3,331.0,76.4,57.0,108.0,188.0,281.0,70.0,200.0,439.0,739.0,6.0,11.0,61.0,184.0
7,Delhi,RARE_NEED,test,20082,49708,30078,60.51,15.47,16.54,0.82,6.66,131.5,323.5,77.1,53.0,106.0,183.0,285.2,72.0,203.0,440.0,749.0,6.0,11.0,62.0,191.0
8,Delhi,UNKNOWN,control,251997,543104,306750,56.48,15.03,19.41,1.26,7.81,135.4,368.0,92.5,55.0,108.0,189.0,286.0,75.0,225.0,490.0,824.0,6.0,11.0,65.0,218.0
9,Delhi,UNKNOWN,test,27947,60051,33646,56.03,14.74,19.34,1.25,8.64,131.0,372.1,92.4,50.0,103.0,183.0,284.0,72.0,224.0,489.0,836.0,6.0,11.0,66.0,222.0


In [55]:
display(get_funnel(df_merge, ['city_name','taxi_regularity_segment','sample_category','dispatch_clicked_cux_tag']))
display(get_funnel(df_merge, ['city_name','taxi_regularity_segment','sample_category','dispatch_viewed_cux_tag']))

Execution time: 28.583 seconds


,city_name,taxi_regularity_segment,sample_category,dispatch_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,control,control,354006,1026703,627985,61.17,16.08,16.62,0.85,5.28,128.9,298.4,70.3,53.00,103.0,180.00,271.0,60.00,178.0,401.00,688.0,6.00,11.0,58.00,172.0
1,Delhi,BI_WEEKLY,test,test,39177,114024,69314,60.79,15.83,16.87,0.87,5.63,126.7,297.0,70.7,51.00,100.0,177.00,274.0,56.00,175.0,399.00,682.0,6.00,11.0,59.00,176.0
2,Delhi,BI_WEEKLY,test,true test,244,1473,692,46.98,21.86,17.38,2.10,11.68,151.9,274.5,80.3,61.00,120.0,212.75,324.4,47.75,145.5,340.00,622.0,6.75,15.5,97.25,233.4
3,Delhi,DAILY,control,control,334267,2625158,1644553,62.65,16.30,15.93,0.55,4.57,127.7,222.6,60.6,52.00,102.0,180.00,272.0,41.00,126.0,300.00,532.0,6.00,11.0,59.00,161.0
4,Delhi,DAILY,test,test,36140,279816,175095,62.58,16.10,16.23,0.49,4.61,124.9,219.1,60.2,48.00,99.0,177.00,271.0,38.00,121.0,299.00,532.0,6.00,11.0,59.00,161.0
5,Delhi,DAILY,test,true test,709,8835,4559,51.60,20.95,18.49,0.92,8.04,141.9,231.9,77.8,51.00,113.0,202.00,306.0,36.00,118.5,295.00,535.7,6.00,13.0,78.50,221.0
6,Delhi,MONTHLY,control,control,183552,453122,275975,60.91,15.78,16.70,0.87,5.75,133.1,325.8,75.6,56.00,107.0,185.00,279.0,68.00,198.0,431.00,736.0,6.00,11.0,60.00,183.0
7,Delhi,MONTHLY,test,test,20306,50528,30420,60.20,15.68,17.09,0.95,6.07,131.6,314.7,76.1,53.00,105.0,184.00,279.0,66.00,191.0,427.00,719.0,6.00,11.0,59.00,186.0
8,Delhi,MONTHLY,test,true test,92,572,289,50.52,21.15,16.26,1.05,11.01,128.6,246.5,75.6,38.00,101.0,192.00,274.0,43.00,127.0,405.00,584.2,6.00,12.0,73.00,162.0
9,Delhi,RARE_NEED,control,control,180887,449883,272826,60.64,16.10,16.36,0.85,6.05,134.3,331.0,76.4,57.00,108.0,188.00,281.0,70.00,200.0,439.00,739.0,6.00,11.0,61.00,184.0


Execution time: 28.595 seconds


,city_name,taxi_regularity_segment,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,control,control,354006,1026703,627985,61.17,16.08,16.62,0.85,5.28,128.9,298.4,70.3,53.0,103.0,180.00,271.0,60.00,178.0,401.00,688.0,6.0,11.0,58.0,172.0
1,Delhi,BI_WEEKLY,test,test,14080,25900,15475,59.75,15.78,18.15,0.85,5.48,118.8,303.0,72.6,47.0,95.0,168.00,253.0,59.00,179.0,404.00,700.0,6.0,11.0,57.0,172.0
2,Delhi,BI_WEEKLY,test,true test,25341,89597,54531,60.86,15.95,16.51,0.90,5.78,129.5,294.7,70.3,51.0,102.0,180.50,281.0,55.00,174.0,396.50,673.0,6.0,11.0,59.0,178.0
3,Delhi,DAILY,control,control,334267,2625158,1644553,62.65,16.30,15.93,0.55,4.57,127.7,222.6,60.6,52.0,102.0,180.00,272.0,41.00,126.0,300.00,532.0,6.0,11.0,59.0,161.0
4,Delhi,DAILY,test,test,5716,19603,12302,62.76,14.51,17.58,0.37,4.78,114.7,222.8,59.3,43.0,93.0,161.00,250.0,38.00,120.0,306.00,559.0,6.0,11.0,56.0,157.0
5,Delhi,DAILY,test,true test,31133,269048,167352,62.20,16.37,16.21,0.51,4.71,126.2,219.3,60.7,48.0,100.0,179.00,275.0,38.00,121.0,298.00,531.0,6.0,11.0,59.0,163.0
6,Delhi,MONTHLY,control,control,183552,453122,275975,60.91,15.78,16.70,0.87,5.75,133.1,325.8,75.6,56.0,107.0,185.00,279.0,68.00,198.0,431.00,736.0,6.0,11.0,60.0,183.0
7,Delhi,MONTHLY,test,test,7808,13153,7684,58.42,15.39,19.34,0.82,6.03,125.8,333.8,82.6,51.0,102.0,176.25,269.7,69.75,200.0,441.00,762.0,6.0,11.0,60.0,190.1
8,Delhi,MONTHLY,test,true test,12590,37947,23025,60.68,15.87,16.29,1.00,6.16,133.5,305.8,73.8,54.0,107.0,187.00,282.0,65.00,188.0,419.00,707.0,6.0,11.0,59.0,184.0
9,Delhi,RARE_NEED,control,control,180887,449883,272826,60.64,16.10,16.36,0.85,6.05,134.3,331.0,76.4,57.0,108.0,188.00,281.0,70.00,200.0,439.00,739.0,6.0,11.0,61.0,184.0


In [56]:
display(get_funnel(df_merge, ['city_name','taxi_regularity_segment','sample_category','otw_clicked_cux_tag']))
display(get_funnel(df_merge, ['city_name','taxi_regularity_segment','sample_category','otw_viewed_cux_tag']))

Execution time: 28.126 seconds


,city_name,taxi_regularity_segment,sample_category,otw_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,control,control,354006,1026703,627985,61.17,16.08,16.62,0.85,5.28,128.9,298.4,70.3,53.00,103.0,180.00,271.0,60.00,178.0,401.00,688.0,6.0,11.0,58.00,172.0
1,Delhi,BI_WEEKLY,test,test,39159,113956,69029,60.58,15.96,16.85,0.89,5.73,127.1,296.6,70.9,51.00,100.0,177.00,275.0,56.00,175.0,399.00,683.0,6.0,11.0,59.00,177.0
2,Delhi,BI_WEEKLY,test,true test,262,1541,977,63.40,12.46,18.75,0.78,4.61,128.2,305.8,63.4,51.75,104.0,182.25,261.9,55.00,172.0,359.00,585.4,6.0,10.0,57.00,166.7
3,Delhi,DAILY,control,control,334267,2625158,1644553,62.65,16.30,15.93,0.55,4.57,127.7,222.6,60.6,52.00,102.0,180.00,272.0,41.00,126.0,300.00,532.0,6.0,11.0,59.00,161.0
4,Delhi,DAILY,test,test,36105,280311,174515,62.26,16.31,16.20,0.51,4.72,125.5,219.2,60.5,48.00,100.0,178.00,273.0,38.00,121.0,298.00,532.0,6.0,11.0,59.00,162.0
5,Delhi,DAILY,test,true test,744,8340,5139,61.62,13.90,19.69,0.32,4.47,127.1,229.0,63.9,44.00,101.0,188.50,282.2,40.00,119.0,293.75,547.5,6.0,11.0,56.00,173.0
6,Delhi,MONTHLY,control,control,183552,453122,275975,60.91,15.78,16.70,0.87,5.75,133.1,325.8,75.6,56.00,107.0,185.00,279.0,68.00,198.0,431.00,736.0,6.0,11.0,60.00,183.0
7,Delhi,MONTHLY,test,test,20298,50593,30366,60.02,15.78,17.09,0.96,6.15,131.6,314.0,76.2,53.00,105.0,184.00,279.0,66.00,190.0,426.00,717.0,6.0,11.0,59.00,186.0
8,Delhi,MONTHLY,test,true test,100,507,343,67.65,12.23,15.38,0.39,4.34,132.5,308.1,68.9,56.00,103.5,199.25,268.7,85.75,209.0,412.75,740.6,6.0,10.0,54.50,136.0
9,Delhi,RARE_NEED,control,control,180887,449883,272826,60.64,16.10,16.36,0.85,6.05,134.3,331.0,76.4,57.00,108.0,188.00,281.0,70.00,200.0,439.00,739.0,6.0,11.0,61.00,184.0


Execution time: 31.174 seconds


,city_name,taxi_regularity_segment,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,control,control,354006,1026703,627985,61.17,16.08,16.62,0.85,5.28,128.9,298.4,70.3,53.00,103.0,180.0,271.0,60.00,178.0,401.00,688.0,6.0,11.0,58.0,172.0
1,Delhi,BI_WEEKLY,test,test,15982,30492,16374,53.70,20.17,17.03,1.12,7.98,125.4,298.0,74.0,50.00,101.0,176.0,268.0,56.75,177.5,398.00,691.0,6.0,11.0,58.0,179.0
2,Delhi,BI_WEEKLY,test,true test,23439,85005,53632,63.09,14.38,16.83,0.80,4.90,128.0,296.2,69.8,51.00,100.0,178.0,277.0,55.00,174.0,399.00,677.0,6.0,11.0,59.0,176.0
3,Delhi,DAILY,control,control,334267,2625158,1644553,62.65,16.30,15.93,0.55,4.57,127.7,222.6,60.6,52.00,102.0,180.0,272.0,41.00,126.0,300.00,532.0,6.0,11.0,59.0,161.0
4,Delhi,DAILY,test,test,6698,24050,13775,57.28,18.81,16.82,0.52,6.57,123.7,214.2,60.1,45.00,99.0,176.0,272.0,35.00,112.0,288.00,521.0,6.0,11.0,57.0,160.0
5,Delhi,DAILY,test,true test,30151,264601,165879,62.69,16.01,16.26,0.50,4.54,125.7,220.0,60.7,48.00,100.0,179.0,273.0,39.00,122.0,299.00,533.0,6.0,11.0,59.0,162.0
6,Delhi,MONTHLY,control,control,183552,453122,275975,60.91,15.78,16.70,0.87,5.75,133.1,325.8,75.6,56.00,107.0,185.0,279.0,68.00,198.0,431.00,736.0,6.0,11.0,60.0,183.0
7,Delhi,MONTHLY,test,test,8895,15731,8165,51.90,20.63,17.39,1.28,8.79,130.7,334.5,82.0,52.00,105.0,186.0,277.0,76.00,205.0,438.25,758.5,6.0,11.0,61.0,193.2
8,Delhi,MONTHLY,test,true test,11503,35369,22544,63.74,13.57,16.94,0.81,4.94,132.2,304.6,73.9,54.00,106.0,183.0,280.0,64.00,182.0,420.00,708.0,6.0,11.0,59.0,182.0
9,Delhi,RARE_NEED,control,control,180887,449883,272826,60.64,16.10,16.36,0.85,6.05,134.3,331.0,76.4,57.00,108.0,188.0,281.0,70.00,200.0,439.00,739.0,6.0,11.0,61.0,184.0


### First Mile

In [57]:
display(get_funnel(df_merge, ['city_name','fm_bin','sample_category','both_view_tag']))

Execution time: 28.337 seconds


,city_name,fm_bin,sample_category,both_view_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,0-500m,control,control,977909,1847522,1562773,84.59,0.00,15.03,0.00,0.38,NaN,254.2,47.4,NaN,NaN,NaN,NaN,44.0,148.0,343.0,590.0,5.0,8.0,24.0,108.0
1,Delhi,0-500m,test,test,37029,49832,40809,81.89,0.00,17.56,0.00,0.55,NaN,271.1,53.4,NaN,NaN,NaN,NaN,45.0,155.0,365.0,631.0,6.0,8.0,25.0,113.0
2,Delhi,0-500m,test,true test,71153,154586,131198,84.87,0.00,14.81,0.00,0.32,NaN,239.9,45.2,NaN,NaN,NaN,NaN,40.0,135.0,327.0,567.0,5.0,8.0,23.0,106.0
3,Delhi,0.5-1km,control,control,933619,1627762,1302722,80.03,0.00,19.36,0.00,0.61,NaN,274.8,63.6,NaN,NaN,NaN,NaN,51.0,160.0,375.0,641.0,6.0,11.0,54.0,152.0
4,Delhi,0.5-1km,test,test,32556,40922,31296,76.48,0.00,22.53,0.00,0.99,NaN,293.8,68.9,NaN,NaN,NaN,NaN,49.0,169.0,396.0,692.0,6.0,10.0,54.0,159.0
5,Delhi,0.5-1km,test,true test,70519,138653,111297,80.27,0.00,19.17,0.00,0.56,NaN,258.9,61.1,NaN,NaN,NaN,NaN,45.0,149.0,358.0,612.0,6.0,10.0,54.0,150.0
6,Delhi,1-2km,control,control,891423,1486840,1086465,73.07,0.00,25.81,0.00,1.11,NaN,289.1,85.3,NaN,NaN,NaN,NaN,57.0,165.0,386.0,676.0,6.0,17.0,86.0,217.0
7,Delhi,1-2km,test,test,30159,37042,25748,69.51,0.00,28.80,0.00,1.69,NaN,316.1,93.9,NaN,NaN,NaN,NaN,61.0,182.0,423.0,737.0,6.0,16.0,86.0,227.0
8,Delhi,1-2km,test,true test,68341,127176,93411,73.45,0.00,25.46,0.00,1.09,NaN,274.5,83.8,NaN,NaN,NaN,NaN,52.0,155.5,370.0,652.0,6.0,17.0,88.0,218.0
9,Delhi,2+km,control,control,421491,573052,349385,60.97,0.00,37.14,0.00,1.89,NaN,295.2,109.4,NaN,NaN,NaN,NaN,58.0,161.0,374.0,701.0,7.0,26.0,118.0,277.0


In [58]:
display(get_funnel(df_merge, ['city_name','fm_bin','sample_category','dispatch_clicked_cux_tag']))
display(get_funnel(df_merge, ['city_name','fm_bin','sample_category','dispatch_viewed_cux_tag']))

Execution time: 27.934 seconds


,city_name,fm_bin,sample_category,dispatch_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,0-500m,control,control,977909,1847522,1562773,84.59,0.00,15.03,0.00,0.38,NaN,254.2,47.4,NaN,NaN,NaN,NaN,44.00,148.0,343.00,590.0,5.0,8.0,24.0,108.0
1,Delhi,0-500m,test,test,107069,201353,169504,84.18,0.00,15.45,0.00,0.37,NaN,248.4,47.1,NaN,NaN,NaN,NaN,41.00,141.0,339.00,584.0,5.0,8.0,24.0,108.0
2,Delhi,0-500m,test,true test,1113,3065,2503,81.66,0.00,17.81,0.00,0.52,NaN,253.9,50.4,NaN,NaN,NaN,NaN,33.25,115.5,284.50,535.0,6.0,9.0,36.0,133.8
3,Delhi,0.5-1km,control,control,933619,1627762,1302722,80.03,0.00,19.36,0.00,0.61,NaN,274.8,63.6,NaN,NaN,NaN,NaN,51.00,160.0,375.00,641.0,6.0,11.0,54.0,152.0
4,Delhi,0.5-1km,test,test,101920,176660,140378,79.46,0.00,19.88,0.00,0.66,NaN,268.7,62.7,NaN,NaN,NaN,NaN,46.00,155.0,368.00,633.0,6.0,10.0,54.0,151.0
5,Delhi,0.5-1km,test,true test,1155,2915,2215,75.99,0.00,23.16,0.00,0.86,NaN,224.9,71.6,NaN,NaN,NaN,NaN,36.00,127.0,330.00,540.0,6.0,12.0,70.0,203.0
6,Delhi,1-2km,control,control,891423,1486840,1086465,73.07,0.00,25.81,0.00,1.11,NaN,289.1,85.3,NaN,NaN,NaN,NaN,57.00,165.0,386.00,676.0,6.0,17.0,86.0,217.0
7,Delhi,1-2km,test,test,97292,161270,117159,72.65,0.00,26.14,0.00,1.22,NaN,285.9,85.8,NaN,NaN,NaN,NaN,54.00,162.0,383.00,675.0,6.0,16.0,87.0,220.0
8,Delhi,1-2km,test,true test,1208,2948,2000,67.84,0.00,30.19,0.00,1.97,NaN,233.4,98.3,NaN,NaN,NaN,NaN,44.00,129.5,309.75,581.5,7.0,26.0,116.0,259.6
9,Delhi,2+km,control,control,421491,573052,349385,60.97,0.00,37.14,0.00,1.89,NaN,295.2,109.4,NaN,NaN,NaN,NaN,58.00,161.0,374.00,701.0,7.0,26.0,118.0,277.0


Execution time: 28.060 seconds


,city_name,fm_bin,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,0-500m,control,control,977909,1847522,1562773,84.59,0.00,15.03,0.00,0.38,NaN,254.2,47.4,NaN,NaN,NaN,NaN,44.00,148.0,343.00,590.0,5.0,8.0,24.0,108.0
1,Delhi,0-500m,test,test,29004,38390,31409,81.82,0.00,17.63,0.00,0.55,NaN,273.4,54.7,NaN,NaN,NaN,NaN,47.00,156.0,367.00,637.0,6.0,9.0,26.0,114.0
2,Delhi,0-500m,test,true test,79178,166028,140598,84.68,0.00,14.98,0.00,0.33,NaN,241.7,45.4,NaN,NaN,NaN,NaN,39.75,136.0,330.00,571.0,5.0,8.0,23.0,107.0
3,Delhi,0.5-1km,control,control,933619,1627762,1302722,80.03,0.00,19.36,0.00,0.61,NaN,274.8,63.6,NaN,NaN,NaN,NaN,51.00,160.0,375.00,641.0,6.0,11.0,54.0,152.0
4,Delhi,0.5-1km,test,test,25352,31369,23932,76.29,0.00,22.70,0.00,1.01,NaN,303.7,69.6,NaN,NaN,NaN,NaN,51.00,174.0,405.00,707.0,6.0,11.0,54.0,158.0
5,Delhi,0.5-1km,test,true test,77723,148206,118661,80.06,0.00,19.35,0.00,0.59,NaN,259.0,61.4,NaN,NaN,NaN,NaN,44.00,150.0,358.00,615.0,6.0,10.0,54.0,151.0
6,Delhi,1-2km,control,control,891423,1486840,1086465,73.07,0.00,25.81,0.00,1.11,NaN,289.1,85.3,NaN,NaN,NaN,NaN,57.00,165.0,386.00,676.0,6.0,17.0,86.0,217.0
7,Delhi,1-2km,test,test,23044,27941,19285,69.02,0.00,29.33,0.00,1.65,NaN,321.4,95.4,NaN,NaN,NaN,NaN,63.00,186.0,430.00,743.6,6.0,16.0,85.0,227.0
8,Delhi,1-2km,test,true test,75456,136277,99874,73.29,0.00,25.57,0.00,1.14,NaN,276.2,84.1,NaN,NaN,NaN,NaN,52.00,156.0,372.00,655.0,6.0,17.0,88.0,219.0
9,Delhi,2+km,control,control,421491,573052,349385,60.97,0.00,37.14,0.00,1.89,NaN,295.2,109.4,NaN,NaN,NaN,NaN,58.00,161.0,374.00,701.0,7.0,26.0,118.0,277.0


In [59]:
display(get_funnel(df_merge, ['city_name','fm_bin','sample_category','otw_clicked_cux_tag']))
display(get_funnel(df_merge, ['city_name','fm_bin','sample_category','otw_viewed_cux_tag']))

Execution time: 28.470 seconds


,city_name,fm_bin,sample_category,otw_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,0-500m,control,control,977909,1847522,1562773,84.59,0.00,15.03,0.00,0.38,NaN,254.2,47.4,NaN,NaN,NaN,NaN,44.0,148.0,343.0,590.0,5.0,8.0,24.0,108.0
1,Delhi,0-500m,test,test,106969,201181,169301,84.15,0.00,15.47,0.00,0.38,NaN,249.1,47.2,NaN,NaN,NaN,NaN,41.0,141.0,339.0,584.0,5.0,8.0,24.0,108.0
2,Delhi,0-500m,test,true test,1213,3237,2706,83.60,0.00,16.13,0.00,0.28,NaN,213.5,48.4,NaN,NaN,NaN,NaN,35.0,112.0,287.5,526.8,5.0,8.0,21.0,112.0
3,Delhi,0.5-1km,control,control,933619,1627762,1302722,80.03,0.00,19.36,0.00,0.61,NaN,274.8,63.6,NaN,NaN,NaN,NaN,51.0,160.0,375.0,641.0,6.0,11.0,54.0,152.0
4,Delhi,0.5-1km,test,test,101746,176163,139912,79.42,0.00,19.91,0.00,0.66,NaN,267.7,63.0,NaN,NaN,NaN,NaN,46.0,155.0,367.0,632.0,6.0,10.0,54.0,152.0
5,Delhi,0.5-1km,test,true test,1329,3412,2681,78.58,0.00,20.98,0.00,0.44,NaN,277.6,57.0,NaN,NaN,NaN,NaN,38.0,132.5,344.0,589.0,6.0,9.0,47.0,144.9
6,Delhi,1-2km,control,control,891423,1486840,1086465,73.07,0.00,25.81,0.00,1.11,NaN,289.1,85.3,NaN,NaN,NaN,NaN,57.0,165.0,386.0,676.0,6.0,17.0,86.0,217.0
7,Delhi,1-2km,test,test,97201,160880,116706,72.54,0.00,26.22,0.00,1.23,NaN,285.2,86.2,NaN,NaN,NaN,NaN,54.0,162.0,382.0,674.0,6.0,16.0,88.0,220.0
8,Delhi,1-2km,test,true test,1299,3338,2453,73.49,0.00,25.55,0.00,0.96,NaN,265.6,81.1,NaN,NaN,NaN,NaN,49.0,151.0,370.0,624.8,6.0,14.5,79.0,225.3
9,Delhi,2+km,control,control,421491,573052,349385,60.97,0.00,37.14,0.00,1.89,NaN,295.2,109.4,NaN,NaN,NaN,NaN,58.0,161.0,374.0,701.0,7.0,26.0,118.0,277.0


Execution time: 28.678 seconds


,city_name,fm_bin,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,0-500m,control,control,977909,1847522,1562773,84.59,0.00,15.03,0.00,0.38,NaN,254.2,47.4,NaN,NaN,NaN,NaN,44.0,148.0,343.0,590.0,5.0,8.0,24.0,108.0
1,Delhi,0-500m,test,test,31416,41691,33988,81.52,0.00,17.90,0.00,0.58,NaN,273.5,54.0,NaN,NaN,NaN,NaN,45.0,157.0,367.0,628.0,6.0,8.0,26.0,115.0
2,Delhi,0-500m,test,true test,76766,162727,138019,84.82,0.00,14.86,0.00,0.32,NaN,240.8,45.4,NaN,NaN,NaN,NaN,40.0,135.0,329.0,571.0,5.0,8.0,23.0,106.0
3,Delhi,0.5-1km,control,control,933619,1627762,1302722,80.03,0.00,19.36,0.00,0.61,NaN,274.8,63.6,NaN,NaN,NaN,NaN,51.0,160.0,375.0,641.0,6.0,11.0,54.0,152.0
4,Delhi,0.5-1km,test,test,26979,33427,25292,75.66,0.00,23.31,0.00,1.02,NaN,293.1,70.3,NaN,NaN,NaN,NaN,48.0,168.0,396.0,695.8,6.0,11.0,55.0,163.0
5,Delhi,0.5-1km,test,true test,76096,146148,117301,80.26,0.00,19.16,0.00,0.58,NaN,260.9,61.2,NaN,NaN,NaN,NaN,45.0,151.0,360.0,617.0,6.0,10.0,54.0,150.0
6,Delhi,1-2km,control,control,891423,1486840,1086465,73.07,0.00,25.81,0.00,1.11,NaN,289.1,85.3,NaN,NaN,NaN,NaN,57.0,165.0,386.0,676.0,6.0,17.0,86.0,217.0
7,Delhi,1-2km,test,test,24901,30178,20708,68.62,0.00,29.56,0.00,1.82,NaN,318.2,96.2,NaN,NaN,NaN,NaN,62.0,185.5,425.0,742.0,6.0,16.0,90.0,231.3
8,Delhi,1-2km,test,true test,73599,134040,98451,73.45,0.00,25.46,0.00,1.10,NaN,276.0,83.8,NaN,NaN,NaN,NaN,52.0,156.0,371.0,654.0,6.0,17.0,87.0,218.0
9,Delhi,2+km,control,control,421491,573052,349385,60.97,0.00,37.14,0.00,1.89,NaN,295.2,109.4,NaN,NaN,NaN,NaN,58.0,161.0,374.0,701.0,7.0,26.0,118.0,277.0


### Last Mile

In [95]:
display(get_funnel(df_merge, ['city_name','lm_bin','sample_category','both_view_tag']))

,city_name,lm_bin,sample_category,both_view_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,tta_p25,tta_p50,tta_p75,tta_p90,tta_p95
0,Bangalore,0-3km,control,control,586827,1587111,776941,48.95,26.45,13.85,0.95,9.80,120.1,193.2,109.8,52.00,98.0,170.00,249.0,291.00,39.0,109.0,263.0,471.0,628.00,8.0,31.0,125.00,263.0,394.00
1,Bangalore,0-3km,test,test,44612,97035,41470,42.74,30.65,12.96,1.06,12.59,123.8,193.9,110.7,53.00,101.0,176.00,256.0,297.00,39.0,109.0,267.0,477.0,638.00,8.0,32.0,133.00,261.0,392.00
2,Bangalore,0-3km,test,true test,30742,76758,44322,57.74,19.14,15.16,0.82,7.13,115.8,192.4,111.6,51.00,95.0,162.00,240.0,284.00,39.0,109.0,263.0,477.0,627.00,8.0,29.0,122.00,265.0,415.10
3,Bangalore,3-6km,control,control,735601,2098188,1004974,47.90,27.42,12.93,0.70,11.05,128.5,216.6,117.3,56.00,107.0,183.00,262.0,302.00,44.0,123.0,292.0,524.0,700.00,9.0,45.0,142.00,273.0,403.00
4,Bangalore,3-6km,test,test,54740,124650,51853,41.60,31.12,12.23,0.78,14.26,131.5,217.2,117.7,57.00,109.0,187.00,269.0,307.00,43.0,122.0,290.0,526.9,716.95,10.0,47.0,146.00,275.0,395.95
5,Bangalore,3-6km,test,true test,39962,103954,58431,56.21,20.94,13.88,0.54,8.43,124.7,213.7,114.4,54.00,102.0,176.00,257.0,300.00,44.0,121.0,290.0,526.0,697.00,10.0,44.0,138.00,271.0,399.00
6,Bangalore,6-9km,control,control,504518,1234442,623307,50.49,24.34,13.67,0.75,10.74,131.0,232.7,118.3,56.00,108.0,187.00,268.0,309.00,46.0,131.0,312.0,567.0,757.00,8.0,32.0,136.00,279.0,429.00
7,Bangalore,6-9km,test,test,35334,72939,32307,44.29,27.85,12.76,0.88,14.22,133.2,237.0,119.1,56.00,110.0,192.00,273.0,316.00,45.0,131.0,314.0,584.0,783.00,9.0,36.0,140.00,278.0,408.00
8,Bangalore,6-9km,test,true test,27726,62939,36639,58.21,18.95,14.29,0.58,7.96,124.1,238.1,116.4,53.00,101.0,176.00,255.0,301.00,45.0,131.0,315.0,566.0,751.80,8.0,31.0,133.00,274.0,423.40
9,Bangalore,9+km,control,control,746003,1967872,1034154,52.55,21.42,15.34,1.36,9.31,134.6,276.0,113.2,56.00,109.0,191.00,279.0,330.00,50.0,151.0,366.0,659.0,890.00,6.0,20.0,115.00,277.0,454.00


In [50]:
display(get_funnel(df_merge, ['city_name','lm_bin','sample_category','dispatch_clicked_cux_tag']))
display(get_funnel(df_merge, ['city_name','lm_bin','sample_category','dispatch_viewed_cux_tag']))

,city_name,lm_bin,sample_category,dispatch_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95
0,Bangalore,0-3km,control,control,586827,1587111,776941,48.95,26.45,13.85,0.95,9.80,120.1,193.2,52.00,98.0,170.00,249.0,291.00,39.00,109.0,263.00,471.0,628.00
1,Bangalore,0-3km,test,test,64456,172410,85388,49.53,25.50,13.94,0.94,10.09,120.9,193.3,52.00,99.0,171.00,250.0,293.00,39.00,109.0,265.00,477.0,634.00
2,Bangalore,0-3km,test,true test,530,1383,404,29.21,33.98,12.87,2.02,21.91,146.6,177.2,63.00,123.5,204.75,281.0,333.00,35.25,95.5,249.50,424.4,646.55
3,Bangalore,3-6km,control,control,735601,2098188,1004974,47.90,27.42,12.93,0.70,11.05,128.5,216.6,56.00,107.0,183.00,262.0,302.00,44.00,123.0,292.00,524.0,700.00
4,Bangalore,3-6km,test,test,80753,226326,109702,48.47,26.37,12.98,0.67,11.51,128.9,215.5,56.00,107.0,183.00,265.0,304.00,43.00,121.0,290.00,526.0,708.00
5,Bangalore,3-6km,test,true test,728,2278,582,25.55,38.76,12.69,1.14,21.82,140.4,216.6,59.00,116.0,197.50,289.4,333.40,44.00,142.0,280.00,509.8,623.60
6,Bangalore,6-9km,control,control,504518,1234442,623307,50.49,24.34,13.67,0.75,10.74,131.0,232.7,56.00,108.0,187.00,268.0,309.00,46.00,131.0,312.00,567.0,757.00
7,Bangalore,6-9km,test,test,55194,134390,68546,51.01,23.64,13.48,0.73,11.14,129.3,237.8,54.00,107.0,185.00,266.0,309.00,45.00,131.0,315.00,574.0,768.00
8,Bangalore,6-9km,test,true test,518,1488,400,26.88,31.38,12.50,1.41,27.82,164.8,209.8,67.25,145.0,238.25,321.8,384.00,40.00,121.0,288.50,511.6,749.80
9,Bangalore,9+km,control,control,746003,1967872,1034154,52.55,21.42,15.34,1.36,9.31,134.6,276.0,56.00,109.0,191.00,279.0,330.00,50.00,151.0,366.00,659.0,890.00


,city_name,lm_bin,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95
0,Bangalore,0-3km,control,control,586827,1587111,776941,48.95,26.45,13.85,0.95,9.80,120.1,193.2,52.00,98.0,170.00,249.0,291.00,39.0,109.0,263.00,471.0,628.00
1,Bangalore,0-3km,test,test,31715,61355,30690,50.02,24.72,14.84,0.84,9.57,119.6,195.4,52.00,97.0,169.00,250.0,291.00,39.0,108.0,267.00,483.0,639.00
2,Bangalore,0-3km,test,true test,41711,112438,55102,49.01,26.03,13.44,1.01,10.51,122.0,191.8,53.00,100.0,173.00,251.0,294.00,39.0,109.0,264.00,474.0,627.35
3,Bangalore,3-6km,control,control,735601,2098188,1004974,47.90,27.42,12.93,0.70,11.05,128.5,216.6,56.00,107.0,183.00,262.0,302.00,44.0,123.0,292.00,524.0,700.00
4,Bangalore,3-6km,test,test,39402,77252,37894,49.05,25.54,14.21,0.65,10.54,125.7,219.0,55.00,104.0,178.00,260.0,298.00,44.0,122.0,291.00,532.0,726.20
5,Bangalore,3-6km,test,true test,52871,151352,72390,47.83,26.97,12.35,0.68,12.16,130.6,213.5,57.00,108.0,186.00,268.0,308.00,43.0,121.0,289.00,522.0,696.00
6,Bangalore,6-9km,control,control,504518,1234442,623307,50.49,24.34,13.67,0.75,10.74,131.0,232.7,56.00,108.0,187.00,268.0,309.00,46.0,131.0,312.00,567.0,757.00
7,Bangalore,6-9km,test,test,25250,44531,23342,52.42,22.32,14.79,0.75,9.71,126.0,244.5,54.00,104.0,180.00,259.0,300.65,46.0,135.0,326.00,595.9,805.95
8,Bangalore,6-9km,test,true test,36186,91347,45604,49.92,24.41,12.82,0.73,12.11,131.6,233.6,55.00,108.0,188.00,271.0,315.00,44.0,128.0,309.00,559.2,747.00
9,Bangalore,9+km,control,control,746003,1967872,1034154,52.55,21.42,15.34,1.36,9.31,134.6,276.0,56.00,109.0,191.00,279.0,330.00,50.0,151.0,366.00,659.0,890.00


In [51]:
display(get_funnel(df_merge, ['city_name','lm_bin','sample_category','otw_clicked_cux_tag']))
display(get_funnel(df_merge, ['city_name','lm_bin','sample_category','otw_viewed_cux_tag']))

,city_name,lm_bin,sample_category,otw_clicked_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95
0,Bangalore,0-3km,control,control,586827,1587111,776941,48.95,26.45,13.85,0.95,9.80,120.1,193.2,52.00,98.0,170.00,249.0,291.00,39.00,109.0,263.00,471.0,628.00
1,Bangalore,0-3km,test,test,64493,172864,85253,49.32,25.62,13.91,0.95,10.20,121.2,193.0,52.00,99.0,171.00,250.0,293.00,39.00,109.0,265.00,477.0,634.00
2,Bangalore,0-3km,test,true test,462,929,539,58.02,16.36,18.08,0.43,7.10,115.6,210.9,53.50,91.0,169.00,245.5,292.35,38.75,128.5,319.50,535.2,708.20
3,Bangalore,3-6km,control,control,735601,2098188,1004974,47.90,27.42,12.93,0.70,11.05,128.5,216.6,56.00,107.0,183.00,262.0,302.00,44.00,123.0,292.00,524.0,700.00
4,Bangalore,3-6km,test,test,80794,227199,109525,48.21,26.53,12.95,0.67,11.64,129.0,215.5,56.00,107.0,183.00,265.0,305.00,43.00,121.0,290.00,526.0,707.00
5,Bangalore,3-6km,test,true test,632,1405,759,54.02,19.79,17.65,0.71,7.83,130.2,224.5,54.75,103.0,177.00,271.1,320.70,38.25,129.5,312.25,552.0,772.45
6,Bangalore,6-9km,control,control,504518,1234442,623307,50.49,24.34,13.67,0.75,10.74,131.0,232.7,56.00,108.0,187.00,268.0,309.00,46.00,131.0,312.00,567.0,757.00
7,Bangalore,6-9km,test,test,55240,134996,68419,50.68,23.79,13.44,0.74,11.34,129.8,236.8,55.00,107.0,186.00,267.0,311.00,45.00,131.0,314.00,573.0,766.00
8,Bangalore,6-9km,test,true test,457,882,527,59.75,14.29,17.01,0.34,8.62,134.5,322.2,52.00,118.0,198.75,274.6,293.65,41.50,153.5,435.75,774.2,1031.05
9,Bangalore,9+km,control,control,746003,1967872,1034154,52.55,21.42,15.34,1.36,9.31,134.6,276.0,56.00,109.0,191.00,279.0,330.00,50.00,151.0,366.00,659.0,890.00


,city_name,lm_bin,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95
0,Bangalore,0-3km,control,control,586827,1587111,776941,48.95,26.45,13.85,0.95,9.80,120.1,193.2,52.00,98.0,170.00,249.0,291.00,39.0,109.0,263.00,471.0,628.00
1,Bangalore,0-3km,test,test,41332,87921,35720,40.63,32.19,12.70,1.11,13.36,124.4,193.7,54.00,102.0,177.00,257.0,297.00,39.0,110.0,267.00,475.0,636.00
2,Bangalore,0-3km,test,true test,34400,85872,50072,58.31,18.78,15.19,0.78,6.93,115.5,192.7,51.00,94.0,161.00,239.0,284.00,39.0,108.0,263.00,479.1,632.00
3,Bangalore,3-6km,control,control,735601,2098188,1004974,47.90,27.42,12.93,0.70,11.05,128.5,216.6,56.00,107.0,183.00,262.0,302.00,44.0,123.0,292.00,524.0,700.00
4,Bangalore,3-6km,test,test,50644,113292,44770,39.52,32.67,11.93,0.82,15.06,131.8,217.4,57.00,109.0,188.00,269.0,308.00,43.0,122.0,290.00,527.0,714.00
5,Bangalore,3-6km,test,true test,44553,115312,65514,56.81,20.42,14.01,0.53,8.23,124.7,213.9,54.00,102.0,176.00,257.0,300.00,44.0,121.0,290.00,526.0,700.30
6,Bangalore,6-9km,control,control,504518,1234442,623307,50.49,24.34,13.67,0.75,10.74,131.0,232.7,56.00,108.0,187.00,268.0,309.00,46.0,131.0,312.00,567.0,757.00
7,Bangalore,6-9km,test,test,32394,66184,27956,42.24,29.32,12.49,0.92,15.01,133.7,235.4,56.00,111.0,192.00,274.0,317.00,45.0,131.0,314.00,582.0,781.00
8,Bangalore,6-9km,test,true test,30872,69694,40990,58.81,18.41,14.39,0.56,7.82,124.0,239.3,52.00,101.0,176.00,255.0,300.25,45.0,131.0,315.00,568.0,758.00
9,Bangalore,9+km,control,control,746003,1967872,1034154,52.55,21.42,15.34,1.36,9.31,134.6,276.0,56.00,109.0,191.00,279.0,330.00,50.0,151.0,366.00,659.0,890.00


In [69]:
df_test

,city,engaged_customeres,dispatch_viewed_cux,otw_viewed_cux,both_view,dispatch_viewed_cux %,otw_viewed_cux %,both_view %
0,Bangalore,136439,127210,109331,98158,93.235805,80.131781,71.942773
1,Delhi,158407,145471,135041,119860,91.833694,85.249389,75.665848
2,Hyderabad,158051,144496,136727,121434,91.423654,86.508152,76.832162


### Others

In [74]:
df_merge.shape

(7835049, 56)

In [75]:
df_merge\
[(df_merge['modified_order_status'] == 'COBRA') & (df_merge['sample_category'] == 'control')]\
[['cobra_ttc']].quantile([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

,cobra_ttc
0.1,25.0
0.2,44.0
0.3,62.0
0.4,81.0
0.5,103.0
0.6,128.0
0.7,161.0
0.8,205.0
0.9,273.0


In [76]:
df_merge\
[(~df_merge['dispatch_viewed_cux'].isna()) & (df_merge['modified_order_status'] == 'COBRA') & (df_merge['sample_category'] == 'test')]\
[['cobra_ttc']].quantile([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

,cobra_ttc
0.1,20.0
0.2,40.0
0.3,59.0
0.4,78.0
0.5,101.0
0.6,126.0
0.7,159.0
0.8,205.0
0.9,277.0


In [77]:
df_merge.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,time_bucket,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view,gross_orders,accepted_orders,aor,mp_tag,frequency,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_view_tag,hex_8_route,updated_segment
7707735,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682e90f919e59a597fb36199,dropped,NaN,1.747882e+12,1.747882e+12,NaN,NaN,NaN,7.0,1747882233633,0815,082033,6.98,...,morning_peak,None,None,NaN,None,None,None,None,None,None,None,2783.0,2394.0,86.02,1. Good,0.0,0-500m,6-9km,control,control,control,control,control,883da1118dfffff - 883da1034dfffff,1. control


In [110]:
df_distr = df_merge[df_merge['updated_segment'].isin(['1. control', '5. clicked'])]\
[['sample_category', 'city_name', 'customer_id', 'order_id', 'modified_order_status', 'cobra_ttc', 'tta', 'updated_segment']].reset_index(drop=True)
df_distr

,sample_category,city_name,customer_id,order_id,modified_order_status,cobra_ttc,tta,updated_segment
0,control,Delhi,5dffc63387a252727580ccac,682e90f919e59a597fb36199,dropped,NaN,7.0,1. control
1,control,Delhi,5dffc63387a252727580ccac,682eeaf88f45c100810348c9,dropped,NaN,6.0,1. control
2,control,Delhi,675460206841a4df6b4aa9bd,682edca59ddad10070bc80fa,dropped,NaN,24.0,1. control
3,control,Delhi,604664d47699f6661479a5d1,682f5b46d657147021309148,OCARA,NaN,7.0,1. control
4,control,Delhi,604664d47699f6661479a5d1,682f59b6d657147021308cc6,OCARA,NaN,4.0,1. control
...,...,...,...,...,...,...,...,...
7056664,control,Delhi,637e611927c3ff45b4851197,682d51a6f82a504357ea8ae0,dropped,NaN,499.0,1. control
7056665,control,Delhi,5d8db5d7a1f0a71c604ab047,682d4965acc02e2acb5ceccf,COBRA,138.0,NaN,1. control
7056666,control,Delhi,5d8db5d7a1f0a71c604ab047,682d492609b90e311db9a1a8,expired,NaN,NaN,1. control
7056667,control,Delhi,5d8db5d7a1f0a71c604ab047,682d4a10f9fb0423027fa162,COBRA,337.0,NaN,1. control


In [111]:
df_distr['end_time'] = np.where(df_distr['tta'].isna(), df_distr['cobra_ttc'], df_distr['tta'])
df_distr['end_time'] = df_distr['end_time'].fillna(300.0)
df_distr

,sample_category,city_name,customer_id,order_id,modified_order_status,cobra_ttc,tta,updated_segment,end_time
0,control,Delhi,5dffc63387a252727580ccac,682e90f919e59a597fb36199,dropped,NaN,7.0,1. control,7.0
1,control,Delhi,5dffc63387a252727580ccac,682eeaf88f45c100810348c9,dropped,NaN,6.0,1. control,6.0
2,control,Delhi,675460206841a4df6b4aa9bd,682edca59ddad10070bc80fa,dropped,NaN,24.0,1. control,24.0
3,control,Delhi,604664d47699f6661479a5d1,682f5b46d657147021309148,OCARA,NaN,7.0,1. control,7.0
4,control,Delhi,604664d47699f6661479a5d1,682f59b6d657147021308cc6,OCARA,NaN,4.0,1. control,4.0
...,...,...,...,...,...,...,...,...,...
7056664,control,Delhi,637e611927c3ff45b4851197,682d51a6f82a504357ea8ae0,dropped,NaN,499.0,1. control,499.0
7056665,control,Delhi,5d8db5d7a1f0a71c604ab047,682d4965acc02e2acb5ceccf,COBRA,138.0,NaN,1. control,138.0
7056666,control,Delhi,5d8db5d7a1f0a71c604ab047,682d492609b90e311db9a1a8,expired,NaN,NaN,1. control,300.0
7056667,control,Delhi,5d8db5d7a1f0a71c604ab047,682d4a10f9fb0423027fa162,COBRA,337.0,NaN,1. control,337.0


In [112]:
check_bins = [0, 10, 30, 60, 90, 120, 180, 240, 300, float('inf')]
check_labels = ['0-10', '10-30', '30-60', '60-90', '90-120', '120-180', '180-240', '240-300', '300+']

# Create a new column with the binned data
df_distr['end_time_bin'] = pd.cut(df_distr['end_time'], bins=check_bins, labels=check_labels, right=False)

df_distr

,sample_category,city_name,customer_id,order_id,modified_order_status,cobra_ttc,tta,updated_segment,end_time,end_time_bin
0,control,Delhi,5dffc63387a252727580ccac,682e90f919e59a597fb36199,dropped,NaN,7.0,1. control,7.0,0-10
1,control,Delhi,5dffc63387a252727580ccac,682eeaf88f45c100810348c9,dropped,NaN,6.0,1. control,6.0,0-10
2,control,Delhi,675460206841a4df6b4aa9bd,682edca59ddad10070bc80fa,dropped,NaN,24.0,1. control,24.0,10-30
3,control,Delhi,604664d47699f6661479a5d1,682f5b46d657147021309148,OCARA,NaN,7.0,1. control,7.0,0-10
4,control,Delhi,604664d47699f6661479a5d1,682f59b6d657147021308cc6,OCARA,NaN,4.0,1. control,4.0,0-10
...,...,...,...,...,...,...,...,...,...,...
7056664,control,Delhi,637e611927c3ff45b4851197,682d51a6f82a504357ea8ae0,dropped,NaN,499.0,1. control,499.0,300+
7056665,control,Delhi,5d8db5d7a1f0a71c604ab047,682d4965acc02e2acb5ceccf,COBRA,138.0,NaN,1. control,138.0,120-180
7056666,control,Delhi,5d8db5d7a1f0a71c604ab047,682d492609b90e311db9a1a8,expired,NaN,NaN,1. control,300.0,300+
7056667,control,Delhi,5d8db5d7a1f0a71c604ab047,682d4a10f9fb0423027fa162,COBRA,337.0,NaN,1. control,337.0,300+


In [113]:
df_testing = df_distr.groupby(['sample_category', 'end_time_bin', 'modified_order_status']).agg(orders = ('order_id', 'nunique')).reset_index()
df_testing

,sample_category,end_time_bin,modified_order_status,orders
0,control,0-10,COBRA,33112
1,control,0-10,COBRM,0
2,control,0-10,OCARA,449958
3,control,0-10,aborted,2003
4,control,0-10,dropped,2093459
5,control,0-10,expired,6905
6,control,0-10,requested,0
7,control,10-30,COBRA,107050
8,control,10-30,COBRM,0
9,control,10-30,OCARA,242877


In [114]:
df_testing.pivot(index=['sample_category', 'modified_order_status'], columns=['end_time_bin'], values=['orders'])

orders                          \
end_time_bin                              0-10   10-30   30-60   60-90   
sample_category modified_order_status                                    
control         COBRA                    33112  107050  184526  169854   
                COBRM                        0       0       0       0   
                OCARA                   449958  242877   95393   75881   
                aborted                   2003    1281     509     396   
                dropped                2093459  930406  339229  248986   
                expired                   6905    6047    3165    2604   
                requested                    0       0       0       0   
test            COBRA                       14      37      40      42   
                COBRM                        0       0       0       0   
                OCARA                      155      84      24      23   
                aborted                      0       0       0       0   
                dropped                    514     218      83      60   
                expired                      1       2       2       1   
                requested                    0       0       0       0   

                                                                               
end_time_bin                           90-120 120-180 180-240 240-300    300+  
sample_category modified_order_status                                          
control         COBRA                  144894  202067  124112   79581   82075  
                COBRM                       0       0       0       0   55498  
                OCARA                   59265   70212   45982   32735  117199  
                aborted                   287     394     278     173    1163  
                dropped                186526  188155  106715   65566  142303  
                expired                  2286    3297    2569    2016  345612  
                requested                   0       0       0       0      30  
test            COBRA                      36      67      37      32      31  
                COBRM                       0       0       0       0      18  
                OCARA                      20      19      23      13      49  
                aborted                     0       0       0       0       0  
                dropped                    46      61      38      24      68  
                expired                     1       1       1       0     153  
                requested                   0       0       0       0       0

In [29]:
df_merge

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,pickup_location_hex_12,drop_location_hex_12,pickup_location_hex_8,drop_location_hex_8,channel_host,time_bucket,city,userid,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag
0,loyalty_experiment2,loyalty_experiment2_bangalore,control,PHH,PLATINUM,WEEKLY,HIGH_INCOME,LOW,20250520,60534bbe34613875161fd5b6,Bangalore,Auto,682c58fb83086a6bde3f830b,dropped,NaN,1.747737e+12,1.747737e+12,NaN,NaN,NaN,NaN,1747736827728,1545,155707,17.48,2.392,8c6189253c72bff,8c61892f47687ff,886189253dfffff,8861892f47fffff,ios,afternoon,NaN,NaN,NaN,NaN,NaN,NaN,control,control,control,control
1,loyalty_experiment2,loyalty_experiment2_bangalore,control,PHH,PLATINUM,WEEKLY,HIGH_INCOME,LOW,20250520,60534bbe34613875161fd5b6,Bangalore,Auto,682c422b69b326452faa00b6,dropped,NaN,1.747731e+12,1.747731e+12,NaN,NaN,NaN,NaN,1747730987364,1415,141947,11.57,0.370,8c6189252a691ff,8c60145b54639ff,886189252bfffff,8860145b55fffff,ios,afternoon,NaN,NaN,NaN,NaN,NaN,NaN,control,control,control,control
2,loyalty_experiment2,loyalty_experiment2_bangalore,control,PHH,ELITE,DAILY,MEDIUM_INCOME,HIGH,20250520,6530b9ea11b5fe81f6eb7917,Bangalore,Link,682c2f5f130e233584a387f8,COBRA,1.747726e+12,1.747726e+12,NaN,NaN,181.0,NaN,NaN,1747726175068,1245,125935,6.82,NaN,8c618925c248bff,8c61892466913ff,88618925d5fffff,8861892467fffff,android,afternoon,NaN,NaN,NaN,NaN,NaN,NaN,control,control,control,control
3,loyalty_experiment2,loyalty_experiment2_bangalore,control,PHH,ELITE,DAILY,MEDIUM_INCOME,HIGH,20250520,6530b9ea11b5fe81f6eb7917,Bangalore,Link,682c3018269de97232169d7d,OCARA,1.747726e+12,1.747726e+12,1.747726e+12,0.002289,NaN,34.0,71.0,1747726360959,1300,130240,6.82,2.304,8c618925c24d7ff,8c61892466913ff,88618925d5fffff,8861892467fffff,android,afternoon,NaN,NaN,NaN,NaN,NaN,NaN,control,control,control,control
4,loyalty_experiment2,loyalty_experiment2_bangalore,control,PHH,ELITE,DAILY,MEDIUM_INCOME,HIGH,20250520,6530b9ea11b5fe81f6eb7917,Bangalore,Link,682c2e24130e233584a3839a,expired,NaN,1.747726e+12,NaN,NaN,NaN,NaN,NaN,1747725860322,1245,125420,6.82,NaN,8c618925c248bff,8c61892466913ff,88618925d5fffff,8861892467fffff,android,afternoon,NaN,NaN,NaN,NaN,NaN,NaN,control,control,control,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27848165,loyalty_experiment2,loyalty_experiment2_hyd,control,PHH,DORMANT,WEEKLY,MEDIUM_INCOME,LOW,20250522,6131fc0c30f72af22df085bb,Hyderabad,Auto,682ecd24306b9c6838a2c132,dropped,NaN,1.747898e+12,1.747898e+12,NaN,NaN,NaN,NaN,1747897636965,1230,123716,2.08,2.745,8c60b1965222bff,8c60b19656b23ff,8860b19653fffff,8860b19657fffff,android,afternoon,NaN,NaN,NaN,NaN,NaN,NaN,control,control,control,control
27848166,loyalty_experiment2,loyalty_experiment2_hyd,control,PHH,GOLD,BI_WEEKLY,HIGH_INCOME,MEDIUM,20250522,616fba334c6ba14747d9f789,Hyderabad,Link,682e974df16ab108fa32a88a,COBRA,1.747884e+12,1.747884e+12,NaN,NaN,168.0,NaN,NaN,1747883853083,0845,084733,3.65,NaN,8c60a259b4445ff,8c60a25995883ff,8860a259b5fffff,8860a25995fffff,android,morning_peak,NaN,NaN,NaN,NaN,NaN,NaN,control,control,control,control
27848167,loyalty_experiment2,loyalty_experiment2_hyd,control,PHH,GOLD,BI_WEEKLY,HIGH_INCOME,MEDIUM,20250522,616fba334c6ba14747d9f789,Hyderabad,Link,682e97fceb3dad435bb4dc57,COBRA,1.747884e+12,1.747884e+12,NaN,NaN,105.0,NaN,NaN,1747884028471,0845,085028,3.65,NaN,8c60a259b4441ff,8c60a25995883ff,8860a259b5fffff,8860a25995fffff,android,morning_pe

In [39]:
df_merge.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,pickup_location_hex_12,drop_location_hex_12,pickup_location_hex_8,drop_location_hex_8,channel_host,time_bucket,city,userid,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag
0,loyalty_experiment2,loyalty_experiment2_bangalore,control,PHH,PLATINUM,WEEKLY,HIGH_INCOME,LOW,20250520,60534bbe34613875161fd5b6,Bangalore,Auto,682c58fb83086a6bde3f830b,dropped,NaN,1.747737e+12,1.747737e+12,NaN,NaN,NaN,NaN,1747736827728,1545,155707,17.48,2.392,8c6189253c72bff,8c61892f47687ff,886189253dfffff,8861892f47fffff,ios,afternoon,NaN,NaN,NaN,NaN,NaN,NaN,2+km,9+km,control,control,control,control


In [ ]:
df_merge['exp_bucket'] = df_merge['exp_bucket']

- Route level comparison 
- Segment
  
- Control vs NoViewed/Viewed at (search/otw)/view_both/clicked

- Resampling (Sample control and bring it to the test level)
- Frequency 

## Resampling

In [13]:
df_merge.updated_segment.unique()

array(['1. control', '3. viewed', '4. both_viewed', '5. clicked',
       '2. not viewed'], dtype=object)

In [106]:
df_merge.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,time_bucket,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view,gross_orders,accepted_orders,aor,mp_tag,frequency,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_view_tag,hex_8_route,updated_segment
7707735,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682e90f919e59a597fb36199,dropped,NaN,1.747882e+12,1.747882e+12,NaN,NaN,NaN,7.0,1747882233633,0815,082033,6.98,...,morning_peak,None,None,NaN,None,None,None,None,None,None,None,2783.0,2394.0,86.02,1. Good,0.0,0-500m,6-9km,control,control,control,control,control,883da1118dfffff - 883da1034dfffff,1. control


In [117]:
group_cols = ['execution_name', 'taxi_regularity_segment', 'service_name', 'modified_order_status']
viewed_test = df_merge[~df_merge['updated_segment'].isin(['1. control', '2. not viewed'])]

# Calculate proportions of the viewed test group
dist = viewed_test.groupby(group_cols).size().div(len(viewed_test))


In [118]:
%%time

control = df_merge[df_merge['sample_category'] == 'control']
resampled_control = pd.DataFrame()

# Loop through the distribution
for group_values, proportion in dist.items():
    # Make sure group_values is always a tuple
    if not isinstance(group_values, tuple):
        group_values = (group_values,)
    
    n_samples = int(proportion * len(viewed_test))

    # Build a boolean mask dynamically
    mask = pd.Series(True, index=control.index)
    for col, val in zip(group_cols, group_values):
        mask &= (control[col] == val)

    candidates = control[mask]
    sample = candidates.sample(n=min(n_samples, len(candidates)), replace=False)
    resampled_control = pd.concat([resampled_control, sample])


CPU times: user 11min 6s, sys: 22.9 s, total: 11min 29s
Wall time: 11min 33s


In [119]:
# Add identifiers
viewed_test = viewed_test.copy()
viewed_test['group_type'] = 'test_viewed'

resampled_control = resampled_control.copy()
resampled_control['group_type'] = 'control_matched'

# Combine both into one DataFrame
comparison_df = pd.concat([viewed_test, resampled_control], ignore_index=True)

In [120]:
# comparison_df.to_parquet('comparison_df.parquet')

In [121]:
comparison_df.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,city,userid,aduuid_count,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,clicked,both_view,single_view,gross_orders,accepted_orders,aor,mp_tag,frequency,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_view_tag,hex_8_route,updated_segment,group_type
0,loyalty_experiment2,loyalty_experiment2_delhi,test,PHH,GOLD,WEEKLY,MEDIUM_INCOME,LOW,20250522,6054c22bba88aae23364c52c,Delhi,Link,682f11926946f36437a71d7c,dropped,NaN,1.747915e+12,1.747915e+12,NaN,NaN,NaN,143.0,1747915154756,1715,172914,9.86,...,None,None,NaN,None,None,None,None,None,None,None,2976.0,2546.0,85.55,1. Good,0.0,0-500m,9+km,test,test,test,test,test,883da11637fffff - 883da114ebfffff,3. viewed,test_viewed


In [122]:
# taxi_regularity_segment

group_type = ['control_matched', 'test_viewed']

for city_name, execution_name in  city.items():
    for group_tc in group_type:
        print(city_name, group_tc)
        print(comparison_df.query(f"execution_name == '{execution_name}' & group_type == '{group_tc}'").taxi_regularity_segment.value_counts(normalize=True))

Bangalore control_matched
Series([], Name: proportion, dtype: float64)
Bangalore test_viewed
Series([], Name: proportion, dtype: float64)
Delhi control_matched
taxi_regularity_segment
DAILY        0.413006
WEEKLY       0.227491
BI_WEEKLY    0.152311
UNKNOWN      0.076096
MONTHLY      0.066484
RARE_NEED    0.064612
Name: proportion, dtype: float64
Delhi test_viewed
taxi_regularity_segment
DAILY        0.412998
WEEKLY       0.227491
BI_WEEKLY    0.152310
UNKNOWN      0.076098
MONTHLY      0.066485
RARE_NEED    0.064618
Name: proportion, dtype: float64
Hyderabad control_matched
Series([], Name: proportion, dtype: float64)
Hyderabad test_viewed
Series([], Name: proportion, dtype: float64)


In [123]:
df1_city_group_tc = get_funnel(comparison_df, ['city_name', 'group_type'])
df1_city_group_tc

Execution time: 4.084 seconds


,city_name,group_type,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control_matched,485577,659801,406571,61.62,15.46,17.15,0.66,5.11,129.0,271.9,67.3,53.0,104.0,181.0,273.0,51.0,156.0,361.0,635.0,6.0,11.0,59.0,170.0
1,Delhi,test_viewed,179101,695421,427255,61.44,15.29,17.35,0.69,5.22,125.6,272.2,68.5,49.0,100.0,177.0,272.0,48.0,153.0,364.0,640.0,6.0,11.0,59.0,173.0


In [54]:
df1_city_group_tc = get_funnel(comparison_df, ['city_name', 'group_type'])
df1_city_group_tc

Execution time: 11.528 seconds


,city_name,group_type,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Bangalore,control_matched,408393,541550,269869,49.83,25.19,13.94,0.86,10.17,127.8,230.2,112.1,55.0,105.0,182.0,263.0,44.0,129.0,307.0,551.0,8.0,28.0,127.0,271.0
1,Bangalore,test_viewed,159134,561213,305878,54.50,21.04,14.96,0.84,8.64,124.6,233.5,114.9,53.0,102.0,176.0,258.0,44.0,128.0,310.0,565.0,8.0,28.0,126.0,274.0
2,Delhi,control_matched,443402,577175,353228,61.20,16.14,16.65,0.73,5.27,129.1,272.0,118.0,53.0,103.0,181.0,274.0,51.0,156.0,363.0,635.0,7.0,17.0,104.0,290.0
3,Delhi,test_viewed,187511,605680,374319,61.80,14.51,17.96,0.68,5.04,124.1,277.1,118.0,48.0,99.0,174.0,268.0,49.0,156.0,370.0,651.0,6.0,15.0,101.0,290.0
4,Hyderabad,control_matched,449685,570506,350079,61.36,17.66,14.70,0.45,5.82,140.0,260.5,125.9,57.0,111.0,196.0,296.0,51.0,151.0,355.0,621.0,7.0,22.0,122.0,318.0
5,Hyderabad,test_viewed,197272,588713,366193,62.20,16.63,15.05,0.40,5.71,141.1,263.7,126.3,56.0,111.0,198.0,301.0,52.0,153.0,362.0,630.0,7.0,22.0,121.0,319.0


In [ ]:
df1_city_updated_seg = get_funnel(comparison_df, ['city_name', 'updated_segment'])
df1_city_updated_seg

In [55]:
df1_city_updated_seg = get_funnel(comparison_df, ['city_name', 'updated_segment'])
df1_city_updated_seg

Execution time: 11.630 seconds


,city_name,updated_segment,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Bangalore,1. control,408393,541550,269869,49.83,25.19,13.94,0.86,10.17,127.8,230.2,112.1,55.00,105.0,182.00,263.0,44.0,129.0,307.00,551.0,8.0,28.0,127.00,271.0
1,Bangalore,3. viewed,93414,214113,104902,48.99,24.75,15.04,0.95,10.25,125.3,236.1,117.7,53.00,102.0,177.00,260.0,45.0,128.0,312.00,568.0,8.0,29.0,130.00,277.0
2,Bangalore,4. both_viewed,96890,346803,200850,57.91,18.75,14.90,0.78,7.65,124.0,231.6,113.1,53.00,101.0,175.00,256.0,44.0,129.0,310.00,564.0,8.0,27.0,124.00,272.0
3,Bangalore,5. clicked,78,297,126,42.42,22.90,22.90,0.67,11.11,167.2,442.6,132.4,63.75,151.5,220.50,331.0,43.5,173.5,494.25,848.8,10.0,50.0,148.25,280.7
4,Delhi,1. control,443402,577175,353228,61.20,16.14,16.65,0.73,5.27,129.1,272.0,118.0,53.00,103.0,181.00,274.0,51.0,156.0,363.00,635.0,7.0,17.0,104.00,290.0
5,Delhi,3. viewed,106734,218756,129016,58.98,15.75,18.71,0.73,5.81,119.3,285.0,119.3,47.00,95.0,169.00,256.0,50.0,159.0,375.00,667.0,6.0,15.0,99.00,289.0
6,Delhi,4. both_viewed,117492,386622,245169,63.41,13.80,17.53,0.65,4.60,127.1,272.1,117.2,50.00,100.0,178.00,276.0,49.0,155.0,366.00,641.0,6.0,15.0,102.00,290.0
7,Delhi,5. clicked,87,302,134,44.37,16.56,22.19,2.65,14.24,143.6,485.7,193.1,65.00,118.5,189.25,266.8,59.0,277.0,547.00,793.0,6.5,30.0,203.50,542.6
8,Hyderabad,1. control,449685,570506,350079,61.36,17.66,14.70,0.45,5.82,140.0,260.5,125.9,57.00,111.0,196.00,296.0,51.0,151.0,355.00,621.0,7.0,22.0,122.00,318.0
9,Hyderabad,3. viewed,115812,240352,131106,54.55,22.06,15.00,0.54,7.83,144.5,276.1,135.5,58.00,114.0,205.00,308.0,56.0,160.0,376.00,658.0,8.0,26.0,138.00,339.0


## Biasness Validate

In [27]:
# taxi_regularity_segment

sample_category = ['test', 'control']

for city_name, execution_name in  city.items():
    for group_tc in sample_category:
        print(city_name, group_tc)
        print(df_merge.query(f"execution_name == '{execution_name}' & sample_category == '{group_tc}'").taxi_regularity_segment.value_counts(normalize=True))

Bangalore test
taxi_regularity_segment
DAILY        0.330492
WEEKLY       0.287404
BI_WEEKLY    0.167839
RARE_NEED    0.076843
MONTHLY      0.073898
UNKNOWN      0.063525
Name: proportion, dtype: float64
Bangalore control
taxi_regularity_segment
DAILY        0.330497
WEEKLY       0.287185
BI_WEEKLY    0.168275
RARE_NEED    0.076129
MONTHLY      0.075000
UNKNOWN      0.062915
Name: proportion, dtype: float64
Delhi test
taxi_regularity_segment
DAILY        0.388966
WEEKLY       0.235708
BI_WEEKLY    0.156985
UNKNOWN      0.081734
MONTHLY      0.069456
RARE_NEED    0.067151
Name: proportion, dtype: float64
Delhi control
taxi_regularity_segment
DAILY        0.391564
WEEKLY       0.237115
BI_WEEKLY    0.154571
UNKNOWN      0.081482
MONTHLY      0.068052
RARE_NEED    0.067216
Name: proportion, dtype: float64
Hyderabad test
taxi_regularity_segment
WEEKLY       0.310700
DAILY        0.267230
BI_WEEKLY    0.185849
RARE_NEED    0.092540
MONTHLY      0.086316
UNKNOWN      0.057365
Name: proportio

In [28]:
# taxi_retention_segments

sample_category = ['test', 'control']

for city_name, execution_name in  city.items():
    for group_tc in sample_category:
        print(city_name, group_tc)
        print(df_merge.query(f"execution_name == '{execution_name}' & sample_category == '{group_tc}'").taxi_retention_segments.value_counts(normalize=True))

Bangalore test
taxi_retention_segments
ELITE       0.466152
PLATINUM    0.197965
DORMANT     0.112750
GOLD        0.100346
INACTIVE    0.041625
SILVER      0.040908
HH          0.033127
PRIME       0.007127
Name: proportion, dtype: float64
Bangalore control
taxi_retention_segments
ELITE       0.466458
PLATINUM    0.199016
DORMANT     0.111904
GOLD        0.100973
SILVER      0.041466
INACTIVE    0.041172
HH          0.032424
PRIME       0.006587
Name: proportion, dtype: float64
Delhi test
taxi_retention_segments
ELITE       0.429550
PLATINUM    0.200947
GOLD        0.121202
DORMANT     0.104279
SILVER      0.051380
INACTIVE    0.042801
HH          0.039340
PRIME       0.010502
Name: proportion, dtype: float64
Delhi control
taxi_retention_segments
ELITE       0.433417
PLATINUM    0.198559
GOLD        0.121045
DORMANT     0.104548
SILVER      0.050175
INACTIVE    0.042345
HH          0.039466
PRIME       0.010445
Name: proportion, dtype: float64
Hyderabad test
taxi_retention_segments
ELI

In [29]:
# taxi_retention_segments

sample_category = ['test', 'control']

for city_name, execution_name in  city.items():
    for group_tc in sample_category:
        print(city_name, group_tc)
        print(df_merge.query(f"execution_name == '{execution_name}' & sample_category == '{group_tc}'").service_name.value_counts(normalize=True))

Bangalore test
service_name
Auto          0.605941
Link          0.233304
CabEconomy    0.099394
CabAC         0.021856
Bike Lite     0.015570
Auto Pool     0.008438
Cab SUV       0.007639
CabPremium    0.006164
Auto Pet      0.001694
Name: proportion, dtype: float64
Bangalore control
service_name
Auto          0.605590
Link          0.233454
CabEconomy    0.099321
CabAC         0.021457
Bike Lite     0.015954
Auto Pool     0.008373
Cab SUV       0.007746
CabPremium    0.006413
Auto Pet      0.001692
Name: proportion, dtype: float64
Delhi test
service_name
Auto          0.264745
Bike Lite     0.208577
Link          0.207558
CabEconomy    0.195739
Bike Metro    0.077679
CabPremium    0.022706
Cab SUV       0.011779
Auto NCR      0.011217
Name: proportion, dtype: float64
Delhi control
service_name
Auto          0.263014
Bike Lite     0.210956
Link          0.209425
CabEconomy    0.194124
Bike Metro    0.077186
CabPremium    0.022062
Cab SUV       0.011792
Auto NCR      0.011440
Name: pro